# Sport Machine — kézilabda-mozdulatfelismerés (UNIRI-HBD)

Mit csinál ez a notebook:
1. Minden kivágott klipnél megkeresi a végrehajtót a teljes jelenetben, és előtanított
   **YOLO11-pose** modellel kinyeri a 17 testpontot kockánként (→ `poses.npz`).
2. Kiértékel három feladatot, **jelenet szerint csoportosított 5-szörös keresztvalidációval**:
   lövés / nem lövés · felugrásos / helyből / egyéb · mind a 7 kategória.
3. Minden lövésre kapus-szempontú profilt ír (lövőkéz, kartartás, felugrás) → `kapus_profilok.csv`.
4. Végső modelleket ment (`model_*.joblib`, `model_*.pt`) a Macen futtatáshoz.

**Beállítás (egyszer):**
- *Add Input* → a feltöltött `uniri-hbd` adathalmaz (a zip; a Kaggle kibontja).
- *Settings → Accelerator*: **GPU T4 x2** (egyet használunk). *Internet*: **On** (pip + súlyok).
- *Run All*. Kb. 30–50 perc; a kimenet a jobb oldali *Output* fülön (`/kaggle/working`).

Ha megszakad: újraindítva a már kész klipeket a gyorsítótárból (`poses_cache/`) veszi.

Adat: UNIRI-HBD, Ivašić-Kos, M., Pobar, M., University of Rijeka, DOI 10.21227/0g0a-fe06, CC BY 4.0.


In [ ]:
!pip install -q ultralytics imageio-ffmpeg
!nvidia-smi -L || true

## A csomag forrása (a repó `ml/hbd` mappájából generálva)

In [ ]:
import os
os.makedirs('hbd', exist_ok=True)

In [ ]:
%%writefile hbd/__init__.py
"""hbd — kézilabda-mozdulatfelismerés valódi videóból (UNIRI-HBD adathalmaz).

Ez a csomag SZÁNDÉKOSAN független a `backend/handball` szabályrétegeitől:
itt a gép tanul felvételekből, nem kézzel írt küszöbökből ítél. A kimenete
(testpont-sorozat, mozdulat-osztály, lövés-profil) később a backend
"gondolkodó" rétegeinek bemenete lesz.

Modulok:
    dataset     — az adathalmaz bejárása (klip → jelenet → kategória)
    video       — képkockák olvasása ffmpeg-gel (OpenCV-dekóder nélkül)
    locate      — a kivágott klip megkeresése a teljes jelenetben
    pose        — testpont-kinyerés előtanított YOLO-pose modellel (CLI)
    features    — testpont-normalizálás, sorozat, klip-szintű jellemzők
    goalkeeper  — kapus-szempontú lövés-profil (kartartás, lövőkar, felugrás)
    models      — kis időbeli konvolúciós háló (torch)
    train       — keresztvalidált kiértékelés + végső modellek (CLI)

Adatforrás: UNIRI-HBD, Ivašić-Kos, M., Pobar, M., University of Rijeka,
DOI 10.21227/0g0a-fe06, CC BY 4.0.
"""

__version__ = "0.1.0"


In [ ]:
%%writefile hbd/dataset.py
"""Az UNIRI-HBD adathalmaz bejárása.

Szerkezet (a zip kibontva):

    handball_s/
        actions/<kategória>/<jelenet>_act<N>.avi   — a végrehajtóra kivágott klip
        scenes/<kategória>/<jelenet>.mp4|.avi       — a teljes jelenet
        player_detections/<kategória>/<jelenet>.csv — játékos-dobozok kockánként

Egy tanító minta = egy kivágott klip (`actions/`). A klip a jelenet
képpontjaiból van kivágva (a sablon-egyezés 1,000), így a jelenetben
pontosan visszakereshető (lásd `locate`). A kiértékelés a JELENET szerint
csoportosít: ugyanannak a jelenetnek két klipje sosem kerülhet egyszerre a
tanító és a teszt részbe.
"""

from __future__ import annotations

import re
from dataclasses import dataclass
from pathlib import Path

# A zip kategória-mappái (a "lövés" mappa neve "shot", nem "shooting").
CATEGORIES = ("crossing", "defence", "dribbling", "jump-shot", "passing", "running", "shot")
SHOT_CATEGORIES = ("jump-shot", "shot")

# Feladatok: név → (osztálynevek, kategória → osztály-index)
TASKS = {
    # lövés vagy nem lövés — a kapus első kérdése
    "binary": (("nem_loves", "loves"), {c: int(c in SHOT_CATEGORIES) for c in CATEGORIES}),
    # felugrásos lövés / helyből lövés / egyéb
    "shot3": (
        ("egyeb", "felugrasos", "helybol"),
        {c: {"jump-shot": 1, "shot": 2}.get(c, 0) for c in CATEGORIES},
    ),
    # mind a hét kategória (a szakirodalommal összevethető)
    "all7": (CATEGORIES, {c: i for i, c in enumerate(CATEGORIES)}),
}

VIDEO_EXTS = (".mp4", ".avi", ".mov", ".mkv")
_ACT_RE = re.compile(r"_act\d+$")


@dataclass(frozen=True)
class Clip:
    """Egy kivágott mozdulat-klip és a hozzá tartozó jelenet."""

    name: str  # a klip fájlneve kiterjesztés nélkül, pl. "shot_KS_100_act1"
    category: str  # pl. "shot"
    action_path: Path
    scene_path: Path
    detections_path: Path | None

    @property
    def scene_id(self) -> str:
        """Csoport-azonosító a kiértékeléshez: kategória/jelenet."""
        return f"{self.category}/{self.scene_path.stem}"


def find_root(path: str | Path) -> Path:
    """Megkeresi az adathalmaz gyökerét (ahol `actions/` és `scenes/` van).

    Elfogadja magát a gyökeret, a `handball_s` szülőjét, vagy egy Kaggle
    bemeneti mappát, ahol a kibontott zip mélyebben ül.
    """
    p = Path(path)
    if (p / "actions").is_dir() and (p / "scenes").is_dir():
        return p
    for cand in sorted(p.rglob("actions")):
        root = cand.parent
        if cand.is_dir() and (root / "scenes").is_dir():
            return root
    raise FileNotFoundError(f"Nem találom az UNIRI-HBD gyökerét (actions/ + scenes/) itt: {p}")


def _find_video(folder: Path, stem: str) -> Path | None:
    for ext in VIDEO_EXTS:
        f = folder / f"{stem}{ext}"
        if f.exists():
            return f
    return None


def index_dataset(root: str | Path, categories: tuple[str, ...] | None = None) -> list[Clip]:
    """Az összes kivágott klip listája, determinisztikus (név szerinti) sorrendben.

    A jelenet nélküli ("árva") klipeket kihagyja — ezekből nem kereshető
    vissza a végrehajtó helye.
    """
    root = find_root(root)
    cats = categories or CATEGORIES
    clips: list[Clip] = []
    for cat in cats:
        act_dir = root / "actions" / cat
        if not act_dir.is_dir():
            continue
        for act in sorted(act_dir.iterdir()):
            if act.suffix.lower() not in VIDEO_EXTS:
                continue
            scene_stem = _ACT_RE.sub("", act.stem)
            scene = _find_video(root / "scenes" / cat, scene_stem)
            if scene is None:
                continue
            det = root / "player_detections" / cat / f"{scene_stem}.csv"
            clips.append(Clip(act.stem, cat, act, scene, det if det.exists() else None))
    return clips


def read_detections(path: str | Path) -> list[list[tuple[float, float, float, float, float]]]:
    """A `player_detections` CSV beolvasása: kockánként a dobozok listája.

    Formátum: "Frame N" sor, alatta soronként "x1, y1, x2, y2, megbízhatóság".
    Követés (ki kicsoda) és a végrehajtó jelölése NINCS benne.
    """
    frames: list[list[tuple[float, float, float, float, float]]] = []
    for line in Path(path).read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        if line.startswith("Frame"):
            frames.append([])
            continue
        parts = [float(v) for v in line.split(",")]
        if len(parts) == 4:
            parts.append(1.0)
        if frames and len(parts) == 5:
            frames[-1].append(tuple(parts))  # type: ignore[arg-type]
    return frames


In [ ]:
%%writefile hbd/video.py
"""Képkockák olvasása ffmpeg-gel.

Miért nem OpenCV? A felhős gépen az OpenCV videó-dekódere ezeken a
fájlokon összeomlott (szegmentálási hiba), az ffmpeg viszont stabil és a
Kaggle-képen is megvan. Ha nincs rendszer-ffmpeg, az `imageio-ffmpeg`
csomag statikus binárisát használjuk.

A kockákat EGYENKÉNT adjuk vissza (generátor): egy 2704×1520-as jelenet
egészében több GB lenne a memóriában.
"""

from __future__ import annotations

import re
import shutil
import subprocess
from pathlib import Path
from typing import Iterator

import numpy as np

_SIZE_RE = re.compile(r"\b(\d{2,5})x(\d{2,5})\b")
_FPS_RE = re.compile(r"(\d+(?:\.\d+)?) fps")


def ffmpeg_exe() -> str:
    exe = shutil.which("ffmpeg")
    if exe:
        return exe
    try:
        import imageio_ffmpeg  # type: ignore

        return imageio_ffmpeg.get_ffmpeg_exe()
    except ImportError as e:  # pragma: no cover - környezetfüggő
        raise RuntimeError("Nincs ffmpeg: telepítsd (apt) vagy `pip install imageio-ffmpeg`") from e


def probe(path: str | Path) -> tuple[int, int, float]:
    """(szélesség, magasság, fps) az ffmpeg fejléc-kiírásából (ffprobe nélkül)."""
    p = subprocess.run(
        [ffmpeg_exe(), "-hide_banner", "-i", str(path)], capture_output=True, text=True
    )
    for line in p.stderr.splitlines():
        if "Video:" in line:
            m = _SIZE_RE.search(line.split("Video:", 1)[1])
            f = _FPS_RE.search(line)
            if m:
                return int(m.group(1)), int(m.group(2)), float(f.group(1)) if f else 0.0
    raise RuntimeError(f"Nem olvasható videó: {path}")


def iter_frames(path: str | Path, gray: bool = False) -> Iterator[np.ndarray]:
    """A videó kockái sorban: (H, W) szürke vagy (H, W, 3) BGR uint8."""
    w, h, _ = probe(path)
    ch = 1 if gray else 3
    size = w * h * ch
    proc = subprocess.Popen(
        [ffmpeg_exe(), "-v", "error", "-i", str(path), "-f", "rawvideo",
         "-pix_fmt", "gray" if gray else "bgr24", "-"],
        stdout=subprocess.PIPE,
        # a klip végén szándékosan bezárjuk a csövet → az ffmpeg "Broken pipe"-ot
        # írna; ez nem hiba, ne szemetelje a naplót
        stderr=subprocess.DEVNULL,
    )
    assert proc.stdout is not None
    try:
        while True:
            buf = proc.stdout.read(size)
            if len(buf) < size:
                break
            arr = np.frombuffer(buf, np.uint8)
            yield arr.reshape(h, w) if gray else arr.reshape(h, w, 3)
    finally:
        proc.stdout.close()
        proc.kill()
        proc.wait()


def read_frames(path: str | Path, gray: bool = False) -> np.ndarray:
    """Az összes kocka egy tömbben — csak KIS videóra (a kivágott klipekre)."""
    frames = list(iter_frames(path, gray=gray))
    if not frames:
        raise RuntimeError(f"Üres videó: {path}")
    return np.stack(frames)


In [ ]:
%%writefile hbd/locate.py
"""A kivágott klip visszakeresése a teljes jelenetben.

Az UNIRI-HBD `actions/` klipjei a jelenet képpontjainak szó szerinti
kivágásai (állandó méretű ablak, ami a játékost követi). Ezért a klip
k-adik kockája a jelenet (eltolás + k)-adik kockájában sablon-egyezéssel
pontosan megtalálható. Így tudjuk meg, MELYIK játékos a végrehajtó és HOL
áll — a `player_detections` ezt nem mondja meg.

Két lépés:
  1. eltolás: a klip első kockáját a jelenet kockáin keressük, az első
     (gyakorlatilag) pontos egyezésnél megállunk;
  2. követés: a további kockákon csak az előző hely környezetében
     keresünk (gyors), rossz egyezésnél az egész képen.
"""

from __future__ import annotations

from typing import Iterable, Iterator

import cv2
import numpy as np

# A klip a jelenet képpontjainak másolata → a helyes kockán az egyezés 1,000.
# Ennél jobb egyezésnél az eltolás-keresés megállhat. SZIGORÚ kell: álló
# kameránál a SZOMSZÉD kocka is ~0,98-at ad, és 0,98-as küszöbnél a keresés
# egy kockával korán megállt (az egész klip elcsúszott, 0,85-ös egyezésekkel).
EXACT_MATCH = 0.999
# Követéskor ennél gyengébb ablakos egyezésnél az egész képen keresünk.
TRACK_MIN_SCORE = 0.90
# A keresőablak a sablon körül: a sablon-méret ennyiszerese, de min. ennyi px.
SEARCH_PAD_FRAC = 0.6
SEARCH_PAD_MIN = 32


def _gray(frame: np.ndarray) -> np.ndarray:
    return frame if frame.ndim == 2 else cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)


def match_full(frame: np.ndarray, tmpl: np.ndarray) -> tuple[float, tuple[int, int]]:
    """Legjobb egyezés az egész képen: (pontszám, (x, y) bal-felső sarok)."""
    r = cv2.matchTemplate(_gray(frame), tmpl, cv2.TM_CCOEFF_NORMED)
    _, score, _, loc = cv2.minMaxLoc(r)
    return float(score), (int(loc[0]), int(loc[1]))


def match_window(
    frame: np.ndarray, tmpl: np.ndarray, prev_xy: tuple[int, int]
) -> tuple[float, tuple[int, int]]:
    """Egyezés az előző hely környezetében; ha az ablak nem fér, az egész képen."""
    g = _gray(frame)
    th, tw = tmpl.shape
    pad = max(SEARCH_PAD_MIN, int(SEARCH_PAD_FRAC * max(th, tw)))
    x0 = max(0, prev_xy[0] - pad)
    y0 = max(0, prev_xy[1] - pad)
    x1 = min(g.shape[1], prev_xy[0] + tw + pad)
    y1 = min(g.shape[0], prev_xy[1] + th + pad)
    if x1 - x0 < tw or y1 - y0 < th:
        return match_full(g, tmpl)
    r = cv2.matchTemplate(g[y0:y1, x0:x1], tmpl, cv2.TM_CCOEFF_NORMED)
    _, score, _, loc = cv2.minMaxLoc(r)
    return float(score), (int(loc[0]) + x0, int(loc[1]) + y0)


def find_offset(frames: Iterable[np.ndarray], first_tmpl: np.ndarray) -> tuple[int, float]:
    """A jelenet melyik kockájánál kezdődik a klip: (eltolás, pontszám).

    Az első pontos egyezésnél megáll; ha nincs pontos, a legjobbat adja.
    """
    best = (-1, -1.0)
    for i, fr in enumerate(frames):
        score, _ = match_full(fr, first_tmpl)
        if score > best[1]:
            best = (i, score)
        if score >= EXACT_MATCH:
            break
    return best


def track(
    frames: Iterable[np.ndarray], action_gray: np.ndarray, offset: int
) -> Iterator[tuple[int, np.ndarray, tuple[int, int, int, int], float]]:
    """Végigköveti a klipet a jeleneten.

    Bemenet: a jelenet kockái (szürke vagy BGR, sorban), a klip szürke kockái
    (T, h, w) és az eltolás. Kimenet kockánként: (k, a jelenet kockája,
    doboz (x1, y1, x2, y2), pontszám). A jelenet végén megáll.
    """
    th, tw = action_gray.shape[1:]
    prev: tuple[int, int] | None = None
    for i, fr in enumerate(frames):
        k = i - offset
        if k < 0:
            continue
        if k >= len(action_gray):
            break
        tmpl = action_gray[k]
        if prev is None:
            score, xy = match_full(fr, tmpl)
        else:
            score, xy = match_window(fr, tmpl, prev)
            if score < TRACK_MIN_SCORE:
                s2, xy2 = match_full(fr, tmpl)
                if s2 > score:
                    score, xy = s2, xy2
        prev = xy
        yield k, fr, (xy[0], xy[1], xy[0] + tw, xy[1] + th), score


In [ ]:
%%writefile hbd/pose.py
"""Testpont-kinyerés: minden klip végrehajtójára 17 COCO-testpont kockánként.

Folyamat klipenként:
  1. a kivágott klip megkeresése a jelenetben (`locate`) → a végrehajtó doboza,
  2. a doboz körül négyzetes, MARGÓS kivágás a teljes felbontású jelenetből
     (a pózmodellnek kell a környezet; a klip maga szorosan vágott),
  3. előtanított YOLO-pose (Ultralytics) a kivágásokon, kötegben,
  4. a kivágáson talált emberek közül az, amelyik a végrehajtó dobozával a
     legjobban fed (több ember is lehet a kivágásban),
  5. a testpontok visszaszámolása a jelenet képkoordinátáiba.

Kimenet: egyetlen tömör .npz (néhány MB) — erre épül minden tanítás.
Megszakítás után folytatható: klipenként gyorsítótár-fájl készül.

Futtatás (Kaggle GPU-n):
    python -m hbd.pose --data /kaggle/input/uniri-hbd --out /kaggle/working/poses.npz --device 0
Gyors próba CPU-n:
    python -m hbd.pose --data ./handball_s --out poses.npz --limit 5 --weights yolo11n-pose.pt
"""

from __future__ import annotations

import argparse
import sys
import time
from pathlib import Path

import numpy as np

from .dataset import Clip, index_dataset
from .locate import find_offset, track
from .video import iter_frames, probe, read_frames

KP_COUNT = 17
# A pózkivágás oldala = a végrehajtó dobozának hosszabbik oldala × ennyi.
CROP_SCALE = 1.6
# A pózmodell bemeneti mérete (a ~110 px-es játékost erre nagyítjuk).
POSE_IMGSZ = 384
# Ennél kisebb fedés (IoU) a végrehajtó dobozával → nem őt találta a modell.
MIN_PICK_IOU = 0.15
# Alapértelmezett súlyok: GPU-n a legpontosabb, CPU-n a legkisebb.
DEFAULT_WEIGHTS_GPU = "yolo11x-pose.pt"
DEFAULT_WEIGHTS_CPU = "yolo11n-pose.pt"

# COCO-csontváz a rajzoláshoz
SKELETON = [(5, 7), (7, 9), (6, 8), (8, 10), (5, 6), (5, 11), (6, 12), (11, 12),
            (11, 13), (13, 15), (12, 14), (14, 16), (0, 5), (0, 6)]


def square_crop_box(box, frame_w: int, frame_h: int, scale: float = CROP_SCALE):
    """Négyzetes kivágó-ablak a doboz közepe körül, a képre szorítva: (x0, y0, x1, y1)."""
    x1, y1, x2, y2 = box
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    side = max(x2 - x1, y2 - y1) * scale
    x0 = int(round(max(0, cx - side / 2)))
    y0 = int(round(max(0, cy - side / 2)))
    xe = int(round(min(frame_w, cx + side / 2)))
    ye = int(round(min(frame_h, cy + side / 2)))
    return x0, y0, xe, ye


def iou(a, b) -> float:
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0]))
    iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def pick_person(det_boxes: np.ndarray, target_box) -> int | None:
    """A végrehajtó dobozával legjobban fedő észlelés indexe (vagy None)."""
    if len(det_boxes) == 0:
        return None
    scores = [iou(b, target_box) for b in det_boxes]
    j = int(np.argmax(scores))
    return j if scores[j] >= MIN_PICK_IOU else None


class PoseModel:
    """Vékony burok az Ultralytics YOLO-pose köré (lusta import)."""

    def __init__(self, weights: str, device: str | None = None, imgsz: int = POSE_IMGSZ,
                 batch: int = 32):
        from ultralytics import YOLO  # lusta: a tesztekhez nem kell

        self.model = YOLO(weights)
        self.device = device
        self.imgsz = imgsz
        self.batch = batch

    def __call__(self, crops: list[np.ndarray]) -> list[tuple[np.ndarray, np.ndarray]]:
        """Kivágásonként: (dobozok (n, 4), testpontok (n, 17, 3)) a kivágás koordinátáiban."""
        out = []
        for i in range(0, len(crops), self.batch):
            res = self.model.predict(crops[i:i + self.batch], imgsz=self.imgsz,
                                     device=self.device, verbose=False, conf=0.1)
            for r in res:
                if r.keypoints is None or len(r.boxes) == 0:
                    out.append((np.zeros((0, 4), np.float32), np.zeros((0, KP_COUNT, 3), np.float32)))
                    continue
                boxes = r.boxes.xyxy.cpu().numpy().astype(np.float32)
                xy = r.keypoints.xy.cpu().numpy()
                conf = r.keypoints.conf
                conf = conf.cpu().numpy() if conf is not None else np.ones(xy.shape[:2])
                out.append((boxes, np.concatenate([xy, conf[..., None]], -1).astype(np.float32)))
        return out


def extract_clip(clip: Clip, model) -> dict:
    """Egy klip végrehajtójának testpont-sorozata a jelenet koordinátáiban."""
    action = read_frames(clip.action_path, gray=True)
    w, h, fps = probe(clip.scene_path)
    offset, off_score = find_offset(iter_frames(clip.scene_path, gray=True), action[0])

    crops, origins, boxes, scores = [], [], [], []
    for _k, frame, box, score in track(iter_frames(clip.scene_path), action, offset):
        x0, y0, x1, y1 = square_crop_box(box, w, h)
        crops.append(np.ascontiguousarray(frame[y0:y1, x0:x1]))
        origins.append((x0, y0))
        boxes.append(box)
        scores.append(score)

    T = len(crops)
    kps = np.zeros((T, KP_COUNT, 3), np.float32)
    if T:
        for t, (det_boxes, det_kps) in enumerate(model(crops)):
            ox, oy = origins[t]
            tb = boxes[t]
            local_target = (tb[0] - ox, tb[1] - oy, tb[2] - ox, tb[3] - oy)
            j = pick_person(det_boxes, local_target)
            if j is not None:
                kps[t] = det_kps[j]
                kps[t, :, 0] += ox
                kps[t, :, 1] += oy
    return {
        "kps": kps,
        "boxes": np.asarray(boxes, np.float32).reshape(T, 4),
        "match": np.asarray(scores, np.float32),
        "offset": offset,
        "offset_score": off_score,
        "fps": fps,
        "frame_size": (w, h),
        "action_len": len(action),
        "crops_preview": crops[T // 2] if T else None,
    }


def save_dataset(path: Path, clips: list[Clip], results: list[dict]) -> None:
    """Az összes klip egy tömör .npz-ben (pickle nélkül)."""
    lens = [len(r["kps"]) for r in results]
    offsets = np.concatenate([[0], np.cumsum(lens)]).astype(np.int64)
    np.savez_compressed(
        path,
        names=np.array([c.name for c in clips]),
        categories=np.array([c.category for c in clips]),
        scene_ids=np.array([c.scene_id for c in clips]),
        offsets=offsets,
        kps=np.concatenate([r["kps"] for r in results]) if results else np.zeros((0, KP_COUNT, 3)),
        boxes=np.concatenate([r["boxes"] for r in results]) if results else np.zeros((0, 4)),
        match=np.concatenate([r["match"] for r in results]) if results else np.zeros(0),
        fps=np.array([r["fps"] for r in results], np.float32),
        frame_size=np.array([r["frame_size"] for r in results], np.int32).reshape(-1, 2),
    )


def load_dataset(path: str | Path) -> dict:
    """Az .npz visszaolvasása klip-listává: [{name, category, scene_id, kps, boxes, fps}]."""
    z = np.load(path)
    off = z["offsets"]
    clips = []
    for i, name in enumerate(z["names"]):
        a, b = off[i], off[i + 1]
        clips.append({
            "name": str(name),
            "category": str(z["categories"][i]),
            "scene_id": str(z["scene_ids"][i]),
            "kps": z["kps"][a:b],
            "boxes": z["boxes"][a:b],
            "match": z["match"][a:b],
            "fps": float(z["fps"][i]) or 30.0,
        })
    return {"clips": clips}


def draw_pose(img: np.ndarray, kps: np.ndarray, min_conf: float = 0.3) -> np.ndarray:
    import cv2

    out = img.copy()
    for a, b in SKELETON:
        if kps[a, 2] >= min_conf and kps[b, 2] >= min_conf:
            cv2.line(out, tuple(int(v) for v in kps[a, :2]), tuple(int(v) for v in kps[b, :2]),
                     (0, 255, 255), 1, cv2.LINE_AA)
    for j, (x, y, c) in enumerate(kps):
        if c >= min_conf:
            col = (0, 0, 255) if j in (9, 10) else (0, 255, 0)  # csukló piros
            cv2.circle(out, (int(x), int(y)), 2, col, -1, cv2.LINE_AA)
    return out


def _preview_tile(clip: Clip, res: dict, size: int = 200) -> np.ndarray | None:
    import cv2

    T = len(res["kps"])
    if T == 0:
        return None
    t = T // 2
    crop = res["crops_preview"]
    w, h = res["frame_size"]
    x0, y0, _, _ = square_crop_box(res["boxes"][t], w, h)
    k = res["kps"][t].copy()
    k[:, 0] -= x0
    k[:, 1] -= y0
    sc = size / max(crop.shape[:2])
    img = cv2.resize(crop, None, fx=sc, fy=sc, interpolation=cv2.INTER_CUBIC)
    k[:, :2] *= sc
    img = draw_pose(img, k)
    tile = np.zeros((size + 18, size, 3), np.uint8)
    tile[:img.shape[0], :img.shape[1]] = img
    cv2.putText(tile, clip.name[:28], (2, size + 13), cv2.FONT_HERSHEY_SIMPLEX, 0.33,
                (255, 255, 255), 1, cv2.LINE_AA)
    return tile


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description="UNIRI-HBD: testpont-kinyerés a végrehajtóra")
    ap.add_argument("--data", required=True, help="az adathalmaz (vagy egy szülő mappája)")
    ap.add_argument("--out", required=True, help="kimeneti .npz")
    ap.add_argument("--weights", default=None, help="YOLO-pose súlyok (alap: GPU-n x, CPU-n n)")
    ap.add_argument("--device", default=None, help="pl. 0 (GPU) vagy cpu")
    ap.add_argument("--imgsz", type=int, default=POSE_IMGSZ)
    ap.add_argument("--batch", type=int, default=32)
    ap.add_argument("--limit", type=int, default=0, help="csak az első N klip kategóriánként")
    ap.add_argument("--categories", default="", help="vesszővel, pl. shot,jump-shot")
    ap.add_argument("--preview", type=int, default=0, help="ennyi ellenőrző kép kategóriánként")
    args = ap.parse_args(argv)

    cats = tuple(c for c in args.categories.split(",") if c) or None
    clips = index_dataset(args.data, cats)
    if args.limit:
        per: dict[str, int] = {}
        kept = []
        for c in clips:
            per[c.category] = per.get(c.category, 0) + 1
            if per[c.category] <= args.limit:
                kept.append(c)
        clips = kept

    weights = args.weights
    if weights is None:
        try:
            import torch

            gpu = torch.cuda.is_available() and args.device != "cpu"
        except ImportError:
            gpu = False
        weights = DEFAULT_WEIGHTS_GPU if gpu else DEFAULT_WEIGHTS_CPU
    model = PoseModel(weights, device=args.device, imgsz=args.imgsz, batch=args.batch)

    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    cache = out.parent / (out.stem + "_cache")
    cache.mkdir(exist_ok=True)

    results, done_clips, tiles = [], [], {}
    t0 = time.time()
    for i, clip in enumerate(clips):
        cf = cache / f"{clip.category}__{clip.name}.npz"
        want_tile = len(tiles.get(clip.category, [])) < args.preview
        if cf.exists() and not want_tile:
            z = np.load(cf)
            res = {k: z[k] for k in ("kps", "boxes", "match")}
            res.update(fps=float(z["fps"]), frame_size=tuple(z["frame_size"]))
        else:
            try:
                res = extract_clip(clip, model)
            except Exception as e:  # egy hibás videó ne vigye el a futást
                print(f"  ! {clip.name}: {e}", file=sys.stderr)
                continue
            np.savez_compressed(cf, kps=res["kps"], boxes=res["boxes"], match=res["match"],
                                fps=res["fps"], frame_size=np.array(res["frame_size"]))
            if want_tile:
                tile = _preview_tile(clip, res)
                if tile is not None:
                    tiles.setdefault(clip.category, []).append(tile)
        results.append(res)
        done_clips.append(clip)
        if (i + 1) % 25 == 0 or i + 1 == len(clips):
            el = time.time() - t0
            print(f"  {i + 1}/{len(clips)} klip, {el:.0f} s (≈{el / (i + 1):.1f} s/klip)", flush=True)

    save_dataset(out, done_clips, results)
    total = sum(len(r["kps"]) for r in results)
    seen = sum(int((r["kps"][:, :, 2].max(1) > 0).sum()) for r in results)
    low = sum(int((r["match"] < 0.9).sum()) for r in results)
    print(f"Kész: {len(done_clips)} klip, {total} kocka → {out}")
    print(f"  pózt talált a kockák {100 * seen / max(1, total):.1f}%-án; "
          f"gyenge klip-egyezés (<0,9): {low} kocka")

    if tiles:
        import cv2

        rows = []
        width = max(len(v) for v in tiles.values())
        for cat in sorted(tiles):
            row = tiles[cat] + [np.zeros_like(tiles[cat][0])] * (width - len(tiles[cat]))
            rows.append(np.hstack(row))
        pv = out.parent / (out.stem + "_preview.jpg")
        cv2.imwrite(str(pv), np.vstack(rows))
        print(f"  ellenőrző kép: {pv}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile hbd/features.py
"""Testpont-sorozatból tanítható bemenet.

Két alak:
  * SOROZAT (a neurális hálónak): kockánként a 17 testpont a csípő-
    középponthoz viszonyítva, törzshosszal osztva, felfelé pozitív y-nal,
    plusz a test középpontjának elmozdulása; fix hosszra (SEQ_LEN) átmintázva.
  * JELLEMZŐK (a döntési fának): klipenként néhány tucat értelmezhető szám
    (kar-magasság, csukló-sebesség, felugrás, váll-szélesség, ...).

A normalizálás teszi a kis (~110 px) és a nagy játékost, a kép bal és jobb
szélét összevethetővé. Az idő MÁSODPERCBEN számít (a jelenetek 30 és 48
fps-esek vegyesen) — a sebesség tehát egység / s, nem egység / kocka.
"""

from __future__ import annotations

import numpy as np

# COCO testpont-indexek
NOSE = 0
L_SH, R_SH, L_EL, R_EL, L_WR, R_WR = 5, 6, 7, 8, 9, 10
L_HIP, R_HIP, L_KN, R_KN, L_AN, R_AN = 11, 12, 13, 14, 15, 16
# Tükrözéskor (bal ↔ jobb) a testpontok cseréje
FLIP_PERM = [0, 2, 1, 4, 3, 6, 5, 8, 7, 10, 9, 12, 11, 14, 13, 16, 15]

# Ennél kisebb megbízhatóságú testpontot hiányzónak veszünk és időben pótoljuk.
KP_MIN_CONF = 0.3
# A neurális háló bemenetének hossza (kocka).
SEQ_LEN = 32
# A sorozat csatornái: 17×(x, y) + középpont-elmozdulás (dx, dy) + 17 megbízhatóság
SEQ_CHANNELS = 17 * 2 + 2 + 17


def fill_missing(kps: np.ndarray, min_conf: float = KP_MIN_CONF):
    """Hiányzó testpontok időbeli lineáris pótlása.

    Visszaad: xy (T, 17, 2) — a sosem látott testpont NaN —, valid (T, 17).
    """
    T = len(kps)
    xy = kps[..., :2].astype(np.float64).copy()
    valid = kps[..., 2] >= min_conf
    t = np.arange(T)
    for j in range(kps.shape[1]):
        v = valid[:, j]
        if v.sum() == 0:
            xy[:, j] = np.nan
        elif v.sum() < T:
            for d in range(2):
                xy[:, j, d] = np.interp(t, t[v], xy[v, j, d])
    return xy, valid


def _mid(xy, a, b):
    return (xy[:, a] + xy[:, b]) / 2


def body_scale(xy: np.ndarray, boxes: np.ndarray | None = None) -> float:
    """Törzshossz px-ben (váll-közép ↔ csípő-közép medián); ha nincs, a doboz-magasság 1/3-a."""
    torso = np.linalg.norm(_mid(xy, L_SH, R_SH) - _mid(xy, L_HIP, R_HIP), axis=1)
    s = float(np.nanmedian(torso)) if np.isfinite(torso).any() else float("nan")
    if not np.isfinite(s) or s < 1e-3:
        if boxes is not None and len(boxes):
            s = float(np.median(boxes[:, 3] - boxes[:, 1])) / 3
        else:
            s = 1.0
    return max(s, 1e-3)


def normalize(kps: np.ndarray, boxes: np.ndarray | None = None):
    """Testközpontú, törzshosszal osztott, felfelé pozitív koordináták.

    Visszaad: rel (T, 17, 2), center_motion (T, 2) a kezdőponthoz képest,
    conf (T, 17), scale (px).
    """
    xy, _valid = fill_missing(kps)
    scale = body_scale(xy, boxes)
    center = _mid(xy, L_HIP, R_HIP)
    if boxes is not None and len(boxes):
        box_c = np.stack([(boxes[:, 0] + boxes[:, 2]) / 2, (boxes[:, 1] + boxes[:, 3]) / 2], 1)
        bad = ~np.isfinite(center).all(1)
        center[bad] = box_c[bad]
    center = np.nan_to_num(center)
    rel = (xy - center[:, None, :]) / scale
    rel[..., 1] *= -1  # kép-y lefelé nő → felfelé pozitívra fordítjuk
    motion = (center - center[:1]) / scale
    motion[:, 1] *= -1
    return np.nan_to_num(rel), motion, kps[..., 2].astype(np.float64), scale


def resample(arr: np.ndarray, n: int = SEQ_LEN) -> np.ndarray:
    """Időbeli lineáris átmintázás n kockára (az első tengely az idő)."""
    T = len(arr)
    if T == 0:
        return np.zeros((n,) + arr.shape[1:])
    if T == 1:
        return np.repeat(arr, n, axis=0)
    src = np.linspace(0, T - 1, n)
    i0 = np.floor(src).astype(int)
    i1 = np.minimum(i0 + 1, T - 1)
    w = (src - i0).reshape((-1,) + (1,) * (arr.ndim - 1))
    return arr[i0] * (1 - w) + arr[i1] * w


def sequence(kps: np.ndarray, boxes: np.ndarray | None = None, n: int = SEQ_LEN) -> np.ndarray:
    """A neurális háló bemenete: (n, SEQ_CHANNELS) float32."""
    rel, motion, conf, _ = normalize(kps, boxes)
    T = len(kps)
    flat = np.concatenate([rel.reshape(T, -1), motion, conf], axis=1)
    return resample(flat, n).astype(np.float32)


def flip_sequence(seq: np.ndarray) -> np.ndarray:
    """Tükrözött sorozat (adatbővítéshez): x negálva, bal ↔ jobb testpontok cserélve."""
    out = seq.copy()
    pts = out[:, :34].reshape(len(out), 17, 2)[:, FLIP_PERM]
    pts[..., 0] *= -1
    out[:, :34] = pts.reshape(len(out), -1)
    out[:, 34] *= -1  # vízszintes elmozdulás
    out[:, 36:] = seq[:, 36:][:, FLIP_PERM]
    return out


def _angle(a, b, c):
    """A b csúcsú szög fokban (a-b-c), kockánként."""
    v1, v2 = a - b, c - b
    cos = (v1 * v2).sum(-1) / (np.linalg.norm(v1, axis=-1) * np.linalg.norm(v2, axis=-1) + 1e-9)
    return np.degrees(np.arccos(np.clip(cos, -1, 1)))


def _speed(p: np.ndarray, fps: float) -> np.ndarray:
    """Pont-pálya sebessége egység / s (kockánként, az első kocka 0)."""
    if len(p) < 2:
        return np.zeros(len(p))
    v = np.linalg.norm(np.diff(p, axis=0), axis=-1) * fps
    return np.concatenate([[0.0], v])


def _stats(prefix: str, x: np.ndarray) -> dict:
    if len(x) == 0:
        return {f"{prefix}_{k}": 0.0 for k in ("mean", "std", "min", "max")}
    return {f"{prefix}_mean": float(np.mean(x)), f"{prefix}_std": float(np.std(x)),
            f"{prefix}_min": float(np.min(x)), f"{prefix}_max": float(np.max(x))}


def clip_features(kps: np.ndarray, boxes: np.ndarray | None, fps: float) -> dict:
    """Klip-szintű, értelmezhető jellemzők (név → szám, rögzített sorrend)."""
    fps = fps or 30.0
    rel, motion, conf, _ = normalize(kps, boxes)
    T = len(kps)
    f: dict[str, float] = {
        "duration_s": T / fps,
        "kp_conf_mean": float(conf.mean()) if T else 0.0,
        "kp_valid_frac": float((conf >= KP_MIN_CONF).mean()) if T else 0.0,
    }
    for side, sh, el, wr in (("l", L_SH, L_EL, L_WR), ("r", R_SH, R_EL, R_WR)):
        h = rel[:, wr, 1] - rel[:, sh, 1]  # csukló a váll fölött (+)
        f.update(_stats(f"wrist_h_{side}", h))
        f[f"wrist_h_{side}_argmax"] = float(np.argmax(h) / max(1, T - 1)) if T else 0.0
        f.update(_stats(f"wrist_v_{side}", _speed(rel[:, wr], fps)))
        f.update(_stats(f"elbow_ang_{side}", _angle(rel[:, sh], rel[:, el], rel[:, wr])))
    both_h = np.maximum(rel[:, L_WR, 1] - rel[:, L_SH, 1], rel[:, R_WR, 1] - rel[:, R_SH, 1])
    f["wrist_h_max_both"] = float(both_h.max()) if T else 0.0
    head_y = rel[:, NOSE, 1]
    f["wrist_above_head_frac"] = float(
        (np.maximum(rel[:, L_WR, 1], rel[:, R_WR, 1]) > head_y).mean()) if T else 0.0
    # test-mozgás: vízszintes sebesség, függőleges kilengés (felugrás)
    f.update(_stats("center_v", _speed(motion, fps)))
    f["center_dy_range"] = float(np.ptp(motion[:, 1])) if T else 0.0
    f["center_dx_range"] = float(np.ptp(motion[:, 0])) if T else 0.0
    # váll-szélesség a törzshez: kicsi = oldalt áll a kamerának, nagy = szemben
    f.update(_stats("shoulder_w", np.abs(rel[:, L_SH, 0] - rel[:, R_SH, 0])))
    # törzs dőlése a függőlegestől (fok)
    sh_mid = (rel[:, L_SH] + rel[:, R_SH]) / 2
    lean = np.degrees(np.arctan2(np.abs(sh_mid[:, 0]), np.abs(sh_mid[:, 1]) + 1e-9))
    f.update(_stats("lean", lean))
    f.update(_stats("knee_ang_l", _angle(rel[:, L_HIP], rel[:, L_KN], rel[:, L_AN])))
    f.update(_stats("knee_ang_r", _angle(rel[:, R_HIP], rel[:, R_KN], rel[:, R_AN])))
    f["ankle_spread_max"] = float(np.abs(rel[:, L_AN, 0] - rel[:, R_AN, 0]).max()) if T else 0.0
    return f


def feature_matrix(clips: list[dict]) -> tuple[np.ndarray, list[str]]:
    rows = [clip_features(c["kps"], c.get("boxes"), c.get("fps", 30.0)) for c in clips]
    names = list(rows[0]) if rows else []
    X = np.array([[r[n] for n in names] for r in rows], np.float64)
    return np.nan_to_num(X), names


In [ ]:
%%writefile hbd/goalkeeper.py
"""Kapus-szempontú lövés-profil a testpont-sorozatból.

A kapus azt nézi: melyik kézzel lő, hol a lövőkar az elengedéskor (fent /
oldalt / lent), felugrik-e, merre fordul a teste. Ebből ítéli meg, hova
tud a lövő reálisan lőni. Ez a modul ezeket méri — CÍMKE NÉLKÜL, a
testpontokból. A hova-ment-a-lövés tanításához (következő lépés) ezek
lesznek a bemenet.

Ellenőrizhetőség: a felugrás-ítélet összevethető az adathalmaz címkéjével
(jump-shot = felugrásos, shot = helyből) — ezt a `train` meg is méri. A
kartartásra nincs címke; azt egy kis kézi mintán kell ellenőrizni.

Korlát: a "kamerához képest" szavak szó szerint értendők. A pálya-
koordinátás irányhoz (kapu felé fordulás) kamera-kalibráció kell.
"""

from __future__ import annotations

import numpy as np

from .features import L_HIP, L_SH, L_WR, NOSE, R_HIP, R_SH, R_WR, normalize

# Kartartás az elengedéskor: a csukló magassága a vállhoz képest, törzshosszban.
# Ennél magasabban → "fent" (a fej vonalában vagy fölötte).
ARM_HIGH_MIN = 0.35
# Ennél mélyebben → "lent" (csípő felé); a kettő között → "oldalt".
ARM_LOW_MAX = -0.25
# Felugrás: a csípő ennyi törzshossznyit emelkedik az elengedés körüli
# alapvonal fölé (a futás okozta mélységi elmozdulást a lineáris alapvonal
# kiszűri).
JUMP_MIN_RISE = 0.20
# Az elengedés körüli ablak (másodperc) a felugrás-méréshez — IDŐTARTAM,
# ezért másodpercben, nem kockában (a jelenetek 30 és 48 fps-esek).
JUMP_WINDOW_S = 0.6
# Az alapvonal két végének hossza (másodperc).
JUMP_EDGE_S = 0.12
# Kevés mintánál nincs ítélet (None), nem hallgatólagos alapérték.
MIN_FRAMES = 8
MIN_VALID_FRAC = 0.5


def _smooth(x: np.ndarray) -> np.ndarray:
    if len(x) < 3:
        return x
    return np.convolve(np.pad(x, 1, mode="edge"), np.ones(3) / 3, mode="valid")


def jump_rise(center_up: np.ndarray, release: int, fps: float) -> float:
    """A csípő legnagyobb emelkedése a lineáris alapvonal fölé az elengedés körül."""
    half = max(2, int(round(JUMP_WINDOW_S * fps)))
    edge = max(1, int(round(JUMP_EDGE_S * fps)))
    a = max(0, release - half)
    b = min(len(center_up), release + half + 1)
    seg = center_up[a:b]
    if len(seg) < 2 * edge + 1:
        return 0.0
    y0, y1 = seg[:edge].mean(), seg[-edge:].mean()
    base = np.linspace(y0, y1, len(seg))
    return float(np.max(seg - base))


def shot_profile(kps: np.ndarray, boxes: np.ndarray | None, fps: float) -> dict | None:
    """Egy lövés kapus-szempontú profilja; kevés/rossz mintánál None.

    Kulcsok:
        elengedes_kocka, elengedes_ido_s — a lövőkéz csuklójának leggyorsabb pillanata
        lovokez        — "jobb" / "bal" (a lövő SAJÁT keze, anatómiailag)
        kartartas      — "fent" / "oldalt" / "lent"
        kar_magassag   — a csukló a váll fölött, törzshosszban (+ = fölötte)
        felugras       — bool; csipo_emelkedes — törzshosszban
        vallszelesseg  — |bal váll − jobb váll| / törzs: kicsi = oldalt áll a kamerának
        torzs_doles_fok — a törzs dőlése a függőlegestől
        lovo_helye_px  — (x, y) a két boka közepe a jelenet képén (pálya-kalibrációhoz)
    """
    fps = fps or 30.0
    T = len(kps)
    if T < MIN_FRAMES:
        return None
    valid = kps[..., 2] >= 0.3
    arm_valid = valid[:, [L_SH, R_SH, L_WR, R_WR]].mean()
    if arm_valid < MIN_VALID_FRAC:
        return None
    rel, motion, _conf, _scale = normalize(kps, boxes)

    # Lövőkéz: a csukló testhez mért sebességének csúcsa a nagyobb.
    def wrist_speed(j):
        d = np.linalg.norm(np.diff(rel[:, j], axis=0), axis=1) * fps
        return _smooth(np.concatenate([[0.0], d]))

    vl, vr = wrist_speed(L_WR), wrist_speed(R_WR)
    right = vr.max() >= vl.max()
    wr, sh = (R_WR, R_SH) if right else (L_WR, L_SH)
    release = int(np.argmax(vr if right else vl))

    h = float(rel[release, wr, 1] - rel[release, sh, 1])
    if h >= ARM_HIGH_MIN:
        arm = "fent"
    elif h <= ARM_LOW_MAX:
        arm = "lent"
    else:
        arm = "oldalt"

    rise = jump_rise(motion[:, 1], release, fps)
    sh_w = float(abs(rel[release, L_SH, 0] - rel[release, R_SH, 0]))
    sh_mid = (rel[release, L_SH] + rel[release, R_SH]) / 2
    lean = float(np.degrees(np.arctan2(abs(sh_mid[0]), abs(sh_mid[1]) + 1e-9)))
    feet = kps[release, [15, 16], :2].mean(0)
    if not (kps[release, [15, 16], 2] >= 0.3).all() and boxes is not None and len(boxes) > release:
        b = boxes[release]
        feet = np.array([(b[0] + b[2]) / 2, b[3]])
    return {
        "elengedes_kocka": release,
        "elengedes_ido_s": round(release / fps, 3),
        "lovokez": "jobb" if right else "bal",
        "kartartas": arm,
        "kar_magassag": round(h, 3),
        "fej_folott": bool(rel[release, wr, 1] > rel[release, NOSE, 1]),
        "felugras": bool(rise >= JUMP_MIN_RISE),
        "csipo_emelkedes": round(rise, 3),
        "vallszelesseg": round(sh_w, 3),
        "torzs_doles_fok": round(lean, 1),
        "lovo_helye_px": (round(float(feet[0]), 1), round(float(feet[1]), 1)),
    }


__all__ = ["shot_profile", "jump_rise", "ARM_HIGH_MIN", "ARM_LOW_MAX", "JUMP_MIN_RISE",
           "L_HIP", "R_HIP"]


In [ ]:
%%writefile hbd/models.py
"""Kis időbeli konvolúciós háló a testpont-sorozatra (torch).

Miért ilyen kicsi? ~800 klipünk van: egy nagy háló bemagolná a tanító
részt. Két konvolúciós réteg + átlag/max-összevonás + adatbővítés
(tükrözés, kis nagyítás, időbeli eltolás) a reális méret.
"""

from __future__ import annotations

import numpy as np

from .features import SEQ_CHANNELS, flip_sequence


def _torch():
    import torch  # lusta import: a csomag többi része torch nélkül is fut

    return torch


def build_model(n_classes: int, channels: int = SEQ_CHANNELS, width: int = 64):
    torch = _torch()
    nn = torch.nn

    class TemporalCNN(nn.Module):
        def __init__(self):
            super().__init__()
            self.net = nn.Sequential(
                nn.Conv1d(channels, width, 5, padding=2), nn.BatchNorm1d(width), nn.ReLU(),
                nn.Conv1d(width, width * 2, 5, padding=2), nn.BatchNorm1d(width * 2), nn.ReLU(),
                nn.Conv1d(width * 2, width * 2, 3, padding=1), nn.BatchNorm1d(width * 2), nn.ReLU(),
            )
            self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(width * 4, n_classes))

        def forward(self, x):  # x: (B, T, C)
            h = self.net(x.transpose(1, 2))
            return self.head(torch.cat([h.mean(-1), h.amax(-1)], 1))

    return TemporalCNN()


def _augment(batch: np.ndarray, rng: np.random.Generator) -> np.ndarray:
    out = batch.copy()
    for i in range(len(out)):
        if rng.random() < 0.5:
            out[i] = flip_sequence(out[i])
        out[i, :, :36] *= rng.uniform(0.9, 1.1)
        shift = int(rng.integers(-2, 3))
        if shift:
            out[i] = np.roll(out[i], shift, axis=0)
    return out


class SeqClassifier:
    """sklearn-szerű burok: fit(X, y) / predict(X) / predict_proba(X), X: (N, T, C)."""

    def __init__(self, n_classes: int, epochs: int = 80, lr: float = 2e-3, batch: int = 32,
                 seed: int = 0, device: str | None = None):
        self.n_classes, self.epochs, self.lr, self.batch, self.seed = n_classes, epochs, lr, batch, seed
        torch = _torch()
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.model = None
        self.mu = self.sd = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        torch = _torch()
        torch.manual_seed(self.seed)
        rng = np.random.default_rng(self.seed)
        self.mu = X.reshape(-1, X.shape[-1]).mean(0)
        self.sd = X.reshape(-1, X.shape[-1]).std(0) + 1e-6
        self.model = build_model(self.n_classes, X.shape[-1]).to(self.device)
        counts = np.bincount(y, minlength=self.n_classes).astype(np.float64)
        w = counts.sum() / (self.n_classes * np.maximum(counts, 1))
        loss_fn = torch.nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=self.device))
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-3)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, self.epochs)
        self.model.train()
        for _ in range(self.epochs):
            order = rng.permutation(len(X))
            for i in range(0, len(X), self.batch):
                idx = order[i:i + self.batch]
                if len(idx) < 2:
                    continue
                xb = (_augment(X[idx], rng) - self.mu) / self.sd
                xb = torch.tensor(xb, dtype=torch.float32, device=self.device)
                yb = torch.tensor(y[idx], dtype=torch.long, device=self.device)
                opt.zero_grad()
                loss_fn(self.model(xb), yb).backward()
                opt.step()
            sched.step()
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        torch = _torch()
        self.model.eval()
        with torch.no_grad():
            xb = torch.tensor((X - self.mu) / self.sd, dtype=torch.float32, device=self.device)
            return torch.softmax(self.model(xb), -1).cpu().numpy()

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self.predict_proba(X).argmax(1)

    def save(self, path) -> None:
        torch = _torch()
        torch.save({"state": self.model.state_dict(), "mu": self.mu, "sd": self.sd,
                    "n_classes": self.n_classes}, path)


In [ ]:
%%writefile hbd/train.py
"""Keresztvalidált kiértékelés + végső modellek.

Feladatok (`dataset.TASKS`): binary (lövés / nem lövés), shot3 (felugrásos /
helyből / egyéb), all7 (mind a 7 kategória).

Modellek:
  majority — mindig a leggyakoribb osztály (a viszonyítási alap: ennél
             jobbnak KELL lennie mindennek, különben nem tanult semmit),
  logreg   — logisztikus regresszió a klip-jellemzőkön,
  hgb      — gradiens-erősítéses döntési fák a klip-jellemzőkön,
  tcn      — kis időbeli konvolúciós háló a testpont-sorozaton (torch).

Felosztás: StratifiedGroupKFold a JELENET szerint (ugyanaz a jelenet sosem
lehet egyszerre tanító és teszt). Mérőszámok: pontosság, kiegyensúlyozott
pontosság, makró-F1, osztályonkénti felidézés, tévesztési mátrix.

Kapus-profil: minden lövésre `goalkeeper.shot_profile`; a felugrás-ítélet
a címkével (jump-shot vs shot) összevetve is ki van értékelve.

    python -m hbd.train --poses poses.npz --out results/
"""

from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path

import numpy as np

from .dataset import TASKS
from .features import feature_matrix, sequence
from .goalkeeper import JUMP_MIN_RISE, shot_profile
from .pose import load_dataset

FOLDS = 5


def _sk_models(seed: int):
    from sklearn.dummy import DummyClassifier
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    return {
        "majority": lambda: DummyClassifier(strategy="most_frequent"),
        "logreg": lambda: make_pipeline(
            StandardScaler(), LogisticRegression(max_iter=3000, class_weight="balanced", C=0.5)),
        "hgb": lambda: HistGradientBoostingClassifier(
            max_iter=300, learning_rate=0.05, max_leaf_nodes=15, l2_regularization=1.0,
            class_weight="balanced", random_state=seed),
    }


def metrics(y_true: np.ndarray, y_pred: np.ndarray, n_classes: int) -> dict:
    from sklearn.metrics import (accuracy_score, balanced_accuracy_score, confusion_matrix,
                                 f1_score, recall_score)

    labels = list(range(n_classes))
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "macro_f1": float(f1_score(y_true, y_pred, labels=labels, average="macro", zero_division=0)),
        "recall_per_class": [float(v) for v in recall_score(
            y_true, y_pred, labels=labels, average=None, zero_division=0)],
        "confusion": confusion_matrix(y_true, y_pred, labels=labels).tolist(),
    }


def group_folds(y: np.ndarray, groups: np.ndarray, n_splits: int, seed: int):
    """Rétegzett, csoportos felosztás; kis adatnál kevesebb hajtással."""
    from sklearn.model_selection import StratifiedGroupKFold

    n_groups = len(np.unique(groups))
    min_class = np.bincount(y).min() if len(y) else 0
    k = max(2, min(n_splits, n_groups, max(2, min_class)))
    return list(StratifiedGroupKFold(n_splits=k, shuffle=True, random_state=seed).split(
        np.zeros(len(y)), y, groups))


def evaluate_task(task: str, X: np.ndarray, S: np.ndarray, cats: np.ndarray, groups: np.ndarray,
                  seed: int, use_nn: bool, nn_epochs: int, folds: int) -> dict:
    class_names, mapping = TASKS[task]
    y = np.array([mapping[c] for c in cats])
    present = np.unique(y)
    splits = group_folds(y, groups, folds, seed)
    model_makers = dict(_sk_models(seed))
    if use_nn:
        from .models import SeqClassifier

        model_makers["tcn"] = lambda: SeqClassifier(len(class_names), epochs=nn_epochs, seed=seed)

    out = {"classes": list(class_names), "n": int(len(y)),
           "class_counts": np.bincount(y, minlength=len(class_names)).tolist(),
           "folds": len(splits), "models": {}}
    for name, make in model_makers.items():
        pred = np.full(len(y), -1)
        for tr, te in splits:
            m = make()
            if name == "tcn":
                m.fit(S[tr], y[tr])
                pred[te] = m.predict(S[te])
            else:
                m.fit(X[tr], y[tr])
                pred[te] = m.predict(X[te])
        out["models"][name] = metrics(y, pred, len(class_names))
    out["present_classes"] = present.tolist()
    return out


def fit_final(task: str, X, S, cats, out_dir: Path, seed: int, use_nn: bool, nn_epochs: int,
              feature_names: list[str]) -> list[str]:
    """Végső modellek az ÖSSZES adaton (a Macen futtatáshoz)."""
    import joblib

    class_names, mapping = TASKS[task]
    y = np.array([mapping[c] for c in cats])
    saved = []
    hgb = _sk_models(seed)["hgb"]().fit(X, y)
    p = out_dir / f"model_{task}_hgb.joblib"
    joblib.dump({"model": hgb, "classes": list(class_names), "features": feature_names}, p)
    saved.append(p.name)
    if use_nn:
        from .models import SeqClassifier

        nn = SeqClassifier(len(class_names), epochs=nn_epochs, seed=seed).fit(S, y)
        p = out_dir / f"model_{task}_tcn.pt"
        nn.save(p)
        saved.append(p.name)
    return saved


def goalkeeper_report(clips: list[dict], out_dir: Path) -> dict:
    """Kapus-profil minden lövésre + a felugrás-ítélet összevetése a címkével."""
    rows, rises, labels = [], [], []
    for c in clips:
        if c["category"] not in ("jump-shot", "shot"):
            continue
        prof = shot_profile(c["kps"], c["boxes"], c["fps"])
        if prof is None:
            rows.append({"klip": c["name"], "kategoria": c["category"], "itelet": "kevés adat"})
            continue
        rows.append({"klip": c["name"], "kategoria": c["category"], **prof})
        rises.append(prof["csipo_emelkedes"])
        labels.append(int(c["category"] == "jump-shot"))
    if rows:
        keys = sorted({k for r in rows for k in r}, key=lambda k: (k not in ("klip", "kategoria"), k))
        with open(out_dir / "kapus_profilok.csv", "w", newline="") as fh:
            w = csv.DictWriter(fh, fieldnames=keys)
            w.writeheader()
            w.writerows(rows)
    rep: dict = {"loves_db": len(rows), "profil_db": len(rises)}
    if len(set(labels)) == 2:
        from sklearn.metrics import roc_auc_score

        r, lab = np.array(rises), np.array(labels)
        pred = (r >= JUMP_MIN_RISE).astype(int)
        rep["felugras_pontossag"] = float((pred == lab).mean())
        rep["felugras_felidezes_jumpshot"] = float(pred[lab == 1].mean())
        rep["felugras_teves_riasztas_shot"] = float(pred[lab == 0].mean())
        rep["felugras_auc"] = float(roc_auc_score(lab, r))
    prof_rows = [r for r in rows if "kartartas" in r]
    for key in ("kartartas", "lovokez"):
        cnt: dict[str, int] = {}
        for r in prof_rows:
            cnt[r[key]] = cnt.get(r[key], 0) + 1
        rep[f"{key}_eloszlas"] = cnt
    return rep


def _md_table(task_res: dict) -> str:
    lines = ["| modell | pontosság | kiegy. pontosság | makró-F1 | felidézés osztályonként |",
             "|---|---|---|---|---|"]
    for name, m in task_res["models"].items():
        rec = ", ".join(f"{c}: {v:.2f}" for c, v in zip(task_res["classes"], m["recall_per_class"]))
        lines.append(f"| {name} | {m['accuracy']:.3f} | {m['balanced_accuracy']:.3f} | "
                     f"{m['macro_f1']:.3f} | {rec} |")
    best = max((n for n in task_res["models"] if n != "majority"),
               key=lambda n: task_res["models"][n]["macro_f1"], default=None)
    if best:
        cm = task_res["models"][best]["confusion"]
        lines += ["", f"Tévesztési mátrix ({best}; sor = valódi, oszlop = jósolt):", "",
                  "| | " + " | ".join(task_res["classes"]) + " |",
                  "|---" * (len(cm) + 1) + "|"]
        for cname, row in zip(task_res["classes"], cm):
            lines.append(f"| **{cname}** | " + " | ".join(str(v) for v in row) + " |")
    return "\n".join(lines)


def write_report(results: dict, out_dir: Path) -> Path:
    md = ["# UNIRI-HBD — mozdulatfelismerés testpontokból", "",
          f"Klipek: {results['n_clips']}; felosztás: {results['folds']}-szörös, jelenet szerint "
          "csoportosítva (ugyanaz a jelenet sosem tanító és teszt egyszerre).", "",
          "A `majority` a viszonyítási alap: ami nem jobb nála, az nem tanult semmit.", ""]
    for task, res in results["tasks"].items():
        md += [f"## Feladat: {task}", "",
               "Osztályok és darabszámok: " + ", ".join(
                   f"{c} = {n}" for c, n in zip(res["classes"], res["class_counts"])), "",
               _md_table(res), ""]
    gk = results.get("goalkeeper", {})
    if gk:
        md += ["## Kapus-profil (lövésenként, címke nélkül)", "",
               f"Lövések: {gk.get('loves_db')}, ebből profil készült: {gk.get('profil_db')}.", ""]
        if "felugras_pontossag" in gk:
            md += [f"- Felugrás-ítélet a címkéhez mérve (jump-shot vs shot): pontosság "
                   f"{gk['felugras_pontossag']:.3f}, AUC {gk['felugras_auc']:.3f}; felugrásosból "
                   f"felismert {gk['felugras_felidezes_jumpshot']:.2f}, helyből lövésnél tévesen "
                   f"felugrásnak mondott {gk['felugras_teves_riasztas_shot']:.2f}."]
        md += [f"- Kartartás az elengedéskor: {gk.get('kartartas_eloszlas')}",
               f"- Lövőkéz: {gk.get('lovokez_eloszlas')}",
               "- A kartartásra NINCS címke az adathalmazban: kézi mintán kell ellenőrizni "
               "(`kapus_profilok.csv`).", ""]
    if results.get("saved_models"):
        md += ["## Mentett végső modellek", ""] + [f"- `{p}`" for p in results["saved_models"]]
    p = out_dir / "results.md"
    p.write_text("\n".join(md) + "\n")
    return p


def main(argv: list[str] | None = None) -> int:
    ap = argparse.ArgumentParser(description="UNIRI-HBD: kiértékelés és tanítás")
    ap.add_argument("--poses", required=True, help="a hbd.pose kimenete (.npz)")
    ap.add_argument("--out", required=True, help="kimeneti mappa")
    ap.add_argument("--tasks", default="binary,shot3,all7")
    ap.add_argument("--folds", type=int, default=FOLDS)
    ap.add_argument("--seed", type=int, default=0)
    ap.add_argument("--no-nn", action="store_true", help="a torch-háló kihagyása")
    ap.add_argument("--nn-epochs", type=int, default=80)
    ap.add_argument("--no-final", action="store_true", help="ne tanítson végső modellt")
    args = ap.parse_args(argv)

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    clips = [c for c in load_dataset(args.poses)["clips"] if len(c["kps"]) > 0]
    X, feat_names = feature_matrix(clips)
    S = np.stack([sequence(c["kps"], c["boxes"]) for c in clips])
    cats = np.array([c["category"] for c in clips])
    groups = np.array([c["scene_id"] for c in clips])
    use_nn = not args.no_nn
    if use_nn:
        try:
            import torch  # noqa: F401
        except ImportError:
            use_nn = False

    results = {"n_clips": len(clips), "folds": args.folds, "features": feat_names, "tasks": {}}
    saved = []
    for task in [t for t in args.tasks.split(",") if t]:
        print(f"== {task} ==", flush=True)
        res = evaluate_task(task, X, S, cats, groups, args.seed, use_nn, args.nn_epochs, args.folds)
        results["tasks"][task] = res
        for name, m in res["models"].items():
            print(f"  {name:9s} pontosság {m['accuracy']:.3f}  kiegy. {m['balanced_accuracy']:.3f}"
                  f"  makró-F1 {m['macro_f1']:.3f}", flush=True)
        if not args.no_final:
            saved += fit_final(task, X, S, cats, out, args.seed, use_nn, args.nn_epochs, feat_names)
    results["goalkeeper"] = goalkeeper_report(clips, out)
    results["saved_models"] = saved
    (out / "results.json").write_text(json.dumps(results, ensure_ascii=False, indent=1))
    p = write_report(results, out)
    print(f"Jelentés: {p}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile hbd/release.py
"""Tanított lövés-profil: elengedés pillanata, kartartás, lövőkéz — kézi jelölésből.

A `goalkeeper` kézi küszöbei (a "leggyorsabb csukló" kockája) rosszul
találták meg az elengedést, ezért a kartartást is elrontották (a lövések
felét "lent"-nek mondták, a jelölés szerint 337-ből 2 az). Itt ugyanezt a
gép tanulja meg a lövésjelölő oldalon készült kézi jelölésekből:

  1. ELENGEDÉS: kis időbeli konvolúciós háló kockánként pontoz, a legjobb
     kocka az elengedés (tanítás: a jelölt kocka körüli puha célfüggvény).
  2. KARTARTÁS (fent / oldalt) és LÖVŐKÉZ (jobb / bal): logisztikus
     regresszió a testtartás-jellemzőkön az elengedés körüli ablakban.

Kiértékelés: jelenet szerint csoportosított keresztvalidáció, "végponttól
végpontig" — a teszt-részen a kartartást a MODELL által talált elengedésnél
mérjük, nem a kézi jelölésnél (úgy, ahogy élesben futna). Viszonyítás: a
régi szabály és a "mindig a leggyakoribb" tipp.

    python -m hbd.release --poses poses.npz --labels cimkek.csv --out results_release/
"""

from __future__ import annotations

import argparse
import csv
import json
from pathlib import Path

import numpy as np

from .features import (L_EL, L_SH, L_WR, NOSE, R_EL, R_SH, R_WR, normalize)
from .goalkeeper import shot_profile
from .pose import load_dataset

# Az elengedés körüli ablak a kartartás-jellemzőkhöz — IDŐTARTAM, ezért
# másodpercben (a jelenetek 30 és 48 fps-esek).
ARM_WINDOW_S = 0.08
# A puha célfüggvény szélessége (másodperc): a jelölés ±1-2 kockás
# bizonytalanságát nem büntetjük teljes hibaként.
TARGET_SIGMA_S = 0.03
# Kartartás-osztályok: a "lent" a jelölésben 337-ből 2 — ennyiből nem
# tanulható, az "oldalt"-hoz soroljuk (mindkettő: a kar a fej vonala alatt).
ARM_CLASSES = ("fent", "oldalt")
HAND_CLASSES = ("jobb", "bal")
FOLDS = 5


# ---------------------------------------------------------------- jelölések
def load_labels(path: str | Path) -> dict[str, dict]:
    """A jelölések CSV-ből (klip → sor); a "rossz" klipek kimaradnak."""
    out = {}
    with open(path) as fh:
        for r in csv.DictReader(fh):
            if r["rossz"].lower() in ("true", "1"):
                continue
            out[r["klip"]] = {"elengedes": int(r["elengedes"]), "kartartas": r["kartartas"],
                              "lovokez": r["lovokez"], "kategoria": r["kategoria"]}
    return out


# --------------------------------------------------------------- jellemzők
def frame_features(kps: np.ndarray, boxes: np.ndarray, fps: float) -> np.ndarray:
    """Kockánkénti bemenet az elengedés-hálónak: (T, F) float32.

    Testpontok a csípőhöz és törzshosszhoz normálva, a csuklók sebessége
    (egység / s) és magassága a vállhoz, plusz a test elmozdulása.
    """
    fps = fps or 30.0
    rel, motion, conf, _ = normalize(kps, boxes)
    T = len(kps)
    vel = np.zeros((T, 17, 2))
    if T > 1:
        vel[1:] = np.diff(rel, axis=0) * fps
    wr_speed = np.linalg.norm(vel[:, [L_WR, R_WR]], axis=-1) / 10.0
    wr_h = np.stack([rel[:, L_WR, 1] - rel[:, L_SH, 1], rel[:, R_WR, 1] - rel[:, R_SH, 1]], 1)
    mvel = np.zeros((T, 2))
    if T > 1:
        mvel[1:] = np.diff(motion, axis=0) * fps / 10.0
    return np.concatenate([rel.reshape(T, -1), vel.reshape(T, -1) / 10.0, wr_speed, wr_h,
                           motion, mvel, conf], 1).astype(np.float32)


def arm_hand_features(kps: np.ndarray, boxes: np.ndarray, fps: float, r: int) -> np.ndarray:
    """Testtartás az elengedés (r) körüli ablakban → jellemző-vektor."""
    fps = fps or 30.0
    rel, _, _, _ = normalize(kps, boxes)
    T = len(kps)
    w = max(1, int(round(ARM_WINDOW_S * fps)))
    a, b = max(0, r - w), min(T, r + w + 1)
    seg = rel[a:b]
    # a törzs "fel" iránya (csípő-közép → váll-közép); a kar szöge ehhez mérve:
    # fent ≈ 0–40°, oldalt ≈ 60–110° (a "sidearm" dobás lényege a szög, nem a magasság)
    up = (seg[:, L_SH] + seg[:, R_SH]) / 2
    up = up / (np.linalg.norm(up, axis=1, keepdims=True) + 1e-9)

    def ang(v):
        v = v / (np.linalg.norm(v, axis=1, keepdims=True) + 1e-9)
        return np.degrees(np.arccos(np.clip((v * up).sum(1), -1, 1)))

    f = []
    for wr, sh, el in ((L_WR, L_SH, L_EL), (R_WR, R_SH, R_EL)):
        h = seg[:, wr, 1] - seg[:, sh, 1]
        head = seg[:, wr, 1] - seg[:, NOSE, 1]
        side = np.abs(seg[:, wr, 0] - seg[:, sh, 0])
        elb = seg[:, el, 1] - seg[:, sh, 1]
        a_arm = ang(seg[:, wr] - seg[:, sh])
        a_upper = ang(seg[:, el] - seg[:, sh])
        f += [h.max(), h.mean(), head.max(), side.mean(), elb.max(),
              a_arm.min(), a_arm.mean(), a_upper.min(), a_upper.mean()]
    # a teljes klipen a csuklók legnagyobb magassága és sebessége (a lövőkéz
    # a lendítés során magasabbra és gyorsabban mozog)
    hl = rel[:, L_WR, 1] - rel[:, L_SH, 1]
    hr = rel[:, R_WR, 1] - rel[:, R_SH, 1]
    vl = np.linalg.norm(np.diff(rel[:, L_WR], axis=0), axis=1) * fps if T > 1 else np.zeros(1)
    vr = np.linalg.norm(np.diff(rel[:, R_WR], axis=0), axis=1) * fps if T > 1 else np.zeros(1)
    f += [hl.max() - hr.max(), np.percentile(vl, 90) - np.percentile(vr, 90),
          np.abs(seg[:, L_SH, 0] - seg[:, R_SH, 0]).mean()]
    return np.nan_to_num(np.array(f, np.float64))


# ------------------------------------------------------------ elengedés-háló
def _torch():
    import torch

    return torch


class ReleaseNet:
    """Kockánként pontozó időbeli konvolúciós háló; predict → elengedés kockája."""

    def __init__(self, epochs: int = 120, lr: float = 2e-3, seed: int = 0, width: int = 64):
        self.epochs, self.lr, self.seed, self.width = epochs, lr, seed, width
        self.model = None
        self.mu = self.sd = None
        torch = _torch()
        self.device = "cuda" if torch.cuda.is_available() else "cpu"

    def _build(self, f_in: int):
        torch = _torch()
        nn = torch.nn
        w = self.width
        return nn.Sequential(
            nn.Conv1d(f_in, w, 5, padding=2), nn.ReLU(),
            nn.Conv1d(w, w, 5, padding=4, dilation=2), nn.ReLU(),
            nn.Conv1d(w, w, 5, padding=8, dilation=4), nn.ReLU(),
            nn.Dropout(0.2), nn.Conv1d(w, 1, 1),
        ).to(self.device)

    def _batch(self, X: list[np.ndarray]):
        torch = _torch()
        T = max(len(x) for x in X)
        xb = np.zeros((len(X), T, X[0].shape[1]), np.float32)
        mask = np.zeros((len(X), T), bool)
        for i, x in enumerate(X):
            xb[i, :len(x)] = (x - self.mu) / self.sd
            mask[i, :len(x)] = True
        return (torch.tensor(xb, device=self.device).transpose(1, 2),
                torch.tensor(mask, device=self.device))

    def fit(self, X: list[np.ndarray], y: list[int], fps: list[float]):
        torch = _torch()
        torch.manual_seed(self.seed)
        rng = np.random.default_rng(self.seed)
        allx = np.concatenate(X)
        self.mu, self.sd = allx.mean(0), allx.std(0) + 1e-6
        self.model = self._build(allx.shape[1])
        opt = torch.optim.AdamW(self.model.parameters(), lr=self.lr, weight_decay=1e-3)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, self.epochs)
        self.model.train()
        for _ in range(self.epochs):
            for idx in np.array_split(rng.permutation(len(X)), max(1, len(X) // 16)):
                xb, mask = self._batch([X[i] for i in idx])
                logits = self.model(xb).squeeze(1).masked_fill(~mask, -1e9)
                tgt = np.zeros(mask.shape, np.float32)
                t = np.arange(mask.shape[1])
                for j, i in enumerate(idx):
                    sig = max(0.5, TARGET_SIGMA_S * fps[i])
                    g = np.exp(-0.5 * ((t - y[i]) / sig) ** 2) * mask[j].cpu().numpy()
                    tgt[j] = g / g.sum()
                tgt_t = torch.tensor(tgt, device=self.device)
                loss = -(tgt_t * torch.log_softmax(logits, 1)).sum(1).mean()
                opt.zero_grad()
                loss.backward()
                opt.step()
            sched.step()
        return self

    def predict(self, X: list[np.ndarray]) -> np.ndarray:
        torch = _torch()
        self.model.eval()
        out = []
        with torch.no_grad():
            for x in X:
                xb, mask = self._batch([x])
                out.append(int(self.model(xb).squeeze(1)[0, :len(x)].argmax()))
        return np.array(out)

    def save(self, path) -> None:
        torch = _torch()
        torch.save({"state": self.model.state_dict(), "mu": self.mu, "sd": self.sd,
                    "width": self.width}, path)


# ---------------------------------------------------------------- kiértékelés
def _clf():
    from sklearn.linear_model import LogisticRegression
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    return make_pipeline(StandardScaler(),
                         LogisticRegression(max_iter=2000, class_weight="balanced", C=0.5))


def evaluate(clips: list[dict], labels: dict[str, dict], seed: int = 0, epochs: int = 120) -> dict:
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.model_selection import StratifiedGroupKFold

    data = [c for c in clips if c["name"] in labels and len(c["kps"]) >= 8]
    y_rel = np.array([min(labels[c["name"]]["elengedes"], len(c["kps"]) - 1) for c in data])
    y_arm = np.array([int(labels[c["name"]]["kartartas"] != "fent") for c in data])
    y_hand = np.array([int(labels[c["name"]]["lovokez"] == "bal") for c in data])
    fps = np.array([c["fps"] for c in data])
    groups = np.array([c["scene_id"] for c in data])
    F = [frame_features(c["kps"], c["boxes"], c["fps"]) for c in data]

    pred_rel = np.zeros(len(data), int)
    pred_arm = np.zeros(len(data), int)
    pred_hand = np.zeros(len(data), int)
    oracle_arm = np.zeros(len(data), int)
    strat = y_arm * 2 + y_hand
    for tr, te in StratifiedGroupKFold(FOLDS, shuffle=True, random_state=seed).split(
            np.zeros(len(data)), strat, groups):
        net = ReleaseNet(epochs=epochs, seed=seed).fit([F[i] for i in tr], y_rel[tr], fps[tr])
        pred_rel[te] = net.predict([F[i] for i in te])
        Xtr = np.stack([arm_hand_features(data[i]["kps"], data[i]["boxes"], fps[i], y_rel[i]) for i in tr])
        Xte_pred = np.stack([arm_hand_features(data[i]["kps"], data[i]["boxes"], fps[i], pred_rel[i]) for i in te])
        Xte_orc = np.stack([arm_hand_features(data[i]["kps"], data[i]["boxes"], fps[i], y_rel[i]) for i in te])
        arm = _clf().fit(Xtr, y_arm[tr])
        hand = _clf().fit(Xtr, y_hand[tr])
        pred_arm[te] = arm.predict(Xte_pred)
        oracle_arm[te] = arm.predict(Xte_orc)
        pred_hand[te] = hand.predict(Xte_pred)

    # a régi kézi szabály ugyanezeken a klipeken
    rule_rel, rule_arm, rule_hand = [], [], []
    for c in data:
        p = shot_profile(c["kps"], c["boxes"], c["fps"])
        rule_rel.append(p["elengedes_kocka"] if p else len(c["kps"]) // 2)
        rule_arm.append(int(not p or p["kartartas"] != "fent"))
        rule_hand.append(int(bool(p) and p["lovokez"] == "bal"))
    rule_rel, rule_arm, rule_hand = map(np.array, (rule_rel, rule_arm, rule_hand))

    def rel_stats(pred):
        err_f = np.abs(pred - y_rel)
        err_ms = err_f / fps * 1000
        return {"atlag_hiba_kocka": float(err_f.mean()), "median_hiba_ms": float(np.median(err_ms)),
                "pontos_2_kockan_belul": float((err_f <= 2).mean()),
                "pontos_100ms_belul": float((err_ms <= 100).mean())}

    def cls_stats(pred, y):
        return {"pontossag": float((pred == y).mean()),
                "kiegyensulyozott": float(balanced_accuracy_score(y, pred))}

    return {
        "klipek": len(data),
        "osztalyok": {"kartartas": {"fent": int((y_arm == 0).sum()), "oldalt/lent": int(y_arm.sum())},
                      "lovokez": {"jobb": int((y_hand == 0).sum()), "bal": int(y_hand.sum())}},
        "elengedes": {"modell": rel_stats(pred_rel), "regi_szabaly": rel_stats(rule_rel)},
        "kartartas": {"modell": cls_stats(pred_arm, y_arm),
                      "modell_kezi_elengedesnel": cls_stats(oracle_arm, y_arm),
                      "regi_szabaly": cls_stats(rule_arm, y_arm),
                      "mindig_fent": cls_stats(np.zeros_like(y_arm), y_arm)},
        "lovokez": {"modell": cls_stats(pred_hand, y_hand),
                    "regi_szabaly": cls_stats(rule_hand, y_hand),
                    "mindig_jobb": cls_stats(np.zeros_like(y_hand), y_hand)},
    }


def fit_final(clips, labels, out: Path, seed: int = 0, epochs: int = 120) -> list[str]:
    import joblib

    data = [c for c in clips if c["name"] in labels and len(c["kps"]) >= 8]
    y_rel = np.array([min(labels[c["name"]]["elengedes"], len(c["kps"]) - 1) for c in data])
    fps = np.array([c["fps"] for c in data])
    F = [frame_features(c["kps"], c["boxes"], c["fps"]) for c in data]
    ReleaseNet(epochs=epochs, seed=seed).fit(F, y_rel, fps).save(out / "release_net.pt")
    X = np.stack([arm_hand_features(c["kps"], c["boxes"], c["fps"], r) for c, r in zip(data, y_rel)])
    y_arm = np.array([int(labels[c["name"]]["kartartas"] != "fent") for c in data])
    y_hand = np.array([int(labels[c["name"]]["lovokez"] == "bal") for c in data])
    joblib.dump({"model": _clf().fit(X, y_arm), "classes": ARM_CLASSES}, out / "arm_clf.joblib")
    joblib.dump({"model": _clf().fit(X, y_hand), "classes": HAND_CLASSES}, out / "hand_clf.joblib")
    return ["release_net.pt", "arm_clf.joblib", "hand_clf.joblib"]


def write_report(res: dict, out: Path) -> Path:
    e, k, h = res["elengedes"], res["kartartas"], res["lovokez"]
    pct = lambda v: f"{100 * v:.1f}%"  # noqa: E731
    md = [
        "# Tanított lövés-profil (kézi jelölésből)", "",
        f"Klipek: {res['klipek']} (a „rossz” klipek nélkül); 5-szörös, jelenet szerint "
        "csoportosított keresztvalidáció. A kartartást és a lövőkezet a MODELL által talált "
        "elengedésnél mérjük, ahogy élesben futna.", "",
        f"Osztályok: kartartás {res['osztalyok']['kartartas']}, lövőkéz {res['osztalyok']['lovokez']}.", "",
        "## Elengedés pillanata", "",
        "| | átlagos hiba (kocka) | medián hiba | ±2 kockán belül | 100 ms-on belül |",
        "|---|---|---|---|---|",
    ]
    for name, key in (("tanított modell", "modell"), ("régi szabály", "regi_szabaly")):
        s = e[key]
        md.append(f"| {name} | {s['atlag_hiba_kocka']:.1f} | {s['median_hiba_ms']:.0f} ms | "
                  f"{pct(s['pontos_2_kockan_belul'])} | {pct(s['pontos_100ms_belul'])} |")
    md += ["", "## Kartartás (fent / oldalt)", "", "| | pontosság | kiegyensúlyozott |", "|---|---|---|"]
    for name, key in (("tanított modell", "modell"), ("modell a kézi elengedésnél", "modell_kezi_elengedesnel"),
                      ("régi szabály", "regi_szabaly"), ("mindig „fent”", "mindig_fent")):
        md.append(f"| {name} | {pct(k[key]['pontossag'])} | {pct(k[key]['kiegyensulyozott'])} |")
    md += ["", "## Lövőkéz (jobb / bal)", "", "| | pontosság | kiegyensúlyozott |", "|---|---|---|"]
    for name, key in (("tanított modell", "modell"), ("régi szabály", "regi_szabaly"),
                      ("mindig „jobb”", "mindig_jobb")):
        md.append(f"| {name} | {pct(h[key]['pontossag'])} | {pct(h[key]['kiegyensulyozott'])} |")
    p = out / "release_results.md"
    p.write_text("\n".join(md) + "\n")
    return p


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description="Tanított lövés-profil a kézi jelölésből")
    ap.add_argument("--poses", required=True)
    ap.add_argument("--labels", required=True, help="a lövésjelölő CSV-je")
    ap.add_argument("--out", required=True)
    ap.add_argument("--epochs", type=int, default=120)
    ap.add_argument("--seed", type=int, default=0)
    args = ap.parse_args(argv)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    clips = load_dataset(args.poses)["clips"]
    labels = load_labels(args.labels)
    res = evaluate(clips, labels, args.seed, args.epochs)
    res["mentett_modellek"] = fit_final(clips, labels, out, args.seed, args.epochs)
    (out / "release_results.json").write_text(json.dumps(res, ensure_ascii=False, indent=1))
    print(write_report(res, out).read_text())
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile hbd/match_pose.py
"""Teljes meccs: minden (ritkított) kockán az összes ember testpontja.

A lövésfelismerőnek a meccs minden pillanatáról kell tudnia, mi látszik:
hány ember van a képen, felemelt dobókar van-e valakinél, mozog-e a kamera.
Ez a modul egy meccsfelvételből kockánként (STRIDE-onként) lefuttatja a
YOLO-pose-t a teljes képen, és tömören elmenti:

  t        — a kocka ideje (s)
  boxes    — emberek dobozai (n, 4), a kép szélességével/magasságával osztva (0..1)
  kps      — testpontok (n, 17, 3): x, y (0..1), megbízhatóság
  motion   — a kép változása az előző mintához képest (kamera-mozgás / vágás jelző)

Követés (ki kicsoda a kockák között) NINCS benne: ritkított mintán a
kézilabda gyors mozgásánál megbízhatatlan. A lövésfelismerő első változata
kocka-szintű összesítésekkel dolgozik; a követéses változat GPU-n, teljes
kockaszámmal futhat (ugyanezzel a modullal, --stride 1).

Megszakítás után folytatható (darabonkénti gyorsítótár).

    python -m hbd.match_pose --video meccs.mp4 --out meccs_pose.npz --stride 5 --imgsz 1280
"""

from __future__ import annotations

import argparse
import subprocess
import time
from pathlib import Path

import numpy as np

from .video import ffmpeg_exe, probe

# Széles (közvetítési) képen a kicsi, távoli játékosokhoz nagy bemenet kell:
# 640-en ~3, 960-on ~7, 1280-on ~18 embert talál ugyanazon a kockán.
DEFAULT_IMGSZ = 1280
DEFAULT_STRIDE = 5
CHUNK = 1500          # ennyi mintakocka kerül egy gyorsítótár-darabba
MOTION_SIZE = (64, 36)  # a kamera-mozgás jelzőhöz kicsinyített szürke kép


def iter_sampled(video: str, stride: int, w: int, h: int, start_frame: int = 0):
    """(kocka-index, BGR kép) minden stride-adik kockára, a start_frame-től."""
    fps = probe(video)[2] or 25.0
    ss = start_frame / fps
    cmd = [ffmpeg_exe(), "-v", "error"]
    if ss > 0:
        cmd += ["-ss", f"{ss:.3f}"]
    cmd += ["-i", video, "-vf", f"select='not(mod(n\\,{stride}))'", "-vsync", "0",
            "-f", "rawvideo", "-pix_fmt", "bgr24", "-"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    size = w * h * 3
    k = 0
    try:
        while True:
            buf = proc.stdout.read(size)
            if len(buf) < size:
                break
            yield start_frame + k * stride, np.frombuffer(buf, np.uint8).reshape(h, w, 3)
            k += 1
    finally:
        proc.stdout.close()
        proc.kill()
        proc.wait()


def run(video: str, out: Path, stride: int, imgsz: int, weights: str, device: str | None,
        batch: int = 8, conf: float = 0.1) -> Path:
    import cv2
    from ultralytics import YOLO

    w, h, fps = probe(video)
    fps = fps or 25.0
    cache = out.parent / (out.stem + "_cache")
    cache.mkdir(parents=True, exist_ok=True)
    model = YOLO(weights)
    done = sorted(cache.glob("chunk_*.npz"))
    start_frame = 0
    if done:
        last = np.load(done[-1])
        start_frame = int(last["frame"][-1]) + stride
    print(f"{video}: {w}x{h} @ {fps:.2f} fps; minta minden {stride}. kocka; indulás: {start_frame}", flush=True)

    buf_frames, buf_idx = [], []
    rec = {"frame": [], "n": [], "boxes": [], "kps": [], "motion": []}
    prev_small = None
    t0 = time.time()
    chunk_id = len(done)

    def flush_batch():
        nonlocal prev_small
        res = model.predict(buf_frames, imgsz=imgsz, conf=conf, device=device, verbose=False)
        for fi, img, r in zip(buf_idx, buf_frames, res):
            small = cv2.resize(cv2.cvtColor(img, cv2.COLOR_BGR2GRAY), MOTION_SIZE).astype(np.float32)
            mot = float(np.abs(small - prev_small).mean()) if prev_small is not None else 0.0
            prev_small = small
            if r.keypoints is not None and len(r.boxes):
                b = r.boxes.xyxy.cpu().numpy() / [w, h, w, h]
                xy = r.keypoints.xy.cpu().numpy() / [w, h]
                kc = r.keypoints.conf
                kc = kc.cpu().numpy() if kc is not None else np.ones(xy.shape[:2])
                k = np.concatenate([xy, kc[..., None]], -1)
            else:
                b, k = np.zeros((0, 4)), np.zeros((0, 17, 3))
            rec["frame"].append(fi)
            rec["n"].append(len(b))
            rec["boxes"].append(b.astype(np.float16))
            rec["kps"].append(k.astype(np.float16))
            rec["motion"].append(mot)
        buf_frames.clear()
        buf_idx.clear()

    def save_chunk():
        nonlocal chunk_id
        if not rec["frame"]:
            return
        np.savez_compressed(
            cache / f"chunk_{chunk_id:04d}.npz",
            frame=np.array(rec["frame"], np.int64), n=np.array(rec["n"], np.int32),
            boxes=np.concatenate(rec["boxes"]) if sum(rec["n"]) else np.zeros((0, 4), np.float16),
            kps=np.concatenate(rec["kps"]) if sum(rec["n"]) else np.zeros((0, 17, 3), np.float16),
            motion=np.array(rec["motion"], np.float32))
        chunk_id += 1
        for v in rec.values():
            v.clear()

    count = 0
    for fi, img in iter_sampled(video, stride, w, h, start_frame):
        buf_frames.append(img)
        buf_idx.append(fi)
        if len(buf_frames) == batch:
            flush_batch()
        count += 1
        if len(rec["frame"]) >= CHUNK:
            save_chunk()
            el = time.time() - t0
            print(f"  {fi / fps / 60:5.1f}. perc · {count} minta · {el / count * 1000:.0f} ms/minta", flush=True)
    if buf_frames:
        flush_batch()
    save_chunk()
    return combine(cache, out, fps, stride, (w, h))


def combine(cache: Path, out: Path, fps: float, stride: int, size) -> Path:
    parts = [np.load(f) for f in sorted(cache.glob("chunk_*.npz"))]
    n = np.concatenate([p["n"] for p in parts])
    np.savez_compressed(
        out, frame=np.concatenate([p["frame"] for p in parts]), n=n,
        offsets=np.concatenate([[0], np.cumsum(n)]).astype(np.int64),
        boxes=np.concatenate([p["boxes"] for p in parts]), kps=np.concatenate([p["kps"] for p in parts]),
        motion=np.concatenate([p["motion"] for p in parts]), fps=np.float32(fps), stride=np.int32(stride),
        size=np.array(size, np.int32))
    print(f"Kész: {len(n)} mintakocka, átlag {n.mean():.1f} ember/kocka → {out}")
    return out


def load(path: str | Path) -> dict:
    z = np.load(path)
    d = {k: z[k] for k in z.files}
    d["t"] = d["frame"] / float(d["fps"])
    return d


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description="Teljes meccs: testpontok minden ritkított kockán")
    ap.add_argument("--video", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--stride", type=int, default=DEFAULT_STRIDE)
    ap.add_argument("--imgsz", type=int, default=DEFAULT_IMGSZ)
    ap.add_argument("--weights", default="yolo11n-pose.pt")
    ap.add_argument("--device", default=None)
    ap.add_argument("--batch", type=int, default=8)
    args = ap.parse_args(argv)
    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    run(args.video, out, args.stride, args.imgsz, args.weights, args.device, args.batch)
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile hbd/broadcast.py
"""A közvetítés saját jelei: eredményjelző és kamera-mozgás.

A tévés közvetítés többet tud a meccsről, mint amit a játékosok testtartása
mutat. Két ilyen jel, ami a lövésfelismerésnek segít:

  eredményjelző — a bal felső sarokban. GÓL után az állás megváltozik;
                  ISMÉTLÉS alatt (és sok közelinél) a grafika eltűnik.
  kamera        — a vezető kamera követi a játékot: támadásépítésnél lassan
                  úszik, lövés után megáll vagy MEGFORDUL (indul a másik
                  irányba a játék), vágásnál a kép hirtelen kicserélődik.

Ugyanazokat a mintakockákat nézi, mint a `match_pose` (minden STRIDE-adik),
így a két kimenet kockánként összefésülhető.

Az eredményjelző helye a közvetítés grafikájától függ; az alapérték az EHF
Bajnokok Ligája közvetítéseié (a kép arányában megadva, bármilyen
felbontásra jó). Más grafikánál: --sb-static / --sb-score.

    python -m hbd.broadcast --video meccs.mp4 --out meccs_broadcast.npz --stride 5
"""

from __future__ import annotations

import argparse
import subprocess
import time
from pathlib import Path

import numpy as np

from .video import ffmpeg_exe, probe

# A feldolgozó kép (szürke). Az eredményjelző számjegyei ezen ~20×25 pixelesek.
WORK = (640, 360)
# A kamera-becsléshez tovább kicsinyített kép (fázis-korreláció).
CAM = (160, 90)
# Az eredményjelző részei a kép arányában (x0, y0, x1, y1).
#   STATIC: a logó és a csapatnevek — ez mindig ugyanúgy néz ki, ha látszik;
#   SCORE:  az állás két számjegye — ez változik gólnál.
SB_STATIC = (0.090, 0.050, 0.186, 0.117)
SB_SCORE = (0.189, 0.050, 0.222, 0.117)
# Az eredményjelző "látszik", ha az állandó része ennyire hasonlít a mintára (NCC).
SB_PRESENT_NCC = 0.80
# A kamera-elmozdulás becslése ennél gyengébb csúcsnál megbízhatatlan (vágás, elmosódás).
CAM_MIN_RESPONSE = 0.05


def _box(frac, w, h):
    x0, y0, x1, y1 = frac
    return int(round(x0 * w)), int(round(y0 * h)), int(round(x1 * w)), int(round(y1 * h))


def iter_gray(video: str, stride: int):
    """(kocka-index, szürke WORK-méretű kép) minden stride-adik kockára."""
    w, h = WORK
    cmd = [ffmpeg_exe(), "-v", "error", "-i", video, "-vf",
           f"select='not(mod(n\\,{stride}))',scale={w}:{h},format=gray", "-vsync", "0",
           "-f", "rawvideo", "-pix_fmt", "gray", "-"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    k = 0
    try:
        while True:
            buf = proc.stdout.read(w * h)
            if len(buf) < w * h:
                break
            yield k * stride, np.frombuffer(buf, np.uint8).reshape(h, w)
            k += 1
    finally:
        proc.stdout.close()
        proc.kill()
        proc.wait()


def camera_shift(prev: np.ndarray, cur: np.ndarray, win: np.ndarray) -> tuple[float, float, float]:
    """A kép elmozdulása (dx, dy a CAM-kép szélességének/magasságának arányában) és a csúcs ereje."""
    import cv2

    (dx, dy), resp = cv2.phaseCorrelate(prev, cur, win)
    return dx / CAM[0], dy / CAM[1], float(resp)


def extract(video: str, stride: int, sb_static=SB_STATIC, sb_score=SB_SCORE) -> dict:
    """Nyers jelek mintakockánként: eredményjelző-kivágások + kamera-elmozdulás."""
    import cv2

    w, h = WORK
    st, sc = _box(sb_static, w, h), _box(sb_score, w, h)
    win = cv2.createHanningWindow(CAM, cv2.CV_32F)
    frames, static, score, cam = [], [], [], []
    prev = None
    t0 = time.time()
    fps = probe(video)[2] or 25.0
    for fi, g in iter_gray(video, stride):
        frames.append(fi)
        static.append(g[st[1]:st[3], st[0]:st[2]].copy())
        score.append(g[sc[1]:sc[3], sc[0]:sc[2]].copy())
        small = cv2.resize(g, CAM, interpolation=cv2.INTER_AREA).astype(np.float32)
        cam.append(camera_shift(prev, small, win) if prev is not None else (0.0, 0.0, 0.0))
        prev = small
        if len(frames) % 3000 == 0:
            print(f"  {fi / fps / 60:5.1f}. perc · {len(frames)} minta · "
                  f"{(time.time() - t0) / len(frames) * 1000:.1f} ms/minta", flush=True)
    return {"frame": np.array(frames, np.int64), "sb_static": np.stack(static), "sb_score": np.stack(score),
            "cam": np.array(cam, np.float32), "fps": np.float32(fps), "stride": np.int32(stride)}


def _ncc(a: np.ndarray, tmpl: np.ndarray) -> np.ndarray:
    """Normált keresztkorreláció minden kivágás és a minta között (N,)."""
    a = a.reshape(len(a), -1).astype(np.float32)
    a = a - a.mean(1, keepdims=True)
    t = tmpl.ravel().astype(np.float32)
    t = t - t.mean()
    return (a @ t) / (np.linalg.norm(a, axis=1) * np.linalg.norm(t) + 1e-6)


def scoreboard_presence(static: np.ndarray) -> np.ndarray:
    """Mennyire látszik az eredményjelző (NCC a mintához, -1..1).

    A minta a medián kivágás: a grafika a közvetítés nagy részében látszik,
    így a medián maga az eredményjelző. Egy finomító kör: a medián csak a
    biztosan látszó kockákból."""
    tmpl = np.median(static, 0)
    ncc = _ncc(static, tmpl)
    sure = ncc > SB_PRESENT_NCC
    if sure.sum() > 50:
        ncc = _ncc(static, np.median(static[sure], 0))
    return ncc


def score_change(score: np.ndarray, present: np.ndarray) -> np.ndarray:
    """Az állás-számjegyek változása az előző mintához képest (átlagos abszolút eltérés).

    Csak ott értelmes, ahol MINDKÉT mintán látszik a grafika; máshol 0
    (a grafika be- és kiúszása nem gól)."""
    s = score.reshape(len(score), -1).astype(np.float32)
    d = np.zeros(len(s), np.float32)
    d[1:] = np.abs(s[1:] - s[:-1]).mean(1)
    ok = np.zeros(len(s), bool)
    ok[1:] = present[1:] & present[:-1]
    return np.where(ok, d, 0.0)


# Gólszámlálás az eredményjelzőből: egy GÓL után az állás TARTÓSAN más; a grafika
# animációja (kiúszás, statisztika-sáv, ismétlés-kártya) után a kép visszaáll.
# Egy állapot legalább ennyi ideig látszó, változatlan grafika (s); két gól közt legalább ennyi idő.
GOAL_STABLE_S = 2.0
GOAL_MIN_GAP_S = 8.0
# A jelölt változás: a mintánkénti eltérés legalább ennyi (a zaj ~0,1–0,3).
GOAL_CAND_DIFF = 2.0


def goal_changes(score: np.ndarray, t: np.ndarray, present: np.ndarray, state_diff: float | None = None) -> dict:
    """Tartós állásváltozások (gólok) időpontjai az állás-kivágásokból.

    Az állás ÁLLAPOTOK sorozata: egy állapot a grafika olyan, legalább
    GOAL_STABLE_S hosszú szakasza, ahol látszik és nem változik (szomszédos
    minták eltérése < GOAL_CAND_DIFF). Gól ott van, ahol a következő állapot
    mediánképe eltér az utoljára elfogadott állapotétól — akkor is, ha közben
    a grafika ismétlés miatt percekig nem látszott. Az ennél rövidebb szakaszok
    (animáció, beúszás) nem állapotok, így az A→B→A villanás nem számít.
    A state_diff alapból az állapot-váltások eltéréseinek kétcsúcsú eloszlásából jön (Otsu)."""
    s = score.reshape(len(score), -1).astype(np.float32)
    step = np.full(len(s), np.inf, np.float32)
    step[1:] = np.abs(s[1:] - s[:-1]).mean(1)
    ok = present & (step < GOAL_CAND_DIFF)
    ok[0] = present[0]
    # stabil szakaszok
    segs, i, n = [], 0, len(s)
    while i < n:
        if not ok[i]:
            i += 1
            continue
        j = i
        while j + 1 < n and ok[j + 1] and present[j + 1]:
            j += 1
        if t[j] - t[i] >= GOAL_STABLE_S:
            segs.append((t[i], t[j], np.median(s[i:j + 1], 0)))
        i = j + 1
    if len(segs) < 2:
        return {"t": np.zeros(0), "kuszob": state_diff, "jeloltek": 0, "elteresek": np.zeros(0)}
    d_all = np.array([np.abs(segs[k][2] - segs[k - 1][2]).mean() for k in range(1, len(segs))])
    if state_diff is None:
        state_diff = _otsu(d_all)
    times, state = [], segs[0][2]
    for (t0, _t1, med) in segs[1:]:
        if np.abs(med - state).mean() > state_diff:
            if not times or t0 - times[-1] > GOAL_MIN_GAP_S:
                times.append(float(t0))
            state = med
    return {"t": np.array(times), "kuszob": float(state_diff), "jeloltek": len(segs), "elteresek": d_all}


def _otsu(x: np.ndarray, bins: int = 64) -> float:
    """Két csoportra bontó küszöb (Otsu) — a "nem változott" és a "változott" állás közt."""
    lx = np.log1p(x)
    hist, edges = np.histogram(lx, bins)
    centers = (edges[:-1] + edges[1:]) / 2
    best, thr = -1.0, centers[0]
    for k in range(1, bins):
        w0, w1 = hist[:k].sum(), hist[k:].sum()
        if w0 == 0 or w1 == 0:
            continue
        m0 = (hist[:k] * centers[:k]).sum() / w0
        m1 = (hist[k:] * centers[k:]).sum() / w1
        v = w0 * w1 * (m0 - m1) ** 2
        if v > best:
            best, thr = v, edges[k]
    return float(np.expm1(thr))


def features(raw: dict) -> dict:
    """A nyers jelekből a lövésfelismerő mintakockánkénti jelei."""
    present_ncc = scoreboard_presence(raw["sb_static"])
    present = present_ncc > SB_PRESENT_NCC
    dx, dy, resp = raw["cam"].T
    reliable = resp >= CAM_MIN_RESPONSE
    return {"frame": raw["frame"], "fps": raw["fps"], "stride": raw["stride"],
            "sb_ncc": present_ncc.astype(np.float32),
            "sb_change": score_change(raw["sb_score"], present),
            "cam_dx": np.where(reliable, dx, 0.0).astype(np.float32),
            "cam_dy": np.where(reliable, dy, 0.0).astype(np.float32),
            "cam_resp": resp.astype(np.float32)}


def load(path: str | Path) -> dict:
    z = np.load(path)
    d = {k: z[k] for k in z.files}
    d["t"] = d["frame"] / float(d["fps"])
    return d


def align(pose_frames: np.ndarray, bc: dict) -> dict:
    """A közvetítési jelek a póz-mintakockákra igazítva (azonos kocka-indexek)."""
    idx = np.searchsorted(bc["frame"], pose_frames)
    idx = np.clip(idx, 0, len(bc["frame"]) - 1)
    exact = bc["frame"][idx] == pose_frames
    out = {}
    for k in ("sb_ncc", "sb_change", "cam_dx", "cam_dy", "cam_resp"):
        out[k] = np.where(exact, bc[k][idx], 0.0).astype(np.float32)
    return out


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description="Közvetítési jelek: eredményjelző és kamera-mozgás")
    ap.add_argument("--video", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--stride", type=int, default=5)
    ap.add_argument("--sb-static", type=float, nargs=4, default=SB_STATIC)
    ap.add_argument("--sb-score", type=float, nargs=4, default=SB_SCORE)
    ap.add_argument("--keep-crops", action="store_true", help="a nyers eredményjelző-kivágásokat is menti")
    args = ap.parse_args(argv)
    out = Path(args.out)
    out.parent.mkdir(parents=True, exist_ok=True)
    raw = extract(args.video, args.stride, tuple(args.sb_static), tuple(args.sb_score))
    feat = features(raw)
    if args.keep_crops:
        feat.update(sb_static_crops=raw["sb_static"], sb_score_crops=raw["sb_score"])
    np.savez_compressed(out, **feat)
    pres = (feat["sb_ncc"] > SB_PRESENT_NCC).mean()
    print(f"Kész: {len(feat['frame'])} minta; eredményjelző látszik: {100 * pres:.0f}% → {out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile hbd/court.py
"""Ki áll a pályán? — a nézők és a lelátó kiszűrése a láb alatti színből.

A pózbecslő a lelátón ülő nézőket is embernek látja, és egy gólnál felemelt
szurkolói kar a lövésfelismerőnek "dobókar". Egyszerű, olcsó szűrő: a
játékos lába alatt a PÁLYA PADLÓJA van, a nézőé alatt szék, korlát, másik
néző. A padló színét a meccsből tanuljuk: a láb alatti foltok
leggyakoribb színe (a legtöbb észlelt ember a pályán áll).

Az eredmény emberenkénti jelző (`on_court`), a `match_pose` sorrendjében;
a `filter_pose` ennek alapján szűri a póz-adatot.

    python -m hbd.court --video meccs.mp4 --pose m1_pose.npz --out m1_court.npz
"""

from __future__ import annotations

import argparse
import subprocess
from pathlib import Path

import numpy as np

from .video import ffmpeg_exe

WORK = (640, 360)
# A láb alatti folt: a doboz alsó szélének közepén, a doboz-magasság ennyi részéig lefelé.
FOOT_PATCH = 0.06
# Lab-színtávolság a padló-színtől, ami alatt "pályán áll". Az 1. meccsen a pályán álló
# játékosok 20–36 (fény, vonal, reklám a padlón), a nézők 55–140 között voltak.
FLOOR_DIST = 45.0
# A padló-színt ennyi leggyakoribb színcsoport adja (a pálya több színű lehet: alap + 9-es terület).
FLOOR_BINS = 2
BIN = 16
# A padló világos (a csarnok megvilágítja); ennél sötétebb színcsoport nem padló (OpenCV Lab: L 0..255).
# A 2. meccsen a 2. leggyakoribb "láb alatti" szín L≈7 volt: a sötét lelátón ülő nézők.
FLOOR_MIN_L = 60.0
# Ennyi leggyakoribb csoportból választunk FLOOR_BINS világosat.
FLOOR_CANDIDATES = 6


def foot_colors(video: str, pose: dict) -> np.ndarray:
    """Emberenként a láb alatti folt átlagos Lab-színe (össz_emberek, 3)."""
    import cv2

    w, h = WORK
    stride = int(pose["stride"])
    frames = pose["frame"]
    off, boxes = pose["offsets"], pose["boxes"].astype(np.float32)
    out = np.zeros((len(boxes), 3), np.float32)
    cmd = [ffmpeg_exe(), "-v", "error", "-i", video, "-vf", f"select='not(mod(n\\,{stride}))',scale={w}:{h}",
           "-vsync", "0", "-f", "rawvideo", "-pix_fmt", "bgr24", "-"]
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL)
    i, k = 0, 0
    try:
        while i < len(frames):
            buf = proc.stdout.read(w * h * 3)
            if len(buf) < w * h * 3:
                break
            if k * stride != frames[i]:
                k += 1
                continue
            lab = cv2.cvtColor(np.frombuffer(buf, np.uint8).reshape(h, w, 3), cv2.COLOR_BGR2LAB)
            for j in range(off[i], off[i + 1]):
                x0, y0, x1, y1 = boxes[j] * [w, h, w, h]
                cx, bh = (x0 + x1) / 2, max(y1 - y0, 1.0)
                ya, yb = int(min(h - 1, y1)), int(min(h, y1 + max(2, FOOT_PATCH * bh)))
                xa, xb = int(max(0, cx - 3)), int(min(w, cx + 4))
                patch = lab[ya:max(yb, ya + 1), xa:max(xb, xa + 1)]
                out[j] = patch.reshape(-1, 3).mean(0)
            i += 1
            k += 1
    finally:
        proc.stdout.close()
        proc.kill()
        proc.wait()
    return out


def floor_colors(colors: np.ndarray) -> np.ndarray:
    """A padló színe(i): a láb alatti színek leggyakoribb, VILÁGOS (L ≥ FLOOR_MIN_L) csoportjainak
    mediánja, legfeljebb FLOOR_BINS darab."""
    q = (colors // BIN).astype(np.int32)
    keys = q[:, 0] * 10000 + q[:, 1] * 100 + q[:, 2]
    vals, counts = np.unique(keys, return_counts=True)
    out = []
    for key in vals[np.argsort(-counts)][:FLOOR_CANDIDATES]:
        med = np.median(colors[keys == key], 0)
        if med[0] >= FLOOR_MIN_L:
            out.append(med)
        if len(out) == FLOOR_BINS:
            break
    return np.array(out, np.float32)


def on_court(colors: np.ndarray, floor: np.ndarray) -> np.ndarray:
    d = np.linalg.norm(colors[:, None, :] - floor[None, :, :], axis=2).min(1)
    return d <= FLOOR_DIST


def filter_pose(pose: dict, keep: np.ndarray) -> dict:
    """A póz-adat csak a megtartott emberekkel (azonos mintakockák)."""
    off = pose["offsets"]
    n = np.array([int(keep[off[i]:off[i + 1]].sum()) for i in range(len(pose["n"]))], np.int32)
    out = dict(pose)
    out.update(n=n, offsets=np.concatenate([[0], np.cumsum(n)]).astype(np.int64),
               boxes=pose["boxes"][keep], kps=pose["kps"][keep])
    return out


def main(argv=None) -> int:
    from .match_pose import load

    ap = argparse.ArgumentParser(description="Pályán álló emberek (a nézők kiszűrése)")
    ap.add_argument("--video", required=True)
    ap.add_argument("--pose", required=True)
    ap.add_argument("--out", required=True)
    args = ap.parse_args(argv)
    pose = load(args.pose)
    col = foot_colors(args.video, pose)
    floor = floor_colors(col)
    keep = on_court(col, floor)
    np.savez_compressed(args.out, colors=col.astype(np.float16), floor=floor, on_court=keep)
    print(f"Padló-szín (Lab): {np.round(floor, 1).tolist()}; pályán: {keep.mean():.0%} "
          f"({keep.sum()} / {len(keep)} észlelés) → {args.out}")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile hbd/shot_detect.py
"""Lövésfelismerés a teljes meccsen (időbeli keresés).

Kérdés: a meccs MELYIK pillanataiban van lövés? Bemenet a `match_pose`
kimenete (minden ritkított kockán az összes ember testpontja) és a
meccsjelölő időpontjai (a lövés pillanatában nyomott L).

Módszer (első változat, követés nélkül):
  1. Kockánként összesített jellemzők: hány ember látszik, mekkorák (széles
     kép vagy közeli), van-e magasra emelt dobókar (a csukló a fej fölött),
     milyen gyorsan változik a kép (kamera-mozgás, vágás).
  2. Időablak: minden mintakocka körül a jellemzők alakulása az előtte és
     utána lévő időszakaszokban (a lövés egy mozdulat-SOROZAT: felemelés,
     lendítés, utána a kamera a kapu felé fordul).
  3. Gradiens-erősítéses döntési fák kockánként "lövés-valószínűséget" adnak;
     a csúcsokból lesznek a jelölt lövések.

Kiértékelés: a meccs egymást követő idő-blokkjaira bontva (a szomszédos
kockák ne kerülhessenek a tanító és a teszt részbe is), ESEMÉNY-szinten:
egy jelölt akkor talált, ha MATCH_S-en belül van egy valódi lövés. Mérjük:
hány lövést talál meg (felidézés), és percenként hány téves riasztást ad.

    python -m hbd.shot_detect --pose m1_pose.npz --events meccs1_esemenyek.json --out res/
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np

from .match_pose import load as load_pose

# A gombnyomás a reakcióidő miatt a lövés UTÁN jön; a jelölt lövés-idő ennyivel korábban van.
REACTION_S = 0.4
# Pozitív minta: a becsült lövés-időhöz ennyire közeli mintakocka.
POS_S = 0.6
# A pozitív körüli "bizonytalan" sáv (se nem pozitív, se nem negatív a tanításban).
IGNORE_S = 2.0
# Egy jelölt akkor talált, ha ennyi másodpercen belül van valódi lövés.
MATCH_S = 1.5
# Két jelölt lövés között legalább ennyi idő (csúcskeresés).
MIN_GAP_S = 2.0
# Az időablak szakaszai (másodperc, a mintakockához képest).
WINDOWS = ((-2.0, -1.0), (-1.0, -0.2), (-0.2, 0.2), (0.2, 1.0), (1.0, 2.0))
BLOCKS = 5
PAIR_S = 4.0
# Hosszabb kitekintés a lassú közvetítési jelekre: az állás a gól után
# másodpercekkel frissül, a kamera a lövés után fordul.
LONG_WINDOWS = ((-8.0, -2.0), (2.0, 5.0), (5.0, 10.0))
LONG_FEATURES = ("allas_valtozas", "eredmenyjelzo_latszik", "kamera_dx", "kamera_seb")
# Követés: két mintakocka közti legnagyobb ugrás (doboz-magasságban) és időköz.
TRACK_MAX_JUMP = 0.6
TRACK_MAX_DT_S = 0.5

# COCO testpont-indexek
NOSE, L_SH, R_SH, L_WR, R_WR, L_HIP, R_HIP = 0, 5, 6, 9, 10, 11, 12


# ------------------------------------------------------------ események
def shot_times(doc: dict, include_replays: bool = True) -> list[dict]:
    """A meccsjelölő nyomásaiból a lövések (a lövés pillanata = első nyomás − REACTION_S)."""
    from collections import deque

    ev = sorted(doc["esemenyek"], key=lambda e: e["t"])
    out, i = [], 0
    while i < len(ev):
        e = ev[i]
        n = ev[i + 1] if i + 1 < len(ev) else None
        if e.get("tipus") != "loves":
            i += 1
            continue
        paired = (not e.get("eredmeny") and n and n.get("tipus") == "loves" and n.get("eredmeny")
                  and not n.get("kipattano") and n["t"] - e["t"] <= PAIR_S
                  and bool(n.get("ismetles")) == bool(e.get("ismetles")))
        res = n["eredmeny"] if paired else e.get("eredmeny")
        out.append({"t": e["t"] - REACTION_S, "eredmeny": res, "ismetles": bool(e.get("ismetles"))})
        i += 2 if paired else 1
    del deque
    return [s for s in out if include_replays or not s["ismetles"]]


# ------------------------------------------------------------ jellemzők
def frame_features(pose: dict) -> tuple[np.ndarray, list[str]]:
    """Mintakockánkénti összesített jellemzők (N, F)."""
    off, kps, boxes = pose["offsets"], pose["kps"].astype(np.float32), pose["boxes"].astype(np.float32)
    N = len(pose["n"])
    names = ["n_ember", "doboz_mag_median", "doboz_mag_max", "kep_valtozas",
             "kar_fent_max", "kar_fent_db", "csuklo_fej_felett_db", "kar_nyujtas_max",
             "csuklo_magassag_p90", "kozep_x_szoras"]
    F = np.zeros((N, len(names)), np.float32)
    for i in range(N):
        a, b = off[i], off[i + 1]
        F[i, 0] = b - a
        F[i, 3] = pose["motion"][i]
        if b == a:
            continue
        bx = boxes[a:b]
        hgt = bx[:, 3] - bx[:, 1]
        F[i, 1], F[i, 2] = np.median(hgt), hgt.max()
        k = kps[a:b]
        ok = k[..., 2] > 0.3
        sh = (k[:, L_SH, :2] + k[:, R_SH, :2]) / 2
        hip = (k[:, L_HIP, :2] + k[:, R_HIP, :2]) / 2
        torso = np.linalg.norm(sh - hip, axis=1)
        torso = np.where((ok[:, [L_SH, R_SH, L_HIP, R_HIP]].all(1)) & (torso > 1e-3), torso, np.nan)
        # csukló a váll fölött, törzshosszban (kép-y lefelé nő → előjelváltás)
        up_l = np.where(ok[:, L_WR] & ok[:, L_SH], (k[:, L_SH, 1] - k[:, L_WR, 1]) / torso, np.nan)
        up_r = np.where(ok[:, R_WR] & ok[:, R_SH], (k[:, R_SH, 1] - k[:, R_WR, 1]) / torso, np.nan)
        up = np.fmax(up_l, up_r)
        ext_l = np.linalg.norm(k[:, L_WR, :2] - k[:, L_SH, :2], axis=1) / torso
        ext_r = np.linalg.norm(k[:, R_WR, :2] - k[:, R_SH, :2], axis=1) / torso
        ext = np.fmax(np.where(ok[:, L_WR], ext_l, np.nan), np.where(ok[:, R_WR], ext_r, np.nan))
        head = np.where(ok[:, NOSE], k[:, NOSE, 1], np.nan)
        above = (np.where(ok[:, L_WR], k[:, L_WR, 1], np.inf) < head) | (np.where(ok[:, R_WR], k[:, R_WR, 1], np.inf) < head)
        if np.isfinite(up).any():
            F[i, 4] = np.nanmax(up)
            F[i, 5] = np.nansum(up > 0.5)
            F[i, 8] = np.nanpercentile(up, 90)
        F[i, 6] = above.sum()
        if np.isfinite(ext).any():
            F[i, 7] = np.nanmax(ext)
        F[i, 9] = np.std((bx[:, 0] + bx[:, 2]) / 2)
    return np.nan_to_num(F), names


def _match_people(c_prev: np.ndarray, c_cur: np.ndarray, scale: np.ndarray) -> list[tuple[int, int]]:
    """Mohó párosítás két mintakocka emberei között (középpont-távolság / doboz-magasság)."""
    if len(c_prev) == 0 or len(c_cur) == 0:
        return []
    d = np.linalg.norm(c_cur[:, None, :] - c_prev[None, :, :], axis=2) / scale[:, None]
    pairs, used_p, used_c = [], set(), set()
    for flat in np.argsort(d, axis=None):
        ci, pi = divmod(int(flat), d.shape[1])
        if d[ci, pi] > TRACK_MAX_JUMP:
            break
        if ci in used_c or pi in used_p:
            continue
        used_c.add(ci)
        used_p.add(pi)
        pairs.append((pi, ci))
    return pairs


def swing_features(pose: dict) -> tuple[np.ndarray, list[str]]:
    """Mozgás-jellemzők rövid követéssel: ki kicsoda az előző mintakockán (N, F).

    A dobás a kar GYORS lendítése: a csukló a vállhoz képest egy-két tized
    másodperc alatt fej mögül előre-le csap. Egy kockából ez nem látszik,
    két egymást követőből igen. A csukló-elmozdulást a vállak közepéhez
    mérjük (így a futás és a kamera-mozgás kiesik), a doboz magasságával
    osztva (így a közeli és távoli játékos összemérhető), másodpercre."""
    off, kps, boxes = pose["offsets"], pose["kps"].astype(np.float32), pose["boxes"].astype(np.float32)
    w, h = (pose["size"] if "size" in pose else (1920, 1080))
    t = pose["t"]
    N = len(pose["n"])
    names = ["csuklo_seb_max", "csuklo_seb_fentrol_max", "kar_lecsapas_max", "ember_seb_max", "kovetett_db"]
    F = np.zeros((N, len(names)), np.float32)
    for i in range(1, N):
        dt = t[i] - t[i - 1]
        a0, b0, a, b = off[i - 1], off[i], off[i], off[i + 1]
        if not (0 < dt <= TRACK_MAX_DT_S) or b0 == a0 or b == a:
            continue
        bp, bc = boxes[a0:b0] * [w, h, w, h], boxes[a:b] * [w, h, w, h]
        cp = np.stack([(bp[:, 0] + bp[:, 2]) / 2, (bp[:, 1] + bp[:, 3]) / 2], 1)
        cc = np.stack([(bc[:, 0] + bc[:, 2]) / 2, (bc[:, 1] + bc[:, 3]) / 2], 1)
        hc = np.maximum(bc[:, 3] - bc[:, 1], 1.0)
        pairs = _match_people(cp, cc, hc)
        if not pairs:
            continue
        kp, kc = kps[a0:b0] * [w, h, 1], kps[a:b] * [w, h, 1]
        best = np.zeros(4, np.float32)
        for pi, ci in pairs:
            p, c, hh = kp[pi], kc[ci], hc[ci]
            best[3] = max(best[3], np.linalg.norm(cc[ci] - cp[pi]) / hh / dt)
            if min(p[L_SH, 2], p[R_SH, 2], c[L_SH, 2], c[R_SH, 2]) < 0.3:
                continue
            shp, shc = (p[L_SH, :2] + p[R_SH, :2]) / 2, (c[L_SH, :2] + c[R_SH, :2]) / 2
            for wr in (L_WR, R_WR):
                if p[wr, 2] < 0.3 or c[wr, 2] < 0.3:
                    continue
                rp, rc = (p[wr, :2] - shp) / hh, (c[wr, :2] - shc) / hh
                v = np.linalg.norm(rc - rp) / dt
                best[0] = max(best[0], v)
                if rp[1] < 0:            # az előző kockán a csukló a váll fölött volt
                    best[1] = max(best[1], v)
                    best[2] = max(best[2], (rc[1] - rp[1]) / dt)  # lefelé csap (kép-y lefelé nő)
        F[i, :4] = best
        F[i, 4] = len(pairs)
    return F, names


def broadcast_features(bc: dict) -> tuple[np.ndarray, list[str]]:
    """A közvetítés jelei (a póz-mintakockákra igazítva, lásd `broadcast.align`)."""
    from .broadcast import SB_PRESENT_NCC

    names = ["eredmenyjelzo_latszik", "allas_valtozas", "kamera_dx", "kamera_dy", "kamera_seb", "kamera_biztos"]
    F = np.stack([(bc["sb_ncc"] > SB_PRESENT_NCC).astype(np.float32), bc["sb_change"], bc["cam_dx"], bc["cam_dy"],
                  np.hypot(bc["cam_dx"], bc["cam_dy"]), bc["cam_resp"]], 1).astype(np.float32)
    return F, names


def window_features(F: np.ndarray, t: np.ndarray, names: list[str],
                    windows=None) -> tuple[np.ndarray, list[str]]:
    """Minden mintakocka körül a WINDOWS szakaszok átlaga és maximuma (N, F*len(WINDOWS)*2)."""
    N = len(t)
    out, onames = [], []
    for lo, hi in (windows or WINDOWS):
        a = np.searchsorted(t, t + lo, side="left")
        b = np.searchsorted(t, t + hi, side="right")
        mean = np.zeros_like(F)
        mx = np.zeros_like(F)
        cs = np.concatenate([np.zeros((1, F.shape[1])), np.cumsum(F, 0)])
        for i in range(N):
            if b[i] > a[i]:
                mean[i] = (cs[b[i]] - cs[a[i]]) / (b[i] - a[i])
                mx[i] = F[a[i]:b[i]].max(0)
        out += [mean, mx]
        onames += [f"{n}@{lo:+.1f}..{hi:+.1f}:atlag" for n in names] + [f"{n}@{lo:+.1f}..{hi:+.1f}:max" for n in names]
    return np.concatenate(out, 1), onames


def labels(t: np.ndarray, shots: list[dict]) -> np.ndarray:
    """1 = lövés-pillanat, 0 = nem lövés, -1 = bizonytalan sáv (tanításból kihagyva)."""
    y = np.zeros(len(t), np.int8)
    st = np.array([s["t"] for s in shots])
    if len(st):
        d = np.abs(t[:, None] - st[None, :]).min(1)
        y[d <= IGNORE_S] = -1
        y[d <= POS_S] = 1
    return y


# ------------------------------------------------------------ események kinyerése
def pick_peaks(t: np.ndarray, score: np.ndarray, thr: float) -> list[tuple[float, float]]:
    """Csúcsok (idő, pontszám) a küszöb fölött, legalább MIN_GAP_S távolságra egymástól."""
    order = np.argsort(-score)
    chosen: list[tuple[float, float]] = []
    for i in order:
        if score[i] < thr:
            break
        if all(abs(t[i] - c) >= MIN_GAP_S for c, _ in chosen):
            chosen.append((float(t[i]), float(score[i])))
    return sorted(chosen)


def match_events(pred: list[tuple[float, float]], truth: np.ndarray) -> tuple[int, int, int]:
    """(találat, téves, kihagyott) — mohó párosítás MATCH_S-en belül."""
    used = np.zeros(len(truth), bool)
    tp = 0
    for tm, _ in sorted(pred, key=lambda p: -p[1]):
        if len(truth) == 0:
            break
        d = np.abs(truth - tm)
        d[used] = np.inf
        j = int(np.argmin(d))
        if d[j] <= MATCH_S:
            used[j] = True
            tp += 1
    return tp, len(pred) - tp, int((~used).sum())


def smooth(score: np.ndarray, k: int = 3) -> np.ndarray:
    if k <= 1:
        return score
    return np.convolve(np.pad(score, k // 2, mode="edge"), np.ones(k) / k, mode="valid")


def build_features(pose: dict, bc: dict | None = None, swing: bool = True):
    """A teljes jellemző-mátrix: póz (+ követéses lendítés) (+ közvetítés), időablakokkal."""
    t = pose["t"]
    F, names = frame_features(pose)
    if swing:
        S, sn = swing_features(pose)
        F, names = np.concatenate([F, S], 1), names + sn
    if bc is not None:
        from .broadcast import align

        B, bn = broadcast_features(align(pose["frame"], bc))
        F, names = np.concatenate([F, B], 1), names + bn
    X, xnames = window_features(F, t, names)
    long_idx = [names.index(n) for n in LONG_FEATURES if n in names]
    if long_idx:
        XL, xl = window_features(F[:, long_idx], t, [names[i] for i in long_idx], LONG_WINDOWS)
        X, xnames = np.concatenate([X, XL], 1), xnames + xl
    return F, names, X, xnames


def evaluate(pose: dict, shots: list[dict], seed: int = 0, bc: dict | None = None, swing: bool = True,
             drop: tuple[str, ...] = ()) -> dict:
    """drop: ezekkel a nevekkel kezdődő jellemzők kimaradnak (pl. az eredményjelző-jelek, hogy a
    modell ne a lövés UTÁNI állásváltozásból tudja meg, hogy lövés volt)."""
    from sklearn.ensemble import HistGradientBoostingClassifier

    t = pose["t"]
    F, names, X, xnames = build_features(pose, bc, swing)
    if drop:
        keep = [i for i, n in enumerate(xnames) if not n.startswith(tuple(drop))]
        X, xnames = X[:, keep], [xnames[i] for i in keep]
    y = labels(t, shots)
    st = np.array([s["t"] for s in shots])
    edges = np.linspace(t.min(), t.max() + 1e-6, BLOCKS + 1)
    score = np.zeros(len(t))
    for b in range(BLOCKS):
        te = (t >= edges[b]) & (t < edges[b + 1])
        # a blokkhatár körüli 5 s-ot kihagyjuk a tanításból (ne szivárogjon át az ablakon)
        tr = ~te & (np.abs(t - edges[b]) > 5) & (np.abs(t - edges[b + 1]) > 5) & (y >= 0)
        clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                             l2_regularization=1.0, class_weight="balanced", random_state=seed)
        clf.fit(X[tr], y[tr])
        score[te] = clf.predict_proba(X[te])[:, 1]
    score = smooth(score, 3)

    minutes = (t.max() - t.min()) / 60
    curve = []
    for thr in np.round(np.arange(0.05, 0.96, 0.05), 2):
        pred = pick_peaks(t, score, thr)
        tp, fp, fn = match_events(pred, st)
        prec = tp / max(1, tp + fp)
        rec = tp / max(1, len(st))
        curve.append({"kuszob": float(thr), "talalt": tp, "teves": fp, "kihagyott": fn,
                      "felidezes": rec, "pontossag": prec, "teves_per_perc": fp / minutes,
                      "f1": 2 * prec * rec / max(1e-9, prec + rec)})
    best = max(curve, key=lambda c: c["f1"])
    # viszonyítás: a legegyszerűbb szabály — "valakinek magasan van a karja"
    rule = F[:, names.index("kar_fent_max")]
    rule = (rule - rule.min()) / (rule.max() - rule.min() + 1e-9)
    rule_curve = []
    for thr in np.round(np.arange(0.05, 0.96, 0.05), 2):
        tp, fp, fn = match_events(pick_peaks(t, smooth(rule, 3), thr), st)
        prec, rec = tp / max(1, tp + fp), tp / max(1, len(st))
        rule_curve.append({"kuszob": float(thr), "felidezes": rec, "pontossag": prec,
                           "teves_per_perc": fp / minutes, "f1": 2 * prec * rec / max(1e-9, prec + rec)})
    clf = HistGradientBoostingClassifier(max_iter=300, learning_rate=0.05, max_leaf_nodes=31,
                                         l2_regularization=1.0, class_weight="balanced", random_state=seed)
    clf.fit(X[y >= 0], y[y >= 0])
    return {"percek": minutes, "mintakockak": int(len(t)), "lovesek": int(len(st)),
            "pozitiv_minta": int((y == 1).sum()), "gorbe": curve, "legjobb": best,
            "szabaly_legjobb": max(rule_curve, key=lambda c: c["f1"]),
            "score": score, "t": t, "model": clf, "features": xnames}


# ------------------------------------------------------------ új meccs: jóslás egy kész modellel
# A jelölt-küszöb: az első meccs keresztvalidációjában ennyi felidézést elérő legkisebb
# téves-riasztású küszöb (a jelölteket ember nézi át → inkább több, mint kevesebb).
PREDICT_RECALL = 0.8


def make_bundle(res: dict, swing: bool, broadcast: bool, court: bool) -> dict:
    """A teljes meccsen tanított modell + a beállítás, amivel egy ÚJ meccsen ugyanúgy kell futtatni."""
    c = at_recall(res["gorbe"], PREDICT_RECALL) or res["legjobb"]
    return {"model": res["model"], "features": res["features"], "swing": swing, "broadcast": broadcast,
            "court": court, "kuszob": c["kuszob"], "kuszob_cv": {k: c[k] for k in ("felidezes", "teves_per_perc")}}


def predict(pose: dict, bundle: dict, bc: dict | None = None) -> tuple[np.ndarray, np.ndarray]:
    """Lövés-pontszám mintakockánként egy kész modellel (ugyanazokkal a jellemzőkkel, mint tanításkor)."""
    if bundle.get("broadcast") and bc is None:
        raise ValueError("a modell közvetítés-jellemzőkkel tanult: --broadcast kell")
    _, _, X, xnames = build_features(pose, bc if bundle.get("broadcast") else None, bundle.get("swing", True))
    idx = {n: i for i, n in enumerate(xnames)}
    missing = [n for n in bundle["features"] if n not in idx]
    if missing:
        raise ValueError(f"hiányzó jellemzők: {missing[:5]}")
    X = X[:, [idx[n] for n in bundle["features"]]]
    return pose["t"], smooth(bundle["model"].predict_proba(X)[:, 1], 3)


def candidates_doc(t: np.ndarray, score: np.ndarray, thr: float, name: str) -> dict:
    """A jelölt lövések a meccsjelölő formátumában (a nyomás = lövés + REACTION_S, mint az embernél),
    így a cut_match_clips változtatás nélkül kivágja őket."""
    peaks = pick_peaks(t, score, thr)
    ev = [{"tipus": "loves", "t": round(tm + REACTION_S, 2), "gepi": True, "pontszam": round(p, 3)}
          for tm, p in peaks]
    return {"nev": name, "hossz_s": float(t.max()), "kuszob": thr, "esemenyek": ev}


def score_against(t: np.ndarray, score: np.ndarray, shots: list[dict]) -> dict:
    """Egy rögzített modell pontszámai az emberi jelöléssel összevetve (nincs újratanítás)."""
    st = np.array([s["t"] for s in shots])
    minutes = (t.max() - t.min()) / 60
    curve = []
    for thr in np.round(np.arange(0.05, 0.96, 0.05), 2):
        tp, fp, fn = match_events(pick_peaks(t, score, thr), st)
        prec, rec = tp / max(1, tp + fp), tp / max(1, len(st))
        curve.append({"kuszob": float(thr), "talalt": tp, "teves": fp, "kihagyott": fn, "felidezes": rec,
                      "pontossag": prec, "teves_per_perc": fp / minutes, "f1": 2 * prec * rec / max(1e-9, prec + rec)})
    return {"percek": minutes, "lovesek": int(len(st)), "gorbe": curve, "legjobb": max(curve, key=lambda c: c["f1"])}


def at_recall(curve: list[dict], target: float) -> dict | None:
    ok = [c for c in curve if c["felidezes"] >= target]
    return min(ok, key=lambda c: c["teves_per_perc"]) if ok else None


def write_report(res: dict, out: Path, title: str, live_only: bool = False) -> Path:
    b, r = res["legjobb"], res["szabaly_legjobb"]
    pct = lambda v: f"{100 * v:.0f}%"  # noqa: E731
    md = [f"# Lövésfelismerés a teljes meccsen — {title}", "",
          f"{res['percek']:.0f} perc, {res['mintakockak']} mintakocka, {res['lovesek']} jelölt lövés "
          f"({'csak élő' if live_only else 'élő + ismétlés'}). Kiértékelés: {BLOCKS} egymást követő idő-blokk, mindig a többi blokkon tanítva. "
          f"Egy jelölt akkor talált, ha {MATCH_S} s-on belül van valódi lövés.", "",
          "| | felidézés (megtalált lövések) | pontosság (a jelöltekből valódi) | téves riasztás / perc |",
          "|---|---|---|---|",
          f"| tanított modell (legjobb F1) | {pct(b['felidezes'])} | {pct(b['pontossag'])} | {b['teves_per_perc']:.2f} |",
          f"| szabály: „valakinek magasan a karja” | {pct(r['felidezes'])} | {pct(r['pontossag'])} | {r['teves_per_perc']:.2f} |",
          "", "Felidézés–téves riasztás egyensúly (tanított modell):", "",
          "| cél-felidézés | elért | téves riasztás / perc | pontosság |", "|---|---|---|---|"]
    for target in (0.5, 0.7, 0.8, 0.9):
        c = at_recall(res["gorbe"], target)
        if c:
            md.append(f"| {pct(target)} | {pct(c['felidezes'])} | {c['teves_per_perc']:.2f} | {pct(c['pontossag'])} |")
        else:
            md.append(f"| {pct(target)} | nem érhető el | – | – |")
    p = out / "shot_detect.md"
    p.write_text("\n".join(md) + "\n")
    return p


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description="Lövésfelismerés a teljes meccsen")
    ap.add_argument("--pose", required=True, help="a hbd.match_pose kimenete")
    ap.add_argument("--events", help="a meccsjelölő dokumentuma (JSON); --model nélkül kötelező")
    ap.add_argument("--model", help="kész modell (shot_detect.joblib) → jóslás egy ÚJ meccsen, újratanítás nélkül")
    ap.add_argument("--threshold", type=float, help="jelölt-küszöb (alapból a modellé)")
    ap.add_argument("--out", required=True)
    ap.add_argument("--live-only", action="store_true", help="csak az élő lövések (ismétlések nélkül)")
    ap.add_argument("--title", default="")
    ap.add_argument("--broadcast", help="a hbd.broadcast kimenete (eredményjelző + kamera)")
    ap.add_argument("--no-swing", action="store_true", help="követéses lendítés-jellemzők nélkül")
    ap.add_argument("--court", help="a hbd.court kimenete: csak a pályán álló emberek (nézők nélkül)")
    ap.add_argument("--drop", nargs="*", default=[], help="ezekkel kezdődő jellemzők nélkül (pl. allas_valtozas)")
    args = ap.parse_args(argv)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    if args.model:
        return _predict_main(args, out)
    if not args.events:
        ap.error("--events vagy --model kell")
    doc = json.loads(Path(args.events).read_text())
    doc = doc.get("data", doc)
    shots = shot_times(doc, include_replays=not args.live_only)
    pose = load_pose(args.pose)
    if args.court:
        from .court import filter_pose

        pose = filter_pose(pose, np.load(args.court)["on_court"])
    bc = None
    if args.broadcast:
        from .broadcast import load as load_bc

        bc = load_bc(args.broadcast)
    res = evaluate(pose, shots, bc=bc, swing=not args.no_swing, drop=tuple(args.drop))
    import joblib

    joblib.dump(make_bundle(res, swing=not args.no_swing, broadcast=bc is not None, court=bool(args.court)),
                out / "shot_detect.joblib")
    res.pop("model")
    np.savez_compressed(out / "shot_scores.npz", t=res.pop("t"), score=res.pop("score"))
    (out / "shot_detect.json").write_text(json.dumps({k: v for k, v in res.items() if k != "features"},
                                                      ensure_ascii=False, indent=1))
    print(write_report(res, out, args.title or doc.get("nev", ""), args.live_only).read_text())
    return 0


def _predict_main(args, out: Path) -> int:
    """Új meccs: jelölt lövések egy kész modellel; ha van emberi jelölés is, összevetés vele."""
    import joblib

    bundle = joblib.load(args.model)
    pose = load_pose(args.pose)
    if bundle.get("court"):
        from .court import filter_pose

        if not args.court:
            raise SystemExit("a modell pálya-szűrővel tanult: --court kell (a hbd.court kimenete)")
        pose = filter_pose(pose, np.load(args.court)["on_court"])
    bc = None
    if bundle.get("broadcast"):
        from .broadcast import load as load_bc

        if not args.broadcast:
            raise SystemExit("a modell közvetítés-jellemzőkkel tanult: --broadcast kell")
        bc = load_bc(args.broadcast)
    t, score = predict(pose, bundle, bc)
    thr = args.threshold if args.threshold is not None else bundle["kuszob"]
    cand = candidates_doc(t, score, thr, args.title or Path(args.pose).stem)
    (out / "jelolt_lovesek.json").write_text(json.dumps(cand, ensure_ascii=False, indent=1))
    np.savez_compressed(out / "shot_scores.npz", t=t, score=score)
    print(f"{len(cand['esemenyek'])} jelölt lövés (küszöb {thr}) → {out / 'jelolt_lovesek.json'}")
    if args.events:
        doc = json.loads(Path(args.events).read_text())
        doc = doc.get("data", doc)
        res = score_against(t, score, shot_times(doc, include_replays=not args.live_only))
        at = next(c for c in res["gorbe"] if abs(c["kuszob"] - thr) < 1e-9) if any(
            abs(c["kuszob"] - thr) < 1e-9 for c in res["gorbe"]) else None
        (out / "shot_detect_uj_meccs.json").write_text(json.dumps(res, ensure_ascii=False, indent=1))
        pct = lambda v: f"{100 * v:.0f}%"  # noqa: E731
        md = [f"# Lövésfelismerés egy ÚJ meccsen (újratanítás nélkül) — {args.title}", "",
              f"{res['percek']:.0f} perc, {res['lovesek']} jelölt lövés (ember). A modell ezt a meccset nem látta.", "",
              "| | felidézés | pontosság | téves riasztás / perc |", "|---|---|---|---|"]
        if at:
            md.append(f"| a rögzített küszöbön ({thr}) | {pct(at['felidezes'])} | {pct(at['pontossag'])} | "
                      f"{at['teves_per_perc']:.2f} |")
        b = res["legjobb"]
        md.append(f"| legjobb F1 (utólag választott küszöb) | {pct(b['felidezes'])} | {pct(b['pontossag'])} | "
                  f"{b['teves_per_perc']:.2f} |")
        for target in (0.7, 0.8, 0.9):
            c = at_recall(res["gorbe"], target)
            md.append(f"| {pct(target)} felidézéshez | " + (f"{pct(c['felidezes'])} | {pct(c['pontossag'])} | "
                      f"{c['teves_per_perc']:.2f} |" if c else "nem érhető el | – | – |"))
        (out / "shot_detect_uj_meccs.md").write_text("\n".join(md) + "\n")
        print("\n".join(md))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile hbd/match_shots.py
"""Lövés-profil valódi meccsről: a meccs-lövésjelölő címkéiből tanulva.

Bemenet: a `tools/cut_match_clips.py` klipjei (manifest.json + v/*.mp4) és a
meccs-lövésjelölő oldal jelölései (klipenként: elengedés kockája, a lövőre
kattintás helye, kartartás, lövőkéz, célzóna, kimenet).

Lépések:
  1. Testpont-kinyerés a klipek MINDEN kockáján, minden emberre (`extract`).
  2. A lövő követése a kattintástól előre és hátra (`track_shooter`).
  3. Kiértékelés, jelenet szerint csoportosítva (az ismétlés ugyanannak a
     lövésnek a másik képe → az élő párjával egy csoportba kerül):
       - LÖVŐ-VÁLASZTÁS: az elengedés kockáján a képen lévő emberek közül
         melyik lő? (tanított rangsoroló vs. "legnagyobb" / "legmagasabb kar")
       - ELENGEDÉS, KARTARTÁS (fent / oldalt), LÖVŐKÉZ: a `release` modul
         ugyanazon csöve, amely az UNIRI-klipeken is fut.
       - CÉLZÓNA: vízszintes (bal/közép/jobb) és függőleges (lent/közép/fent)
         harmad a lövő testtartásából — van-e benne jel a "mindig a
         leggyakoribb" tipphez képest?

A kartartás edzői besorolás: a fej mellől induló lövés is "oldalt" (a
jelölő megerősítése szerint), tehát két osztály: fent / oldalt.

    python -m hbd.match_shots extract --clips m1web --video meccs.mp4 --out m1_clip_poses
    python -m hbd.match_shots zoom --clips m1web --labels cimkek_m1/ --video meccs.mp4 --out m1_zoom
    python -m hbd.match_shots evaluate --clips m1web --poses m1_clip_poses --labels cimkek_m1/ --zoom m1_zoom --out res/
"""

from __future__ import annotations

import argparse
import json
import time
from pathlib import Path

import numpy as np

from .features import L_SH, L_WR, NOSE, R_SH, R_WR

# A pózbecslő az EREDETI (1920 px-es) felvételen fut, 1280-as bemenettel: a 960 px-es
# klipen széles képen a játékosok fele (köztük sokszor a lövő) nem kerül elő.
CLIP_IMGSZ = 1280
# A feldolgozott ablak a gombnyomáshoz képest (s). Az első meccsen az elengedés a
# nyomás előtt legfeljebb 0,4 s-mal, utána legfeljebb 1,2 s-mal volt.
WINDOW_S = (-1.2, 1.6)
# Követés: kockánként legfeljebb ennyit ugorhat a doboz közepe (doboz-magasságban),
# és ekkora méretváltozás fér bele (arány).
TRACK_JUMP = 0.45
TRACK_SIZE_RATIO = (0.6, 1.6)
# Ennyi ideig tartjuk meg a követést, ha a lövő egy-két kockán nem látszik (s).
TRACK_GAP_S = 0.2
# Az ismétlés legfeljebb ennyivel jön az élő lövés után (s) — csoportosításhoz.
REPLAY_MAX_LAG_S = 120.0
# Lövő-választás: a jelöltek mozgását ekkora ablakban nézzük az elengedés körül (s).
PICK_WINDOW_S = 0.3
FOLDS = 5


# ------------------------------------------------------------ 1. testpontok
def _orig_frames(video: str, t0: float, n: int):
    """n kocka az eredeti felvételből a t0 időponttól (BGR)."""
    import subprocess

    from .video import ffmpeg_exe, probe

    w, h, _ = probe(video)
    # -fps_mode passthrough: az alap-időzítés a pontos ugrásnál néha megduplázza az első
    # kockát ([F, F, F+1, …]) → minden későbbi kocka 1-gyel elcsúszna.
    cmd = [ffmpeg_exe(), "-v", "error", "-ss", f"{max(0.0, t0):.3f}", "-i", video, "-fps_mode", "passthrough",
           "-frames:v", str(n), "-f", "rawvideo", "-pix_fmt", "bgr24", "-"]
    raw = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=True).stdout
    k = len(raw) // (w * h * 3)
    return list(np.frombuffer(raw[:k * w * h * 3], np.uint8).reshape(k, h, w, 3))


def extract(clips_dir: Path, out_dir: Path, weights: str, device: str | None, imgsz: int = CLIP_IMGSZ,
            batch: int = 8, video: str | None = None) -> None:
    """Az összes ember testpontja a klipek kockáin → out_dir/<id>.npz (folytatható).

    video megadásával az eredeti felvétel kockáin, csak a gombnyomás körüli
    WINDOW_S ablakban; a kimenet `k0` mezője az ablak első kockája a klipben."""
    from ultralytics import YOLO

    from .video import iter_frames

    man = json.loads((clips_dir / "manifest.json").read_text())
    out_dir.mkdir(parents=True, exist_ok=True)
    model = YOLO(weights)
    t0, done = time.time(), 0
    for c in man["clips"]:
        dst = out_dir / f"{c['id']}.npz"
        if dst.exists():
            continue
        fps = float(c["fps"])
        if video:
            k0 = max(0, c["press_k"] + int(round(WINDOW_S[0] * fps)))
            k1 = min(c["frames"], c["press_k"] + int(round(WINDOW_S[1] * fps)) + 1)
            frames = _orig_frames(video, c["t_start"] + k0 / fps, k1 - k0)
        else:
            k0, frames = 0, list(iter_frames(clips_dir / c["src"]))
        n, boxes, kps = [], [], []
        for i in range(0, len(frames), batch):
            for r in model.predict(frames[i:i + batch], imgsz=imgsz, conf=0.1, device=device, verbose=False):
                if r.keypoints is not None and len(r.boxes):
                    b = r.boxes.xyxy.cpu().numpy()
                    xy = r.keypoints.xy.cpu().numpy()
                    kc = r.keypoints.conf
                    kc = kc.cpu().numpy() if kc is not None else np.ones(xy.shape[:2])
                    k = np.concatenate([xy, kc[..., None]], -1)
                else:
                    b, k = np.zeros((0, 4)), np.zeros((0, 17, 3))
                n.append(len(b))
                boxes.append(b.astype(np.float32))
                kps.append(k.astype(np.float32))
        h, w = frames[0].shape[:2] if frames else (0, 0)
        np.savez_compressed(dst, n=np.array(n, np.int32), offsets=np.concatenate([[0], np.cumsum(n)]),
                            boxes=np.concatenate(boxes) if sum(n) else np.zeros((0, 4), np.float32),
                            kps=np.concatenate(kps) if sum(n) else np.zeros((0, 17, 3), np.float32),
                            size=np.array([w, h], np.int32), fps=np.float32(c["fps"]), k0=np.int32(k0))
        done += 1
        el = time.time() - t0
        print(f"  {c['id']}: {len(frames)} kocka · {el / done:.1f} s/klip", flush=True)


def load_clip_pose(path: Path) -> dict:
    z = np.load(path)
    return {k: z[k] for k in z.files}


def frame_people(p: dict, f: int) -> tuple[np.ndarray, np.ndarray]:
    a, b = p["offsets"][f], p["offsets"][f + 1]
    return p["boxes"][a:b], p["kps"][a:b]


# ------------------------------------------------------------ 2. a lövő követése
def _center(b: np.ndarray) -> np.ndarray:
    return np.stack([(b[..., 0] + b[..., 2]) / 2, (b[..., 1] + b[..., 3]) / 2], -1)


def pick_clicked(boxes: np.ndarray, x: float, y: float) -> int | None:
    """A kattintott ember: akinek a dobozában van a pont (több közül a középhez legközelebbi)."""
    if len(boxes) == 0:
        return None
    c = _center(boxes)
    hgt = np.maximum(boxes[:, 3] - boxes[:, 1], 1.0)
    inside = (boxes[:, 0] <= x) & (x <= boxes[:, 2]) & (boxes[:, 1] <= y) & (y <= boxes[:, 3])
    d = np.hypot(c[:, 0] - x, c[:, 1] - y) / hgt
    if inside.any():
        d = np.where(inside, d, np.inf)
    j = int(np.argmin(d))
    return j if d[j] <= 0.75 else None


def _step(prev_box: np.ndarray, boxes: np.ndarray, gap: int) -> int | None:
    if len(boxes) == 0:
        return None
    ph = max(prev_box[3] - prev_box[1], 1.0)
    hgt = boxes[:, 3] - boxes[:, 1]
    d = np.linalg.norm(_center(boxes) - _center(prev_box), axis=1) / ph
    ratio = hgt / ph
    ok = (d <= TRACK_JUMP * (1 + gap)) & (ratio >= TRACK_SIZE_RATIO[0]) & (ratio <= TRACK_SIZE_RATIO[1])
    if not ok.any():
        return None
    return int(np.argmin(np.where(ok, d, np.inf)))


def track_shooter(p: dict, k: int, x: float, y: float) -> dict | None:
    """A lövő testpont-sorozata a kattintás kockájától előre és hátra.

    Visszaad: {"start": első kocka, "kps": (T,17,3), "boxes": (T,4), "k": a kattintás
    kockája a sorozatban}; None, ha a kattintásnál nincs ember."""
    T = len(p["n"])
    k = int(min(max(k, 0), T - 1))
    boxes, kps = frame_people(p, k)
    j = pick_clicked(boxes, x, y)
    if j is None:
        return None
    max_gap = max(1, int(round(TRACK_GAP_S * float(p["fps"]))))
    seq = {k: (boxes[j], kps[j])}
    for direction in (1, -1):
        last_box, gap, f = boxes[j], 0, k + direction
        while 0 <= f < T and gap <= max_gap:
            bb, kk = frame_people(p, f)
            i = _step(last_box, bb, gap)
            if i is None:
                gap += 1
            else:
                seq[f] = (bb[i], kk[i])
                last_box, gap = bb[i], 0
            f += direction
    a, b = min(seq), max(seq) + 1
    out_b = np.zeros((b - a, 4), np.float32)
    out_k = np.zeros((b - a, 17, 3), np.float32)
    last = None
    for f in range(a, b):
        if f in seq:
            last = seq[f]
            out_b[f - a], out_k[f - a] = seq[f]
        else:                      # rövid kiesés: a doboz marad, a testpont "nem látszik"
            out_b[f - a] = last[0]
            out_k[f - a, :, :2] = last[1][:, :2]
    return {"start": a, "kps": out_k, "boxes": out_b, "k": k - a}


def refine_all(clips_dir: Path, poses_dir: Path, labels_dir: Path, video: str, out_dir: Path, weights: str,
               device: str | None = None) -> None:
    """A követett lövő testpontjai újra, a köré vágott, felnagyított képen (folytatható).

    Széles képen a lövő ~100 px magas, a teljes képen futó pózbecslő karja
    zajos. A UNIRI-csőhöz hasonlóan a lövő dobozát négyzetesen kivágjuk
    (CROP_SCALE), és a kivágáson fut újra a becslő (POSE_IMGSZ); a doboza
    alapján választjuk ki (pick_person)."""
    from .pose import PoseModel, pick_person, square_crop_box

    man = {c["id"]: c for c in json.loads((clips_dir / "manifest.json").read_text())["clips"]}
    labels = load_labels(labels_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    model = PoseModel(weights, device)
    t0, done = time.time(), 0
    for cid, lab in labels.items():
        dst, f = out_dir / f"{cid}.npz", poses_dir / f"{cid}.npz"
        if dst.exists() or not f.exists():
            continue
        p = load_clip_pose(f)
        k0, (w, h), fps = int(p.get("k0", 0)), p["size"], float(p["fps"])
        tr = track_shooter(p, lab["lovo"]["k"] - k0, lab["lovo"]["x"] * w, lab["lovo"]["y"] * h)
        if tr is None:
            continue
        T = len(tr["boxes"])
        frames = _orig_frames(video, man[cid]["t_start"] + (k0 + tr["start"]) / fps, T)
        crops, origins = [], []
        for img, box in zip(frames, tr["boxes"]):
            x0, y0, x1, y1 = square_crop_box(box, w, h)
            crops.append(np.ascontiguousarray(img[y0:y1, x0:x1]))
            origins.append((x0, y0))
        kps = tr["kps"][:len(crops)].copy()
        found = np.zeros(len(crops), bool)
        for t, (db, dk) in enumerate(model(crops)):
            ox, oy = origins[t]
            tb = tr["boxes"][t]
            j = pick_person(db, (tb[0] - ox, tb[1] - oy, tb[2] - ox, tb[3] - oy))
            if j is not None:
                kps[t] = dk[j]
                kps[t, :, 0] += ox
                kps[t, :, 1] += oy
                found[t] = True
        np.savez_compressed(dst, start=np.int32(tr["start"]), kps=kps, found=found)
        done += 1
        print(f"  {cid}: {len(crops)} kocka, {found.mean():.0%} újrabecsülve · "
              f"{(time.time() - t0) / done:.1f} s/klip", flush=True)


# ------------------------------------------------------------ ráközelítő követés
# A kivágás oldala a lövő doboz-magasságának ennyiszerese (korlátok px-ben, 1080p-n).
ZOOM_SIDE_X = 2.5
ZOOM_SIDE_PX = (280, 900)
ZOOM_IMGSZ = 640
# Az induló keresés kivágásai (px). Nem lehet kisebb, mint egy közeli játékos: a csonkán
# kivágott testre a becslő csonka dobozt ad, és a követés ehhez a hibás mérethez igazodna
# (az első változat 280 px-ről indult, a lövő medián magassága 219 px volt).
ZOOM_START_SIDES = (450, 800)


def _zoom_detect(model, img, cx, cy, side):
    """Kivágás (cx, cy) körül, észlelés → dobozok, testpontok a teljes kép koordinátáiban."""
    h, w = img.shape[:2]
    half = side / 2
    x0, y0 = int(max(0, cx - half)), int(max(0, cy - half))
    x1, y1 = int(min(w, cx + half)), int(min(h, cy + half))
    (db, dk), = model([np.ascontiguousarray(img[y0:y1, x0:x1])])
    if len(db):
        db = db + [x0, y0, x0, y0]
        dk = dk.copy()
        dk[..., 0] += x0
        dk[..., 1] += y0
    return db, dk


def zoom_track(frames: list, k: int, x: float, y: float, model, fps: float) -> dict | None:
    """A lövő követése RÁKÖZELÍTVE: minden kockán az előző helye körüli kivágáson keresünk.

    Széles képen a lövő gyakran csak 60–100 px, a 6-os előtti tömegben a
    teljes képen futó becslő nem találja meg; a kivágáson (ZOOM_IMGSZ-re
    nagyítva) igen, és a testpontjai is pontosabbak. Indulás: a kattintás
    kockáján a kattintás körül; ha ott nincs találat, a szomszédos kockákon."""
    T = len(frames)
    k = int(min(max(k, 0), T - 1))
    start = None
    for dk_ in (0, -1, 1, -2, 2, -3, 3):
        f = k + dk_
        if not 0 <= f < T:
            continue
        for side in ZOOM_START_SIDES:
            db, dkp = _zoom_detect(model, frames[f], x, y, side)
            j = pick_clicked(db, x, y)
            if j is not None:
                start = (f, db[j], dkp[j])
                break
        if start:
            break
    if start is None:
        return None
    max_gap = max(1, int(round(TRACK_GAP_S * fps)))
    seq = {start[0]: (start[1], start[2])}
    for direction in (1, -1):
        last, gap, f = start[1], 0, start[0] + direction
        while 0 <= f < T and gap <= max_gap:
            hgt = max(last[3] - last[1], 1.0)
            side = float(np.clip(ZOOM_SIDE_X * hgt, *ZOOM_SIDE_PX))
            c = _center(last)
            db, dkp = _zoom_detect(model, frames[f], c[0], c[1], side)
            i = _step(last, db, gap)
            if i is None:
                gap += 1
            else:
                seq[f] = (db[i], dkp[i])
                last, gap = db[i], 0
            f += direction
    a, b = min(seq), max(seq) + 1
    out_b = np.zeros((b - a, 4), np.float32)
    out_k = np.zeros((b - a, 17, 3), np.float32)
    last = None
    for f in range(a, b):
        if f in seq:
            last = seq[f]
            out_b[f - a], out_k[f - a] = seq[f]
        else:
            out_b[f - a] = last[0]
            out_k[f - a, :, :2] = last[1][:, :2]
    return {"start": a, "kps": out_k, "boxes": out_b, "k": k - a}


def zoom_track_all(clips_dir: Path, labels_dir: Path, video: str, out_dir: Path, weights: str,
                   device: str | None = None) -> None:
    """Minden címkézett klip lövőjének ráközelítő követése → out_dir/<id>.npz (folytatható)."""
    from .pose import PoseModel

    man = {c["id"]: c for c in json.loads((clips_dir / "manifest.json").read_text())["clips"]}
    labels = load_labels(labels_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    model = PoseModel(weights, device, imgsz=ZOOM_IMGSZ, batch=1)
    t0, done = time.time(), 0
    for cid, lab in labels.items():
        dst = out_dir / f"{cid}.npz"
        if dst.exists() or cid not in man:
            continue
        c = man[cid]
        fps = float(c["fps"])
        k0 = max(0, c["press_k"] + int(round(WINDOW_S[0] * fps)))
        k1 = min(c["frames"], c["press_k"] + int(round(WINDOW_S[1] * fps)) + 1)
        frames = _orig_frames(video, c["t_start"] + k0 / fps, k1 - k0)
        h, w = frames[0].shape[:2]
        tr = zoom_track(frames, lab["lovo"]["k"] - k0, lab["lovo"]["x"] * w, lab["lovo"]["y"] * h, model, fps)
        done += 1
        if tr is None:
            np.savez_compressed(dst, lost=True)
            print(f"  {cid}: NINCS lövő a kattintásnál", flush=True)
            continue
        np.savez_compressed(dst, lost=False, start=np.int32(tr["start"]), kps=tr["kps"], boxes=tr["boxes"],
                            k0=np.int32(k0), fps=np.float32(fps), size=np.array([w, h], np.int32))
        print(f"  {cid}: {len(tr['kps'])}/{len(frames)} kocka · {(time.time() - t0) / done:.1f} s/klip", flush=True)


# ------------------------------------------------------------ címkék, csoportok
def load_labels(labels_dir: Path) -> dict[str, dict]:
    """A meccs-lövésjelölő dokumentumai (klip → címke); a "rossz" klipek kimaradnak."""
    out = {}
    for f in sorted(Path(labels_dir).glob("*.json")):
        d = json.loads(f.read_text())
        d = d.get("data", d)
        if d.get("rossz") or d.get("elengedes") is None or not d.get("lovo"):
            continue
        out[d["klip"]] = d
    return out


def replay_groups(clips: list[dict]) -> dict[str, str]:
    """Klip → csoport: az ismétlés a legutóbbi előtte lévő élő lövés csoportjába kerül."""
    live = sorted((c["t_video"], c["id"]) for c in clips if not c.get("ismetles"))
    out = {}
    for c in clips:
        if not c.get("ismetles"):
            out[c["id"]] = c["id"]
            continue
        prev = [i for t, i in live if 0 <= c["t_video"] - t <= REPLAY_MAX_LAG_S]
        out[c["id"]] = prev[-1] if prev else c["id"]
    return out


def arm_class(label: str | None) -> str | None:
    """Edzői besorolás: a fej melletti lövés is oldalt."""
    if label is None:
        return None
    return "fent" if label == "fent" else "oldalt"


# ------------------------------------------------------------ 3a. lövő-választás
def candidate_features(p: dict, r: int) -> tuple[np.ndarray, np.ndarray]:
    """Az r. kocka minden emberére: mennyire néz ki lövőnek (C, F) + a dobozok (C, 4).

    Rövid követés az r körüli ablakban: a csukló legmagasabb pontja a vállhoz
    és a fejhez képest, a csukló legnagyobb sebessége (doboz-magasság / s), a
    doboz mérete a képen lévő legnagyobbhoz képest, és a hely a képen."""
    fps = float(p["fps"])
    w, h = p["size"]
    boxes, kps = frame_people(p, r)
    if len(boxes) == 0:
        return np.zeros((0, 9)), boxes
    win = max(1, int(round(PICK_WINDOW_S * fps)))
    T = len(p["n"])
    hmax = max((boxes[:, 3] - boxes[:, 1]).max(), 1.0)
    feats = []
    for j in range(len(boxes)):
        hj = max(boxes[j, 3] - boxes[j, 1], 1.0)
        traj = {r: kps[j]}
        for direction in (1, -1):
            last, f = boxes[j], r + direction
            while 0 <= f < T and abs(f - r) <= win:
                bb, kk = frame_people(p, f)
                i = _step(last, bb, 0)
                if i is None:
                    break
                traj[f] = kk[i]
                last = bb[i]
                f += direction
        up, head, spd = -1.0, -1.0, 0.0
        fs = sorted(traj)
        for f in fs:
            k = traj[f]
            for wr, sh in ((L_WR, L_SH), (R_WR, R_SH)):
                if k[wr, 2] > 0.3 and k[sh, 2] > 0.3:
                    up = max(up, (k[sh, 1] - k[wr, 1]) / hj)
                if k[wr, 2] > 0.3 and k[NOSE, 2] > 0.3:
                    head = max(head, (k[NOSE, 1] - k[wr, 1]) / hj)
        for f0, f1 in zip(fs, fs[1:]):
            a, b = traj[f0], traj[f1]
            for wr in (L_WR, R_WR):
                if a[wr, 2] > 0.3 and b[wr, 2] > 0.3:
                    spd = max(spd, np.linalg.norm(b[wr, :2] - a[wr, :2]) / hj * fps / (f1 - f0))
        cx, cy = (boxes[j, 0] + boxes[j, 2]) / 2 / w, (boxes[j, 1] + boxes[j, 3]) / 2 / h
        feats.append([up, head, spd, hj / hmax, hj / h, cx, cy, abs(cx - 0.5), len(fs) / (2 * win + 1)])
    return np.array(feats, np.float32), boxes


def evaluate_picker(items: list[dict], seed: int = 0) -> dict:
    """Lövő-választás csoportosított keresztvalidációval: top-1 találat a klipek között."""
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.model_selection import GroupKFold

    data = [it for it in items if len(it["X"]) >= 1 and it["target"] is not None]
    groups = np.array([it["group"] for it in data])
    hit_model = np.zeros(len(data), bool)
    for tr, te in GroupKFold(FOLDS).split(np.zeros(len(data)), groups=groups):
        X = np.concatenate([data[i]["X"] for i in tr])
        y = np.concatenate([np.arange(len(data[i]["X"])) == data[i]["target"] for i in tr]).astype(int)
        clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                             class_weight="balanced", random_state=seed).fit(X, y)
        for i in te:
            hit_model[i] = int(np.argmax(clf.predict_proba(data[i]["X"])[:, 1])) == data[i]["target"]
    hit_big = np.array([int(np.argmax(it["X"][:, 3])) == it["target"] for it in data])
    hit_arm = np.array([int(np.argmax(it["X"][:, 0])) == it["target"] for it in data])
    n_people = np.array([len(it["X"]) for it in data])
    # a teljes képen elő sem került lövő: egyik módszer sem találhatja el
    n_all = len(items)
    return {"klipek": len(data), "osszes_klip": n_all, "lovo_nincs_a_jeloltek_kozt": n_all - len(data),
            "modell_az_osszesbol": float(hit_model.sum() / max(1, n_all)),
            "emberek_atlag": float(n_people.mean()),
            "veletlen": float((1 / n_people).mean()),
            "legnagyobb_ember": float(hit_big.mean()), "legmagasabb_kar": float(hit_arm.mean()),
            "modell": float(hit_model.mean())}


# ------------------------------------------------------------ 3c. célzóna
ZONE_H = {"bal": 0, "kozep": 1, "jobb": 2}
ZONE_V = {"lent": 0, "kozep": 1, "fent": 2}


def split_zone(z: str | None) -> tuple[int | None, int | None]:
    """'lent-jobb' → (függőleges, vízszintes); 'kozep' → (1, 1)."""
    if not z:
        return None, None
    if z == "kozep":
        return 1, 1
    v, hz = z.split("-")
    return ZONE_V[v], ZONE_H[hz]


def evaluate_zone(clips: list[dict], labels: dict, seed: int = 0) -> dict:
    """A célzóna harmadai a lövő testtartásából (a kézi elengedésnél) — van-e jel benne?"""
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.model_selection import GroupKFold

    from .release import _clf, arm_hand_features

    rows = [(c, split_zone(labels[c["name"]].get("zona"))) for c in clips]
    rows = [(c, z) for c, z in rows if z[0] is not None]
    X = np.stack([np.concatenate([arm_hand_features(c["kps"], c["boxes"], c["fps"], c["release"]),
                                  [labels[c["name"]]["lovokez"] == "bal", labels[c["name"]]["lovo"]["x"],
                                   labels[c["name"]]["lovo"]["y"]]]) for c, _ in rows]).astype(np.float64)
    groups = np.array([c["scene_id"] for c, _ in rows])
    out = {"klipek": len(rows)}
    for name, idx in (("fuggoleges", 0), ("vizszintes", 1)):
        y = np.array([z[idx] for _, z in rows])
        pred = np.zeros_like(y)
        for tr, te in GroupKFold(FOLDS).split(X, y, groups):
            pred[te] = _clf().fit(X[tr], y[tr]).predict(X[te])
        major = np.bincount(y).argmax()
        out[name] = {"eloszlas": np.bincount(y, minlength=3).tolist(),
                     "modell": float((pred == y).mean()),
                     "modell_kiegyensulyozott": float(balanced_accuracy_score(y, pred)),
                     "mindig_leggyakoribb": float((y == major).mean()),
                     "kiegyensulyozott_veletlen": 1 / 3}
    return out


# ------------------------------------------------------------ összefűzés
def build(clips_dir: Path, poses_dir: Path, labels_dir: Path, zoom_dir: Path | None = None):
    """A címkézett klipekből: lövő-sorozatok (a `release` modul formátumában) és lövő-választási tételek."""
    man = json.loads((clips_dir / "manifest.json").read_text())
    labels = load_labels(labels_dir)
    groups = replay_groups(man["clips"])
    seqs, pick, rel_labels, lost = [], [], {}, []
    for c in man["clips"]:
        lab = labels.get(c["id"])
        f = poses_dir / f"{c['id']}.npz"
        if lab is None or not f.exists():
            continue
        p = load_clip_pose(f)
        k0 = int(p.get("k0", 0))
        w, h = p["size"]
        lv = lab["lovo"]
        tr = None
        if zoom_dir is not None and (zoom_dir / f"{c['id']}.npz").exists():
            z = np.load(zoom_dir / f"{c['id']}.npz")
            if not bool(z["lost"]) and int(z["k0"]) == k0:
                tr = {"start": int(z["start"]), "kps": z["kps"], "boxes": z["boxes"]}
        if tr is None:
            # a kattintás általában az elengedés kockáján van; ha az ablakon kívül esne, onnan indulunk
            tr = track_shooter(p, lv["k"] - k0, lv["x"] * w, lv["y"] * h)
        if tr is None:
            lost.append(c["id"])
            continue
        r = lab["elengedes"] - k0 - tr["start"]
        if not (0 <= r < len(tr["kps"])):
            lost.append(c["id"])
            continue
        seqs.append({"name": c["id"], "kps": tr["kps"], "boxes": tr["boxes"], "fps": float(p["fps"]),
                     "scene_id": groups[c["id"]], "release": r, "ismetles": bool(c.get("ismetles"))})
        rel_labels[c["id"]] = {"elengedes": r, "kartartas": arm_class(lab["kartartas"]),
                               "lovokez": lab["lovokez"]}
        # lövő-választás az elengedés kockáján: a célpont a követett lövő doboza
        X, boxes = candidate_features(p, lab["elengedes"] - k0)
        target = None
        if len(boxes):
            sb = tr["boxes"][r]
            ious = [_iou(sb, b) for b in boxes]
            target = int(np.argmax(ious)) if max(ious) > 0.5 else None
        pick.append({"name": c["id"], "X": X, "target": target, "group": groups[c["id"]]})
    return seqs, pick, rel_labels, labels, lost


def _iou(a, b) -> float:
    x0, y0, x1, y1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x1 - x0) * max(0.0, y1 - y0)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return inter / ua if ua > 0 else 0.0


def write_report(res: dict, out: Path, title: str) -> Path:
    pct = lambda v: f"{100 * v:.0f}%"  # noqa: E731
    pk, rl, z = res["lovo_valasztas"], res["release"], res["celzona"]
    e = rl["elengedes"]
    md = [f"# Lövés-profil valódi meccsről — {title}", "",
          f"{res['cimkezett']} címkézett lövés-klip, ebből {res['kovetett']} lövőjét sikerült végigkövetni "
          f"(élő + ismétlés; az ismétlés az élő párjával egy csoportban, {FOLDS}-szörös csoportosított "
          "keresztvalidáció).", "",
          "## Ki lő? (lövő-választás az elengedés pillanatában)", "",
          f"Átlagosan {pk['emberek_atlag']:.1f} ember a képen. {pk['lovo_nincs_a_jeloltek_kozt']} klipen a lövőt a "
          "teljes képen futó becslő meg sem találta (ott egyik módszer sem választhatja); a táblázat a többi "
          f"{pk['klipek']} klipre vonatkozik, a tanított rangsoroló az összesből: {pct(pk['modell_az_osszesbol'])}.", "",
          "| módszer | eltalálja a lövőt |", "|---|---|",
          f"| véletlen | {pct(pk['veletlen'])} |",
          f"| a legnagyobb ember | {pct(pk['legnagyobb_ember'])} |",
          f"| a legmagasabbra emelt kar | {pct(pk['legmagasabb_kar'])} |",
          f"| **tanított rangsoroló** | **{pct(pk['modell'])}** |", "",
          "## Elengedés pillanata (a lövő sorozatán)", "",
          "| | medián hiba | ±2 kockán belül | 100 ms-on belül |", "|---|---|---|---|",
          f"| tanított háló | {e['modell']['median_hiba_ms']:.0f} ms | {pct(e['modell']['pontos_2_kockan_belul'])} | "
          f"{pct(e['modell']['pontos_100ms_belul'])} |",
          f"| régi kézi szabály | {e['regi_szabaly']['median_hiba_ms']:.0f} ms | "
          f"{pct(e['regi_szabaly']['pontos_2_kockan_belul'])} | {pct(e['regi_szabaly']['pontos_100ms_belul'])} |", "",
          "## Kartartás (fent / oldalt — a fej melletti is oldalt) és lövőkéz", "",
          f"Osztályok: kartartás {rl['osztalyok']['kartartas']}, lövőkéz {rl['osztalyok']['lovokez']}.", "",
          "| | pontosság | kiegyensúlyozott |", "|---|---|---|"]
    for k, lab in (("modell", "kartartás — modell (a talált elengedésnél)"),
                   ("modell_kezi_elengedesnel", "kartartás — modell (a kézi elengedésnél)"),
                   ("mindig_fent", "kartartás — mindig „fent”")):
        v = rl["kartartas"][k]
        md.append(f"| {lab} | {pct(v['pontossag'])} | {pct(v['kiegyensulyozott'])} |")
    for k, lab in (("modell", "lövőkéz — modell"), ("mindig_jobb", "lövőkéz — mindig „jobb”")):
        v = rl["lovokez"][k]
        md.append(f"| {lab} | {pct(v['pontossag'])} | {pct(v['kiegyensulyozott'])} |")
    md += ["", "## Célzóna a testtartásból (harmadok)", "",
           "| | eloszlás (0/1/2) | modell | kiegyensúlyozott | mindig a leggyakoribb |", "|---|---|---|---|---|"]
    for k, lab in (("fuggoleges", "függőleges (lent/közép/fent)"), ("vizszintes", "vízszintes (bal/közép/jobb)")):
        v = z[k]
        md.append(f"| {lab} | {v['eloszlas']} | {pct(v['modell'])} | {pct(v['modell_kiegyensulyozott'])} | "
                  f"{pct(v['mindig_leggyakoribb'])} |")
    p = out / "match_shots.md"
    p.write_text("\n".join(md) + "\n")
    return p


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description="Lövés-profil valódi meccsről")
    sub = ap.add_subparsers(dest="cmd", required=True)
    e = sub.add_parser("extract")
    e.add_argument("--clips", required=True)
    e.add_argument("--out", required=True)
    e.add_argument("--weights", default="yolo11n-pose.pt")
    e.add_argument("--device", default=None)
    e.add_argument("--imgsz", type=int, default=CLIP_IMGSZ)
    e.add_argument("--video", help="az eredeti meccsfelvétel (ajánlott: nagyobb felbontás)")
    v = sub.add_parser("evaluate")
    v.add_argument("--clips", required=True)
    v.add_argument("--poses", required=True)
    v.add_argument("--labels", required=True)
    v.add_argument("--out", required=True)
    v.add_argument("--title", default="1. meccs")
    v.add_argument("--epochs", type=int, default=120)
    v.add_argument("--zoom", help="a zoom lépés kimenete (a lövő ráközelítő követése)")
    zm = sub.add_parser("zoom")
    zm.add_argument("--clips", required=True)
    zm.add_argument("--labels", required=True)
    zm.add_argument("--video", required=True)
    zm.add_argument("--out", required=True)
    zm.add_argument("--weights", default="yolo11n-pose.pt")
    zm.add_argument("--device", default=None)
    rf = sub.add_parser("refine")
    rf.add_argument("--clips", required=True)
    rf.add_argument("--poses", required=True)
    rf.add_argument("--labels", required=True)
    rf.add_argument("--video", required=True)
    rf.add_argument("--out", required=True)
    rf.add_argument("--weights", default="yolo11n-pose.pt")
    rf.add_argument("--device", default=None)
    args = ap.parse_args(argv)
    if args.cmd == "extract":
        extract(Path(args.clips), Path(args.out), args.weights, args.device, args.imgsz, video=args.video)
        return 0
    if args.cmd == "zoom":
        zoom_track_all(Path(args.clips), Path(args.labels), args.video, Path(args.out), args.weights, args.device)
        return 0
    if args.cmd == "refine":
        refine_all(Path(args.clips), Path(args.poses), Path(args.labels), args.video, Path(args.out),
                   args.weights, args.device)
        return 0
    from .release import evaluate as eval_release

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    seqs, pick, rel_labels, labels, lost = build(Path(args.clips), Path(args.poses), Path(args.labels),
                                                 Path(args.zoom) if args.zoom else None)
    res = {"cimkezett": len(labels), "kovetett": len(seqs), "elveszett": lost,
           "lovo_valasztas": evaluate_picker(pick),
           "release": eval_release(seqs, rel_labels, epochs=args.epochs),
           "celzona": evaluate_zone(seqs, labels)}
    (out / "match_shots.json").write_text(json.dumps(res, ensure_ascii=False, indent=1))
    print(write_report(res, out, args.title).read_text())
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile hbd/ball.py
"""Labdafelismerő a közvetítésen: tanítás a labdajelölő kattintásaiból.

A kész (COCO "sports ball") felismerők a közvetítésen nem találják a labdát:
az elengedésnél 10–25 px-es és elmosódott. Itt egy kis YOLO11 detektort
tanítunk a labdajelölő pontjain (klipenként kézben / elválik / repül).

Adat:
  - a jelölt kocka a labdajelölő éles, lövő köré vágott videójából (640 px, 1:1
    az eredeti 1080p-hez);
  - a labda doboza: középpont = a kattintás, átmérő = BALL_REL × a lövő
    doboz-magassága (kézilabda Ø 19 cm, játékos ~1,9 m), BALL_PX korlátok közt;
  - tanító képek: TILE-os kivágások, POS_PER_POINT olyan, amin a labda rajta van
    (véletlen helyen), és NEG_PER_POINT olyan, amin nincs (egy pályán egy labda
    van → ez biztos "nincs labda" példa).

Kiértékelés: a lövések (csoportok: élő + ismétlése együtt) HOLDOUT része
kimarad a tanításból; azokon a teljes 640-es kivágáson fut a felismerő, és
mérjük: a legbiztosabb jelölt a valódi labdától legfeljebb HIT_X × átmérőre
van-e.

    python -m hbd.ball build --ball-dir m1ball2 --labels cimkek/meccs1_labda_cimkek.json --zoom m1_zoom --out balldata
    python -m hbd.ball train --data balldata --epochs 40
    python -m hbd.ball evaluate --data balldata --weights runs/.../best.pt
"""

from __future__ import annotations

import argparse
import json
import random
from pathlib import Path

import numpy as np

BALL_REL = 0.10
BALL_PX = (8, 40)
TILE = 320
POS_PER_POINT = 3
NEG_PER_POINT = 1
HOLDOUT = 0.2
HIT_X = 1.5
# A kattintások közötti kockák kitöltése (csak tanításhoz), legfeljebb ekkora hézagban.
INTERPOLATE = True
INTERP_MAX_GAP = 8
SEED = 0
# Öntanítás: a jelöletlen kockákon a követett (összefésült) jelölt, ha legalább ennyire biztos.
# A félretett lövéseken mérve 0,1 fölött a jelöltek 89%-a, 0,2 fölött 90%-a valódi labda.
PSEUDO_MIN_CONF = 0.15
PSEUDO_POS_PER_POINT = 1


def read_video_frames(path: Path) -> list[np.ndarray]:
    """Minden kocka (BGR) egy rövid videóból."""
    from .video import iter_frames

    return list(iter_frames(path))


def ball_size(zoom: dict | None, local_k: int, default: float = 20.0) -> float:
    """A labda becsült átmérője px-ben a lövő dobozmagasságából (a ráközelítő követésből)."""
    if zoom is None or bool(zoom["lost"]):
        return default
    t = int(np.clip(local_k - int(zoom["start"]), 0, len(zoom["boxes"]) - 1))
    b = zoom["boxes"][t]
    return float(np.clip(BALL_REL * (b[3] - b[1]), *BALL_PX))


def split_groups(ids: list[str], groups: dict[str, str]) -> set[str]:
    """A kiértékelésre félretett klipek (csoportonként, determinisztikusan)."""
    gs = sorted({groups[i] for i in ids})
    rng = random.Random(SEED)
    rng.shuffle(gs)
    held = set(gs[:int(round(HOLDOUT * len(gs)))])
    return {i for i in ids if groups[i] in held}


def _tile_with(rng, cx, cy, size, W, H):
    x0 = int(np.clip(rng.uniform(cx - TILE + size, cx - size), 0, W - TILE))
    y0 = int(np.clip(rng.uniform(cy - TILE + size, cy - size), 0, H - TILE))
    return x0, y0


def _tile_without(rng, cx, cy, size, W, H, tries=50):
    for _ in range(tries):
        x0, y0 = rng.randint(0, W - TILE), rng.randint(0, H - TILE)
        if not (x0 - size <= cx <= x0 + TILE + size and y0 - size <= cy <= y0 + TILE + size):
            return x0, y0
    return None


def interpolate_points(pontok: list[dict], max_gap: int = INTERP_MAX_GAP) -> list[dict]:
    """A kattintások KÖZÖTTI kockák labdahelye egyenes vonalú közelítéssel.

    Kézben → elválik: 1–3 kocka, a kéz rövid íve jól közelíthető egyenessel;
    elválik → repül: a labda röppályája ilyen rövid szakaszon szinte egyenes.
    Csak a tanító lövésekre (a kiértékelés a valódi kattintásokon fut)."""
    pts = sorted((p for p in pontok if not p.get("nincs")), key=lambda p: p["k"])
    out = []
    for a, b in zip(pts, pts[1:]):
        gap = b["k"] - a["k"]
        if not 2 <= gap <= max_gap:
            continue
        for k in range(a["k"] + 1, b["k"]):
            w = (k - a["k"]) / gap
            out.append({"nev": f"koztes{k}", "k": k, "cx": a["cx"] + w * (b["cx"] - a["cx"]),
                        "cy": a["cy"] + w * (b["cy"] - a["cy"])})
    return out


def build(ball_dir: Path, labels: Path, zoom_dir: Path | None, clips_manifest: Path, out: Path,
          pseudo: dict[str, list[dict]] | None = None) -> dict:
    """YOLO-formátumú adathalmaz (images/labels train+val) + a kiértékelő pontok listája.

    pseudo: a gépi (öntanított) labdapontok klipenként — csak a TANÍTÓ klipekre kerülnek be,
    pontonként PSEUDO_POS_PER_POINT pozitív kivágással és negatív nélkül (ha a gépi pont
    téves, a "nincs labda" kivágáson épp a valódi labda lehetne)."""
    import cv2

    from .match_shots import replay_groups

    man = {c["id"]: c for c in json.loads((ball_dir / "ball_manifest.json").read_text())["klipek"]}
    groups = replay_groups(json.loads(clips_manifest.read_text())["clips"])
    L = {d["klip"]: d for d in json.loads(labels.read_text())["cimkek"]}
    ids = sorted(i for i in L if i in man)
    held = split_groups(ids, groups)
    for sub in ("images/train", "labels/train", "images/val", "labels/val", "full"):
        (out / sub).mkdir(parents=True, exist_ok=True)
    rng = random.Random(SEED)
    evalpts, n_tr = [], 0
    for cid in ids:
        m = man[cid]
        frames = read_video_frames(ball_dir / m["src"])
        zf = (zoom_dir / f"{cid}.npz") if zoom_dir else None
        z = np.load(zf) if zf is not None and zf.exists() else None
        # a zoom kimenet kockái a lövés-klip k0-jához mérve; a labdavideó saját k0-ja más
        pts = list(L[cid]["pontok"])
        if cid not in held and INTERPOLATE:
            pts += interpolate_points(pts)
        if cid not in held and pseudo:
            have = {p["k"] for p in pts if not p.get("nincs")}
            pts += [q for q in pseudo.get(cid, []) if q["k"] not in have]
        for p in pts:
            if p.get("nincs"):
                continue
            f = p["k"] - m["k0"]
            if not 0 <= f < len(frames):
                continue
            img = frames[f]
            H, W = img.shape[:2]
            cx, cy = p["cx"] * W, p["cy"] * H
            zk = p["k"] - int(z["k0"]) if z is not None else 0
            size = ball_size(z, zk)
            name = f"{cid}_{p['nev']}"
            if cid in held:
                cv2.imwrite(str(out / "full" / f"{name}.jpg"), img)
                evalpts.append({"klip": cid, "nev": p["nev"], "kep": f"full/{name}.jpg", "x": cx, "y": cy, "d": size})
                split = "val"
                reps = 1
            else:
                split = "train"
                reps = PSEUDO_POS_PER_POINT if p.get("gepi") else POS_PER_POINT
            for r in range(reps):
                x0, y0 = _tile_with(rng, cx, cy, size, W, H)
                tile = img[y0:y0 + TILE, x0:x0 + TILE]
                stem = f"{name}_p{r}"
                cv2.imwrite(str(out / f"images/{split}/{stem}.jpg"), tile)
                (out / f"labels/{split}/{stem}.txt").write_text(
                    f"0 {(cx - x0) / TILE:.5f} {(cy - y0) / TILE:.5f} {size / TILE:.5f} {size / TILE:.5f}\n")
                n_tr += split == "train"
            for r in range(NEG_PER_POINT if split == "train" and not p.get("gepi") else 0):
                pos = _tile_without(rng, cx, cy, size, W, H)
                if pos is None:
                    continue
                x0, y0 = pos
                stem = f"{name}_n{r}"
                cv2.imwrite(str(out / f"images/{split}/{stem}.jpg"), img[y0:y0 + TILE, x0:x0 + TILE])
                (out / f"labels/{split}/{stem}.txt").write_text("")
    (out / "data.yaml").write_text(f"path: {out.resolve()}\ntrain: images/train\nval: images/val\nnames:\n  0: labda\n")
    (out / "eval_points.json").write_text(json.dumps(evalpts))
    (out / "labels_full.json").write_text(json.dumps({cid: [p for p in L[cid]["pontok"] if not p.get("nincs")]
                                                      for cid in sorted(held)}))
    info = {"klipek": len(ids), "kiertekelo_klipek": len(held), "tanito_kepek": n_tr, "kiertekelo_pontok": len(evalpts)}
    (out / "info.json").write_text(json.dumps(info))
    return info


def evaluate(data: Path, weights: str, imgsz: int = 640, conf: float = 0.01) -> dict:
    """A félretett lövések teljes kivágásain: eltalálja-e a labdát (a legbiztosabb jelölt)."""
    from ultralytics import YOLO

    model = YOLO(weights)
    pts = json.loads((data / "eval_points.json").read_text())
    by = {}
    for p in pts:
        r = model.predict(str(data / p["kep"]), imgsz=imgsz, conf=conf, verbose=False)[0]
        b = r.boxes.xyxy.cpu().numpy()
        c = r.boxes.conf.cpu().numpy()
        hit, top = False, None
        if len(b):
            j = int(np.argmax(c))
            bx, by_ = (b[j, 0] + b[j, 2]) / 2, (b[j, 1] + b[j, 3]) / 2
            top = float(c[j])
            hit = float(np.hypot(bx - p["x"], by_ - p["y"])) <= HIT_X * p["d"]
        by.setdefault(p["nev"], []).append((hit, top))
    out = {}
    for nev, v in by.items():
        hits = np.array([h for h, _ in v])
        out[nev] = {"pontok": len(v), "talalat": float(hits.mean())}
    allh = np.array([h for v in by.values() for h, _ in v])
    out["osszes"] = {"pontok": int(len(allh)), "talalat": float(allh.mean())}
    return out


# Követés: kockánként ennyi jelölt, a hiányzó kocka "ára", és a mozgás korlátja
# labdaátmérőben / kocka (25 m/s, 25 fps → ~1 m/kocka ≈ 5 átmérő; a tartalék a becsült méret hibájára).
TRACK_TOPK = 5
# A 0,03 túl olcsó volt: a pálya inkább "eltűnt", mint hogy a kicsit
# gyengébb, de helyes jelöltet vegye (a félretett lövéseken mérve: 0,003 → 67%).
TRACK_MISS = np.log(0.003)
TRACK_FREE_JUMP = 6.0
TRACK_JUMP_W = 0.5
# A követés ablaka a (korábbi, meccs-lövésjelölős) elengedéshez képest, kockában.
TRACK_WINDOW = (-8, 10)


def detect_candidates(model, frames: list[np.ndarray], imgsz: int = 640, conf: float = 0.01) -> list[np.ndarray]:
    """Kockánként a legjobb TRACK_TOPK jelölt: (x, y, átmérő, megbízhatóság)."""
    out = []
    for i in range(0, len(frames), 16):
        for r in model.predict(frames[i:i + 16], imgsz=imgsz, conf=conf, verbose=False):
            b = r.boxes.xyxy.cpu().numpy()
            c = r.boxes.conf.cpu().numpy()
            o = np.argsort(-c)[:TRACK_TOPK]
            out.append(np.stack([(b[o, 0] + b[o, 2]) / 2, (b[o, 1] + b[o, 3]) / 2,
                                 (b[o, 2] - b[o, 0] + b[o, 3] - b[o, 1]) / 2, c[o]], 1) if len(o) else np.zeros((0, 4)))
    return out


def track(cands: list[np.ndarray]) -> list[np.ndarray | None]:
    """A legvalószínűbb, egyenletes labdapálya (Viterbi): kockánként egy jelölt vagy "nem látszik".

    Pontszám: a jelöltek log-megbízhatósága, mínusz a túl nagy ugrás büntetése
    (TRACK_FREE_JUMP átmérő / kocka fölött négyzetesen). A "nem látszik" állapot
    ára TRACK_MISS; utána a következő jelölt a legutóbbi látott helyhez mérődik."""
    T = len(cands)
    if T == 0:
        return []
    # állapot: (jelölt index | -1 = nem látszik); a "nem látszik" a legutóbbi látott helyet viszi tovább
    score = [np.append(np.log(cands[0][:, 3] + 1e-6), TRACK_MISS)]
    last = [np.vstack([cands[0][:, :3], [np.nan, np.nan, np.nan]])]
    back = []
    for t in range(1, T):
        c = cands[t]
        n = len(c)
        prev_s, prev_last = score[-1], last[-1]
        s = np.full(n + 1, -np.inf)
        bk = np.zeros(n + 1, int)
        lp = np.full((n + 1, 3), np.nan)
        for j in range(n):
            for i in range(len(prev_s)):
                px, py, pd = prev_last[i]
                pen = 0.0
                if not np.isnan(px):
                    jump = np.hypot(c[j, 0] - px, c[j, 1] - py) / max(pd, c[j, 2], 4.0)
                    pen = TRACK_JUMP_W * max(0.0, jump - TRACK_FREE_JUMP) ** 2
                v = prev_s[i] + np.log(c[j, 3] + 1e-6) - pen
                if v > s[j]:
                    s[j], bk[j] = v, i
            lp[j] = c[j, :3]
        i = int(np.argmax(prev_s))
        s[n], bk[n], lp[n] = prev_s[i] + TRACK_MISS, i, prev_last[i]
        score.append(s)
        last.append(lp)
        back.append(bk)
    j = int(np.argmax(score[-1]))
    path = [j]
    for bk in reversed(back):
        j = int(bk[j])
        path.append(j)
    path.reverse()
    return [cands[t][j] if j < len(cands[t]) else None for t, j in enumerate(path)]


# Sebesség-alapú (másodrendű) követés: a labda a következő kockán nagyjából ott van,
# ahová a mostani sebessége viszi. A gyors, de EGYENLETES repülés így nem büntetett,
# a hirtelen irányváltás (átugrás egy álló téves jelöltre) igen.
# (A 40 félretett lövésen keresztvalidálva NEM jobb az elsőrendűnél: 64,9% vs 64,4% —
# a repülő labdánál a felismerő a szűk keresztmetszet, nem a pálya. Összevetésre megmarad.)
TRACK2_FREE_ACC = 4.0    # ennyi átmérő eltérés a jósolt helytől ingyenes (kézmozgás, becslési zaj)
TRACK2_ACC_W = 0.1       # az e fölötti eltérés négyzetes büntetése
TRACK2_MAX_GAP = 3       # ennyi "nem látszik" kocka után a sebesség elfelejtődik


def track2(cands: list[np.ndarray], miss: float | None = None, free_acc: float | None = None,
           acc_w: float | None = None, free_jump: float | None = None, jump_w: float | None = None,
           topk: int | None = None) -> list[np.ndarray | None]:
    """Másodrendű Viterbi: állapot = (előző kocka jelöltje, mostani jelölt), így a sebesség
    az állapotból adódik. Az átmenet ára a jósolt helytől (hely + sebesség × hézag) való
    eltérés; ha még nincs sebesség (pálya eleje, hosszú kiesés után), az elsőrendű
    ugrás-büntetés (TRACK_FREE_JUMP, TRACK_JUMP_W) érvényes. "Nem látszik" állapot: TRACK_MISS,
    alatta a pálya a legutóbbi helyről, a sebességgel tovább vetítve folytatódik."""
    miss = TRACK_MISS if miss is None else miss
    free_acc = TRACK2_FREE_ACC if free_acc is None else free_acc
    acc_w = TRACK2_ACC_W if acc_w is None else acc_w
    free_jump = TRACK_FREE_JUMP if free_jump is None else free_jump
    jump_w = TRACK_JUMP_W if jump_w is None else jump_w
    K = TRACK_TOPK if topk is None else topk
    cands = [c[:K] for c in cands]
    T = len(cands)
    if T == 0:
        return []
    # állapot (i, j) → (pontszám, utolsó látott hely (x, y, d) | None, sebesség (vx, vy) | None, hézag, vissza)
    c0 = cands[0]
    states = {(-1, j): (float(np.log(c0[j, 3] + 1e-6)), tuple(c0[j, :3]), None, 1, None) for j in range(len(c0))}
    states[(-1, -1)] = (miss, None, None, 1, None)
    hist = [states]
    for t in range(1, T):
        c = cands[t]
        new: dict = {}
        for (i, j), (s, pos, vel, gap, _) in states.items():
            # a következő jelölt k (vagy -1 = nem látszik)
            for k in range(-1, len(c)):
                if k < 0:
                    v = s + miss
                    nst = (v, pos, vel if gap < TRACK2_MAX_GAP else None, gap + 1, (i, j))
                else:
                    x, y, d, conf = c[k]
                    v = s + float(np.log(conf + 1e-6))
                    nvel = None
                    if pos is not None:
                        dd = max(pos[2], d, 4.0)
                        if vel is not None:
                            px, py = pos[0] + vel[0] * gap, pos[1] + vel[1] * gap
                            acc = np.hypot(x - px, y - py) / dd
                            v -= acc_w * max(0.0, acc - free_acc) ** 2
                        else:
                            jump = np.hypot(x - pos[0], y - pos[1]) / dd / gap
                            v -= jump_w * max(0.0, jump - free_jump) ** 2
                        nvel = ((x - pos[0]) / gap, (y - pos[1]) / gap)
                    nst = (v, (x, y, d), nvel, 1, (i, j))
                key = (j, k)
                if key not in new or nst[0] > new[key][0]:
                    new[key] = nst
        states = new
        hist.append(states)
    key = max(states, key=lambda q: states[q][0])
    path = []
    for t in range(T - 1, -1, -1):
        path.append(key[1])
        key = hist[t][key][4]
    path.reverse()
    return [cands[t][j] if j >= 0 else None for t, j in enumerate(path)]


# Összefésülés: a követett jelölt helyett a kocka legbiztosabbja, ha az ennyiszer biztosabb,
# vagy ha a követés "nem látszik"-ot mond (a gyors, elmosódott repülő labda gyakran kiesik
# a pályából, de a kockán önmagában felismerhető). Keresztvalidálva 64% → 69%; a 3–10 közti
# arányok mind hasonlóak.
TRACK_FUSE_RATIO = 3.0


def fuse(cands: list[np.ndarray], path: list[np.ndarray | None], ratio: float = TRACK_FUSE_RATIO) -> list[np.ndarray | None]:
    """A követés kimenete, ahol a kocka legjobb jelöltje sokkal biztosabb (vagy nincs követett), azzal felülírva."""
    out = []
    for c, q in zip(cands, path):
        top = c[0] if len(c) else None
        out.append(top if top is not None and (q is None or top[3] > ratio * q[3]) else q)
    return out


def evaluate_track(data: Path, ball_dir: Path, weights: str, imgsz: int = 640) -> dict:
    """A félretett lövéseken: kockánkénti legjobb jelölt vs. követés — eltalálja-e a jelölt labdát.

    imgsz: a felismerő bemeneti mérete a 640-es kivágásokon (a nagyobb felbontáson
    tanított modell 2× nagyítva, imgsz=1280-nal látja ugyanakkorának a labdát)."""
    from ultralytics import YOLO

    model = YOLO(weights)
    man = {c["id"]: c for c in json.loads((ball_dir / "ball_manifest.json").read_text())["klipek"]}
    pts = json.loads((data / "eval_points.json").read_text())
    by_clip = {}
    for p in pts:
        by_clip.setdefault(p["klip"], []).append(p)
    res = {"kockankent": {}, "kovetes": {}, "kovetes_masodrendu": {}, "osszefesult": {}}
    for cid, ps in sorted(by_clip.items()):
        m = man[cid]
        frames = read_video_frames(ball_dir / m["src"])
        e = m["elengedes"] - m["k0"]
        a, b = max(0, e + TRACK_WINDOW[0]), min(len(frames), e + TRACK_WINDOW[1] + 1)
        cands = detect_candidates(model, frames[a:b], imgsz=imgsz)
        paths = {"kovetes": track(cands), "kovetes_masodrendu": track2(cands)}
        paths["osszefesult"] = fuse(cands, paths["kovetes"])
        lab = {q["nev"]: q for q in json.loads((data / "labels_full.json").read_text())[cid]}
        for p in ps:
            f = lab[p["nev"]]["k"] - m["k0"] - a
            ok = 0 <= f < len(cands)
            c = cands[f] if ok else np.zeros((0, 4))
            hit_top = bool(len(c)) and np.hypot(c[0, 0] - p["x"], c[0, 1] - p["y"]) <= HIT_X * p["d"]
            res["kockankent"].setdefault(p["nev"], []).append(bool(hit_top))
            for mode, path in paths.items():
                q = path[f] if ok else None
                hit = q is not None and np.hypot(q[0] - p["x"], q[1] - p["y"]) <= HIT_X * p["d"]
                res[mode].setdefault(p["nev"], []).append(bool(hit))
    out = {}
    for mode, d in res.items():
        out[mode] = {nev: round(float(np.mean(v)), 3) for nev, v in d.items()}
        out[mode]["osszes"] = round(float(np.mean([x for v in d.values() for x in v])), 3)
    return out


def pseudo_labels(weights: str, ball_dir: Path, labels: Path, clips_manifest: Path,
                  min_conf: float = PSEUDO_MIN_CONF) -> dict[str, list[dict]]:
    """Gépi labdapontok a TANÍTÓ klipek összes kockáján (a félretett klipeket nem érinti):
    felismerő → követés → összefésülés, és ahol a választott jelölt legalább min_conf biztos."""
    from ultralytics import YOLO

    from .match_shots import replay_groups

    model = YOLO(weights)
    man = {c["id"]: c for c in json.loads((ball_dir / "ball_manifest.json").read_text())["klipek"]}
    groups = replay_groups(json.loads(clips_manifest.read_text())["clips"])
    L = {d["klip"]: d for d in json.loads(labels.read_text())["cimkek"]}
    ids = sorted(i for i in L if i in man)
    held = split_groups(ids, groups)
    out = {}
    for cid in ids:
        if cid in held:
            continue
        m = man[cid]
        frames = read_video_frames(ball_dir / m["src"])
        cands = detect_candidates(model, frames)
        path = fuse(cands, track(cands))
        H, W = frames[0].shape[:2]
        out[cid] = [{"nev": f"gepi{m['k0'] + f}", "k": m["k0"] + f, "cx": float(q[0] / W), "cy": float(q[1] / H),
                     "gepi": True, "biztos": round(float(q[3]), 3)}
                    for f, q in enumerate(path) if q is not None and q[3] >= min_conf]
    return out


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description="Labdafelismerő a közvetítésen")
    sub = ap.add_subparsers(dest="cmd", required=True)
    b = sub.add_parser("build")
    b.add_argument("--ball-dir", required=True, help="a make_ball_frames kimenete (ball_manifest.json + videók)")
    b.add_argument("--labels", required=True)
    b.add_argument("--clips", required=True, help="a lövés-klipek manifest.json-ja (csoportokhoz)")
    b.add_argument("--zoom", help="a match_shots zoom kimenete (a labda méretéhez)")
    b.add_argument("--out", required=True)
    b.add_argument("--pseudo", help="a pseudo lépés kimenete (gépi labdapontok a tanító klipekre)")
    ps = sub.add_parser("pseudo", help="öntanítás: gépi labdapontok a tanító klipek összes kockáján")
    ps.add_argument("--ball-dir", required=True)
    ps.add_argument("--labels", required=True)
    ps.add_argument("--clips", required=True)
    ps.add_argument("--weights", required=True)
    ps.add_argument("--min-conf", type=float, default=PSEUDO_MIN_CONF)
    ps.add_argument("--out", required=True, help="kimeneti JSON")
    t = sub.add_parser("train")
    t.add_argument("--data", required=True)
    t.add_argument("--weights", default="yolo11n.pt")
    t.add_argument("--epochs", type=int, default=40)
    t.add_argument("--device", default="cpu")
    t.add_argument("--project", default="runs_ball")
    t.add_argument("--name", default="labda")
    t.add_argument("--imgsz", type=int, default=TILE,
                   help=f"a tanító kivágások ({TILE} px) bemeneti mérete; 2×{TILE} = 2× nagyítás a kis labdához")
    t.add_argument("--batch", type=int, default=16)
    t.add_argument("--patience", type=int, default=15)
    e = sub.add_parser("evaluate")
    e.add_argument("--data", required=True)
    e.add_argument("--weights", required=True)
    e.add_argument("--ball-dir", help="megadva a követést is kiértékeli (a labdajelölő videóin)")
    e.add_argument("--imgsz", type=int, default=640,
                   help="a 640-es kivágások bemeneti mérete; a 2× nagyítással tanított modellnél 1280")
    e.add_argument("--out", help="az eredmény JSON-ja ide is (a képernyőn kívül)")
    args = ap.parse_args(argv)
    if args.cmd == "build":
        pseudo = json.loads(Path(args.pseudo).read_text()) if args.pseudo else None
        info = build(Path(args.ball_dir), Path(args.labels), Path(args.zoom) if args.zoom else None,
                     Path(args.clips), Path(args.out), pseudo)
        print(info)
    elif args.cmd == "pseudo":
        res = pseudo_labels(args.weights, Path(args.ball_dir), Path(args.labels), Path(args.clips), args.min_conf)
        Path(args.out).write_text(json.dumps(res))
        print(f"{sum(map(len, res.values()))} gépi pont, {len(res)} klip → {args.out}")
    elif args.cmd == "train":
        from ultralytics import YOLO

        YOLO(args.weights).train(data=str(Path(args.data) / "data.yaml"), imgsz=args.imgsz, epochs=args.epochs,
                                 device=args.device, batch=args.batch, project=args.project, name=args.name, exist_ok=True,
                                 mosaic=0.5, fliplr=0.5, degrees=0, scale=0.3, workers=2, patience=args.patience, verbose=False)
    else:
        res = evaluate(Path(args.data), args.weights, imgsz=args.imgsz)
        if args.ball_dir:
            res["kovetessel"] = evaluate_track(Path(args.data), Path(args.ball_dir), args.weights, imgsz=args.imgsz)
        if args.out:
            Path(args.out).write_text(json.dumps(res, ensure_ascii=False, indent=1))
        print(json.dumps(res, ensure_ascii=False, indent=1))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile hbd/hand_image.py
"""Lövőkéz a KÉPBŐL, testpontok nélkül.

A testpontokból a lövőkéz a közvetítésen nem jön ki megbízhatóan: még a kézzel
bejelölt labdával is csak 71%-os a "melyik csuklóhoz van közelebb" szabály
(a kicsi, elmosódott csuklót a pózbecslő gyakran rossz helyre teszi). Itt a
lövőt a "kézben" kockán (a labdajelölő címkéje) kivágjuk, és egy előtanított
képosztályozó (YOLO11-cls, ImageNet) jellemző-vektorán logisztikus regresszió
dönt: bal vagy jobb kéz.

Tükrözés: a vízszintesen tükrözött képen a bal kezes lövésből jobb kezes lesz,
így a tanító minták száma kétszeres, a jóslás pedig a kép és tükörképe
átlaga (p(kép) + 1 − p(tükör)) / 2 — a modell így nem tanulhat "oldal-előítéletet".

Kiértékelés: csoportosított keresztvalidáció (az ismétlés az élő párjával egy
csoportban).

    python -m hbd.hand_image --video meccs.mp4 --clips m1web/manifest.json --zoom m1_zoom \
        --labels cimkek/meccs1_lovesek_cimkek.json --ball cimkek/meccs1_labda_cimkek.json --out res/
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np

# A kivágás oldala a lövő dobozának hosszabbik oldalához képest.
CROP_X = 1.4
INPUT = 224
FOLDS = 5


def crops_for(video: str, clip: dict, zoom: dict, ks: list[int]) -> list[np.ndarray]:
    """A lövő négyzetes kivágásai a megadott klip-kockákon (az eredeti felvételből)."""
    import cv2

    from .match_shots import _orig_frames

    fps = clip["fps"]
    k0, st = int(zoom["k0"]), int(zoom["start"])
    out = []
    for k in ks:
        t = int(np.clip(k - k0 - st, 0, len(zoom["boxes"]) - 1))
        img = _orig_frames(video, clip["t_start"] + k / fps, 1)[0]
        h, w = img.shape[:2]
        x0, y0, x1, y1 = zoom["boxes"][t]
        cx, cy, side = (x0 + x1) / 2, (y0 + y1) / 2, CROP_X * max(x1 - x0, y1 - y0)
        a, b = int(max(0, cx - side / 2)), int(max(0, cy - side / 2))
        crop = img[b:int(min(h, cy + side / 2)), a:int(min(w, cx + side / 2))]
        out.append(cv2.resize(crop, (INPUT, INPUT), interpolation=cv2.INTER_AREA))
    return out


def embed(model, imgs: list[np.ndarray]) -> np.ndarray:
    return np.stack([e.cpu().numpy().ravel() for e in model.embed(imgs, imgsz=INPUT, verbose=False)])


def evaluate(X: np.ndarray, Xf: np.ndarray, y: np.ndarray, groups: np.ndarray, C: float = 0.1) -> dict:
    """X: kép-jellemzők, Xf: a tükörképeké, y: 1 = bal kéz. Csoportosított CV, tükör-átlagolt jóslás."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.model_selection import GroupKFold
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    p = np.zeros(len(y))
    for tr, te in GroupKFold(FOLDS).split(X, y, groups):
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000))
        clf.fit(np.concatenate([X[tr], Xf[tr]]), np.concatenate([y[tr], 1 - y[tr]]))
        p[te] = (clf.predict_proba(X[te])[:, 1] + 1 - clf.predict_proba(Xf[te])[:, 1]) / 2
    pred = (p > 0.5).astype(int)
    conf = np.abs(p - 0.5) * 2
    sure = conf >= np.median(conf)
    return {"klipek": int(len(y)), "pontossag": float((pred == y).mean()),
            "kiegyensulyozott": float(balanced_accuracy_score(y, pred)),
            "biztosabb_fele_pontossag": float((pred[sure] == y[sure]).mean()),
            "mindig_jobb": float((y == 0).mean())}


def main(argv=None) -> int:
    from ultralytics import YOLO

    from .match_shots import replay_groups

    ap = argparse.ArgumentParser(description="Lövőkéz a képből")
    ap.add_argument("--video", required=True)
    ap.add_argument("--clips", required=True)
    ap.add_argument("--zoom", required=True, help="a match_shots zoom kimenete (a lövő dobozai)")
    ap.add_argument("--labels", required=True)
    ap.add_argument("--ball", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--weights", default="yolo11m-cls.pt")
    args = ap.parse_args(argv)
    man = json.loads(Path(args.clips).read_text())["clips"]
    clips = {c["id"]: c for c in man}
    groups_of = replay_groups(man)
    hands = {d["klip"]: d.get("lovokez") for d in json.loads(Path(args.labels).read_text())["cimkek"]}
    balls = {d["klip"]: {p["nev"]: p for p in d["pontok"]} for d in json.loads(Path(args.ball).read_text())["cimkek"]}
    model = YOLO(args.weights)
    rows = []
    for cid, pts in sorted(balls.items()):
        if hands.get(cid) not in ("jobb", "bal") or cid not in clips:
            continue
        zf = Path(args.zoom) / f"{cid}.npz"
        if not zf.exists():
            continue
        z = np.load(zf)
        if bool(z["lost"]) or pts["kezben"].get("nincs"):
            continue
        im = crops_for(args.video, clips[cid], z, [pts["kezben"]["k"]])[0]
        rows.append((cid, im, int(hands[cid] == "bal")))
    imgs = [r[1] for r in rows]
    X = embed(model, imgs)
    Xf = embed(model, [im[:, ::-1].copy() for im in imgs])
    y = np.array([r[2] for r in rows])
    groups = np.array([groups_of[r[0]] for r in rows])
    res = {c: evaluate(X, Xf, y, groups, c) for c in (0.01, 0.1, 1.0)}
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    (out / "hand_image.json").write_text(json.dumps(res, ensure_ascii=False, indent=1))
    np.savez_compressed(out / "hand_image_features.npz", X=X, Xf=Xf, y=y, groups=groups,
                        ids=np.array([r[0] for r in rows]))
    print(json.dumps(res, ensure_ascii=False, indent=1))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile hbd/shooter_ball.py
"""Ki lő? — a lövő a LABDÁBÓL, kézi kattintás nélkül.

A testpontokból tanult lövő-választó (match_shots.evaluate_picker) a klipek
egy részén el sem jut a lövőig: a teljes képen futó pózbecslő a lövőt sokszor
nem találja meg külön emberként (76/184 klip), és ahol igen, ott is csak 55%.
Itt a labdafelismerő (hbd.ball) dönt: az elengedés körül a labda a lövő
kezében van / épp elválik tőle, tehát a lövő az, akinek a CSUKLÓJA a labdához
legközelebb van.

Menet klipenként (a meccs-lövésjelölő elengedés-kockája körül, ±WIN kocka):
  1. a labdafelismerő a teljes 1920×1080 kockán, TILE px-es átfedő csempéken,
     natív léptékben (a labdajelölő videói is natív 640 px-es kivágások),
  2. minden emberre pontszám: max_labda megbízhatóság × exp(−(d/SIGMA)²), ahol
     d a legközelebbi csukló távolsága a labdától, az ember magasságában mérve,
  3. a lövő a legnagyobb pontszámú ember (ha van PICK_MIN fölött); ha senki,
     a lövő HELYE a legbiztosabb labda (ez a ráközelítő követés kiindulópontja
     lehet — a kattintás helyett).

Célzóna (lent / fent) a labda repülési irányából: a lövő köré vágott
kivágáson a labdát kockáról kockára követjük (hbd.ball.track), és az elválás
→ repülés elmozdulásából becslünk — ugyanazokkal a jellemzőkkel, mint a kézi
labdapontokkal mért 76%.

    python -m hbd.shooter_ball --video meccs.mp4 --clips m1web/manifest.json \
        --poses m1_clip_poses --zoom m1_zoom2 --labels cimkek/meccs1_lovesek_cimkek.json \
        --ball-labels cimkek/meccs1_labda_cimkek.json --ball-manifest m1ball2/ball_manifest.json \
        --weights runs_ball/labda2/weights/best.pt --cache sb_cache --out res/
"""

from __future__ import annotations

import argparse
import json
from pathlib import Path

import numpy as np

# Csempe natív képpontban (mint a labdajelölő 640 px-es kivágásai) és az átfedés aránya.
TILE = 640
TILE_OVERLAP = 1 / 3
# Az elengedés körüli kockák (a jelölt elengedéstől, kockában).
WIN = (-2, 2)
# A labdajelöltek alsó megbízhatósága, és az összevonás sugara (px) a csempék közt.
CONF = 0.05
NMS_PX = 12.0
# Csukló–labda távolság léptéke az ember magasságában (0,25 × ~180 px ≈ 45 px).
SIGMA = 0.25
WRIST_CONF = 0.2
# Kockák összevonása emberenként: "max" (a legjobb kocka) vagy "sum" (a lövő keze több kockán át a labdánál).
ACCUM = "sum"
# Ennél kisebb pontszámmal nem választunk embert (akkor a labda helye a lövő helye).
PICK_MIN = 0.02
# Átfedés, amivel a többi kocka embereit az elengedés kockájának embereihez kötjük.
LINK_IOU = 0.3
# A cél-lövő (a követett lövő doboza az elengedésnél) — ugyanaz, mint match_shots.build.
TARGET_IOU = 0.5
# A lövő HELYE akkor talált, ha a követett lövő dobozától legfeljebb LOC_PAD × doboz-magasságra van
# (a ráközelítő követés a pont körüli kivágáson keres embert — ennyi elég az indításhoz).
LOC_PAD = 0.25
# Repülés-követés: kivágás a lövő köré, az elengedéstől ennyi kockán.
FLIGHT_WIN = (-2, 10)
FLIGHT_MIN_DK = 2

# Álló hamis jelöltek (reklám, vonal, néző): ennyi px-en belül ugyanott legalább STATIC_MIN másik
# kockában is → nem a labda (a labda az elengedés körül kockánként több tíz px-t mozdul).
STATIC_PX = 4.0
STATIC_MIN = 2
# A labda mérete az emberhez: átmérő ≈ BALL_REL × magasság (mint a tanításnál); log-eltérés szórása.
SIZE_SIGMA = 0.5

L_WR, R_WR = 9, 10


# ------------------------------------------------------------ labda a teljes kockán
def tile_origins(W: int, H: int, side: int = TILE, overlap: float = TILE_OVERLAP) -> list[tuple[int, int]]:
    """Átfedő csempék bal felső sarkai, amelyek a teljes képet lefedik."""
    def axis(n):
        if n <= side:
            return [0]
        step = side * (1 - overlap)
        k = int(np.ceil((n - side) / step)) + 1
        return [int(round(v)) for v in np.linspace(0, n - side, k)]
    return [(x, y) for y in axis(H) for x in axis(W)]


def merge_points(c: np.ndarray, radius: float = NMS_PX) -> np.ndarray:
    """Közeli jelöltek összevonása (a csempék átfedéséből): a biztosabb marad. c: (N, 4) x, y, d, conf."""
    if len(c) == 0:
        return c.reshape(0, 4)
    c = c[np.argsort(-c[:, 3])]
    keep = []
    for row in c:
        if all(np.hypot(row[0] - k[0], row[1] - k[1]) > radius for k in keep):
            keep.append(row)
    return np.array(keep, np.float32)


def detect_full(model, frames: list[np.ndarray], conf: float = CONF) -> list[np.ndarray]:
    """Kockánként a labdajelöltek a teljes képen: (N, 4) — x, y, átmérő, megbízhatóság."""
    out = []
    for img in frames:
        H, W = img.shape[:2]
        orig = tile_origins(W, H)
        tiles = [img[y:y + TILE, x:x + TILE] for x, y in orig]
        rows = []
        for (x0, y0), r in zip(orig, model.predict(tiles, imgsz=TILE, conf=conf, verbose=False)):
            b = r.boxes.xyxy.cpu().numpy()
            c = r.boxes.conf.cpu().numpy()
            for bb, cc in zip(b, c):
                rows.append([x0 + (bb[0] + bb[2]) / 2, y0 + (bb[1] + bb[3]) / 2,
                             (bb[2] - bb[0] + bb[3] - bb[1]) / 2, cc])
        out.append(merge_points(np.array(rows, np.float32).reshape(-1, 4)))
    return out


def suppress_static(balls: list[np.ndarray], px: float = STATIC_PX, min_other: int = STATIC_MIN) -> list[np.ndarray]:
    """Az ablak több kockájában ugyanott álló jelöltek elhagyása."""
    out = []
    for i, b in enumerate(balls):
        keep = []
        for row in b:
            n = sum(1 for j, o in enumerate(balls) if j != i and len(o)
                    and np.min(np.hypot(o[:, 0] - row[0], o[:, 1] - row[1])) <= px)
            keep.append(n < min_other)
        out.append(b[np.array(keep, bool)] if len(b) else b)
    return out


# ------------------------------------------------------------ ember ↔ labda
def person_scores(boxes: np.ndarray, kps: np.ndarray, balls: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Emberenként: a legjobb labda pontszáma (megbízhatóság × csukló-közelség) és a labda indexe (−1: nincs)."""
    n = len(boxes)
    s, bi = np.zeros(n), np.full(n, -1)
    if n == 0 or len(balls) == 0:
        return s, bi
    for j in range(n):
        h = max(float(boxes[j, 3] - boxes[j, 1]), 1.0)
        wr = [kps[j, w, :2] for w in (L_WR, R_WR) if kps[j, w, 2] > WRIST_CONF]
        if not wr:
            continue
        from .ball import BALL_REL

        d = np.min([np.hypot(balls[:, 0] - p[0], balls[:, 1] - p[1]) for p in wr], 0) / h
        size = np.exp(-np.log(np.maximum(balls[:, 2], 1.0) / (BALL_REL * h)) ** 2 / (2 * SIZE_SIGMA ** 2))
        v = balls[:, 3] * np.exp(-(d / SIGMA) ** 2) * size
        k = int(np.argmax(v))
        s[j], bi[j] = v[k], k
    return s, bi


def iou(a: np.ndarray, b: np.ndarray) -> float:
    x0, y0, x1, y1 = max(a[0], b[0]), max(a[1], b[1]), min(a[2], b[2]), min(a[3], b[3])
    inter = max(0.0, x1 - x0) * max(0.0, y1 - y0)
    ua = (a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter
    return float(inter / ua) if ua > 0 else 0.0


def pick_shooter(people: list[tuple[np.ndarray, np.ndarray]], balls: list[np.ndarray], r: int) -> dict:
    """A lövő a kockák (people[i], balls[i]) alapján; r = az elengedés kockája a listában.

    Visszaad: {"index": az elengedés-kocka embere vagy None, "pont": (x, y) a lövő helye vagy None,
    "pontszam": a döntés pontszáma, "labda": a döntő labda (x, y, d, conf) vagy None}."""
    R = people[r][0]
    acc = np.zeros(len(R))
    best_ball = [None] * len(R)
    best_s: dict[int, float] = {}
    for (bx, kp), bl in zip(people, balls):
        s, bi = person_scores(bx, kp, bl)
        for j in range(len(bx)):
            if s[j] <= 0:
                continue
            # a kocka emberét az elengedés-kocka emberéhez kötjük (átfedés)
            ov = [iou(bx[j], R[i]) for i in range(len(R))]
            if not ov or max(ov) < LINK_IOU:
                continue
            i = int(np.argmax(ov))
            if s[j] > best_s.get(i, 0.0):
                best_ball[i], best_s[i] = bl[bi[j]], s[j]
            acc[i] = acc[i] + s[j] if ACCUM == "sum" else max(acc[i], s[j])
    if len(R) and acc.max() >= PICK_MIN:
        i = int(np.argmax(acc))
        b = R[i]
        return {"index": i, "pont": ((b[0] + b[2]) / 2, (b[1] + b[3]) / 2), "pontszam": float(acc[i]),
                "labda": best_ball[i], "doboz": b}
    allb = [b for b in balls if len(b)]
    if allb:
        cand = np.concatenate(allb)
        top = cand[int(np.argmax(cand[:, 3]))]
        return {"index": None, "pont": (float(top[0]), float(top[1])), "pontszam": 0.0, "labda": top, "doboz": None}
    return {"index": None, "pont": None, "pontszam": 0.0, "labda": None, "doboz": None}


def inside(pt, box, pad: float = LOC_PAD) -> bool:
    """A pont legfeljebb pad × magasságra van a dobozától (benne: 0)."""
    if pt is None or box is None:
        return False
    dx = max(box[0] - pt[0], 0.0, pt[0] - box[2])
    dy = max(box[1] - pt[1], 0.0, pt[1] - box[3])
    return float(np.hypot(dx, dy)) <= pad * float(box[3] - box[1])


# ------------------------------------------------------------ célzóna a repülésből
def flight_features(p_sep: tuple[float, float], k_sep: int, p_fly: tuple[float, float], k_fly: int,
                    box: np.ndarray, W: int = 1920, H: int = 1080) -> list[float]:
    """Ugyanazok a jellemzők, mint a kézi labdapontokkal mért zóna-becslésnél (pixelben megadott pontok)."""
    hh = max(float(box[3] - box[1]), 1.0)
    dk = max(1, k_fly - k_sep)
    dx = (p_fly[0] - p_sep[0]) / hh / dk
    dy = (p_fly[1] - p_sep[1]) / hh / dk
    return [dx, dy, float(np.arctan2(dy, dx)), p_sep[0] / W, p_sep[1] / H, (p_sep[1] - box[1]) / hh]


def flight_from_path(path: list, r: int) -> tuple | None:
    """A követett labdapályából (hbd.ball.track) az elválás (az elengedéshez legközelebbi látott pont,
    r-től) és a repülés (a legtávolabbi látott pont legalább FLIGHT_MIN_DK kockával később)."""
    seen = [(t, q) for t, q in enumerate(path) if q is not None and t >= r - 1]
    if len(seen) < 2:
        return None
    t0, q0 = seen[0]
    later = [(t, q) for t, q in seen if t - t0 >= FLIGHT_MIN_DK]
    if not later:
        return None
    t1, q1 = later[-1]
    return (float(q0[0]), float(q0[1])), t0, (float(q1[0]), float(q1[1])), t1


# ------------------------------------------------------------ összefűzés
def read_frames(video: str, t0: float, n: int, size: tuple[int, int] = (1920, 1080)) -> list[np.ndarray]:
    """n kocka t0-tól, kockára pontosan.

    A match_shots._orig_frames az ffmpeg alap kocka-időzítésével olvas, ami a pontos
    ugrásnál az első kockát néha megduplázza ([F, F, F+1, …]): ott a lista i. eleme
    (i ≥ 1) valójában az F+i−1. kocka. A `-fps_mode passthrough` ezt megszünteti."""
    import subprocess

    from .video import ffmpeg_exe

    w, h = size
    raw = subprocess.run([ffmpeg_exe(), "-v", "error", "-ss", f"{max(0.0, t0):.3f}", "-i", video,
                          "-fps_mode", "passthrough", "-frames:v", str(n), "-f", "rawvideo", "-pix_fmt", "bgr24", "-"],
                         stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=True).stdout
    k = len(raw) // (w * h * 3)
    return list(np.frombuffer(raw[:k * w * h * 3], np.uint8).reshape(k, h, w, 3))


def _clip_frames(video: str, clip: dict, k_from: int, n: int) -> list[np.ndarray]:
    return read_frames(video, clip["t_start"] + k_from / float(clip["fps"]), n)


def pose_lag(p: dict) -> int:
    """1, ha a klip-póz a duplázott első kockával készült (az i. póz-kocka a k0+i−1. klip-kocka), különben 0."""
    if len(p["n"]) < 2 or p["n"][0] == 0 or p["n"][0] != p["n"][1]:
        return 0
    a, b = p["offsets"][0], p["offsets"][1]
    return int(np.allclose(p["boxes"][a:b], p["boxes"][b:b + (b - a)]))


def heldout_ids(ball_labels: Path, ball_manifest: Path, groups: dict[str, str]) -> set[str]:
    """A labdafelismerő tanításából kimaradt klipek (ugyanaz a felosztás, mint a hbd.ball build-ben)."""
    from .ball import split_groups

    man = {c["id"] for c in json.loads(ball_manifest.read_text())["klipek"]}
    L = {d["klip"] for d in json.loads(ball_labels.read_text())["cimkek"]}
    return split_groups(sorted(i for i in L if i in man), groups)


def run(args) -> dict:
    import torch
    from ultralytics import YOLO

    from .ball import detect_candidates, track
    from .match_shots import candidate_features, frame_people, load_clip_pose, replay_groups, split_zone

    torch.set_num_threads(args.threads)
    model = YOLO(args.weights)
    clips = json.loads(Path(args.clips).read_text())["clips"]
    groups = replay_groups(clips)
    held = heldout_ids(Path(args.ball_labels), Path(args.ball_manifest), groups)
    labs = {d["klip"]: d for d in json.loads(Path(args.labels).read_text())["cimkek"]
            if not d.get("rossz") and d.get("lovo") and d.get("elengedes") is not None}
    bl = {d["klip"]: {p["nev"]: p for p in d["pontok"]} for d in json.loads(Path(args.ball_labels).read_text())["cimkek"]}
    cache = Path(args.cache)
    cache.mkdir(parents=True, exist_ok=True)
    rows = []
    for c in clips:
        cid = c["id"]
        lab = labs.get(cid)
        pf, zf = Path(args.poses) / f"{cid}.npz", Path(args.zoom) / f"{cid}.npz"
        if lab is None or not pf.exists() or not zf.exists() or (args.only and cid not in args.only):
            continue
        p = load_clip_pose(pf)
        z = np.load(zf)
        # a póz-index egy klip-kockához: f − k0 + lag (lásd read_frames)
        k0 = int(p["k0"]) - pose_lag(p)
        rel = int(lab["elengedes"])
        fr = [f for f in range(rel + WIN[0], rel + WIN[1] + 1) if 0 <= f - k0 < len(p["n"])]
        if rel not in fr:
            continue
        cf = cache / f"{cid}_teljes.npz"
        if cf.exists():
            zc = np.load(cf)
            balls = [zc["c"][zc["f"] == i] for i in range(len(fr))]
        else:
            balls = detect_full(model, _clip_frames(args.video, c, fr[0], len(fr)))
            np.savez_compressed(cf, c=np.concatenate(balls) if balls else np.zeros((0, 4)),
                                f=np.concatenate([np.full(len(b), i) for i, b in enumerate(balls)]))
        people = [frame_people(p, f - k0) for f in fr]
        r = fr.index(rel)
        balls = suppress_static(balls)
        res = pick_shooter(people, balls, r)
        # a cél: a követett lövő doboza az elengedésnél (mint a match_shots.build-ben)
        target, sbox = None, None
        if not bool(z["lost"]):
            t = int(np.clip(rel - (int(z["k0"]) - (int(p["k0"]) - k0)) - int(z["start"]), 0, len(z["boxes"]) - 1))
            sbox = z["boxes"][t]
            R = people[r][0]
            if len(R):
                ov = [iou(sbox, b) for b in R]
                target = int(np.argmax(ov)) if max(ov) > TARGET_IOU else None
        X, _ = candidate_features(p, rel - k0)
        row = {"klip": cid, "csoport": groups[cid], "felretett": cid in held, "cel": target,
               "valasztott": res["index"], "talalt": res["index"] is not None and res["index"] == target,
               "hely_talalt": inside(res["pont"], sbox), "pontszam": res["pontszam"],
               "emberek": int(len(people[r][0])), "X": X, "R": people[r][0], "sbox": sbox}
        # emberenkénti labda-jellemző a kombinált választóhoz (elengedés-kocka emberei)
        R = people[r][0]
        ballf = np.zeros((len(R), 4))
        for (bx, kp), b in zip(people, balls):
            s, bi = person_scores(bx, kp, b)
            for j in range(len(bx)):
                ov = [iou(bx[j], R[i]) for i in range(len(R))]
                if ov and max(ov) >= LINK_IOU:
                    i = int(np.argmax(ov))
                    ballf[i, 0] = max(ballf[i, 0], s[j])
                    ballf[i, 2] += s[j]
        if len(R):
            ballf[:, 1] = ballf[:, 0] / max(ballf[:, 0].max(), 1e-6)
            ballf[:, 3] = ballf[:, 2] / max(ballf[:, 2].max(), 1e-6)
        row["Xb"] = np.concatenate([X, ballf], 1) if len(X) else X
        # repülés: a lövő köré vágott kivágáson követett labda
        vz, hz = split_zone(lab.get("zona"))
        row["zona_v"] = vz
        if res["pont"] is not None and not args.no_flight:
            box = res["doboz"] if res["doboz"] is not None else (
                np.array([res["pont"][0] - 50, res["pont"][1] - 100, res["pont"][0] + 50, res["pont"][1] + 100]))
            cx, cy = res["labda"][:2] if res["labda"] is not None else res["pont"]
            x0 = int(np.clip(cx - TILE / 2, 0, 1920 - TILE))
            y0 = int(np.clip(cy - TILE / 2, 0, 1080 - TILE))
            ff = cache / f"{cid}_repules_{x0}_{y0}.npz"
            a = rel + FLIGHT_WIN[0]
            n = FLIGHT_WIN[1] - FLIGHT_WIN[0] + 1
            if ff.exists():
                zc = np.load(ff)
                cands = [zc["c"][zc["f"] == i] for i in range(int(zc["n"]))]
            else:
                frames = [im[y0:y0 + TILE, x0:x0 + TILE] for im in _clip_frames(args.video, c, a, n)]
                cands = detect_candidates(model, frames)
                np.savez_compressed(ff, c=np.concatenate(cands) if cands else np.zeros((0, 4)),
                                    f=np.concatenate([np.full(len(q), i) for i, q in enumerate(cands)]),
                                    n=np.int32(len(cands)))
            fl = flight_from_path(track(cands), rel - a)
            if fl is not None:
                (sx, sy), ts, (fx, fy), tf = fl
                row["repules"] = flight_features((x0 + sx, y0 + sy), ts, (x0 + fx, y0 + fy), tf, box)
        # kézi labdapontokból ugyanez (a zóna-modell tanításához)
        b = bl.get(cid, {})
        if sbox is not None and "elvalik" in b and "repul" in b and not b["elvalik"].get("nincs") \
                and not b["repul"].get("nincs"):
            row["repules_kezi"] = flight_features((b["elvalik"]["x"] * 1920, b["elvalik"]["y"] * 1080),
                                                  b["elvalik"]["k"], (b["repul"]["x"] * 1920, b["repul"]["y"] * 1080),
                                                  b["repul"]["k"], sbox)
        rows.append(row)
        print(f"  {cid}: {'+' if row['talalt'] else '-'}{'+' if row['hely_talalt'] else '-'} "
              f"({row['emberek']} ember)", flush=True)
    return {"rows": rows}


def summarize(rows: list[dict]) -> dict:
    """Lövő-választás és lövő-hely: a félretett klipeken és az összesen (ez utóbbi optimista)."""
    combined = combined_picker(rows)
    out = {}
    for name, sel in (("felretett", [r for r in rows if r["felretett"]]), ("osszes", rows)):
        n = len(sel)
        with_t = [r for r in sel if r["cel"] is not None]
        out[name] = {
            "klipek": n, "lovo_a_jeloltek_kozt": len(with_t),
            "labda_valaszto_ha_jelolt": float(np.mean([r["talalt"] for r in with_t])) if with_t else None,
            "labda_valaszto_az_osszesbol": float(np.mean([r["talalt"] for r in sel])) if sel else None,
            "lovo_helye_az_osszesbol": float(np.mean([r["hely_talalt"] for r in sel])) if sel else None,
            "kombinalt_ha_jelolt": float(np.mean([_hit(r, combined) for r in with_t])) if with_t else None,
            "testpont_valaszto_ha_jelolt": float(np.mean([_hit(r, _BASE_HITS) for r in with_t])) if with_t else None,
            "testpont_hely": float(np.mean([_loc(r, _BASE_HITS) for r in sel])) if sel else None,
            "kombinalt_hely": float(np.mean([_loc(r, combined) for r in sel])) if sel else None,
        }
    return out


_BASE_HITS: dict = {}


def _cv_hits(rows: list[dict], key: str, seed: int = 0) -> dict[str, int]:
    """Csoportosított CV: klipenként a választott ember indexe (az elengedés-kocka emberei közt)."""
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.model_selection import GroupKFold

    # tanítás a lövőt tartalmazó klipeken; jóslás MINDEN klipre, ahol van ember (a hely-méréshez)
    data = [r for r in rows if len(r[key])]
    groups = np.array([r["csoport"] for r in data])
    hits = {}
    if len(set(groups)) < 2:
        return hits
    for tr, te in GroupKFold(min(5, len(set(groups)))).split(np.zeros(len(data)), groups=groups):
        trc = [i for i in tr if data[i]["cel"] is not None]
        X = np.concatenate([data[i][key] for i in trc])
        y = np.concatenate([np.arange(len(data[i][key])) == data[i]["cel"] for i in trc]).astype(int)
        clf = HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, max_leaf_nodes=15,
                                             class_weight="balanced", random_state=seed).fit(X, y)
        for i in te:
            j = int(np.argmax(clf.predict_proba(data[i][key])[:, 1]))
            hits[data[i]["klip"]] = j
    return hits


def combined_picker(rows: list[dict]) -> dict[str, int]:
    """A testpont-jellemzős választó + labda-jellemzők (csoportosított CV): klip → választott index."""
    _BASE_HITS.clear()
    _BASE_HITS.update(_cv_hits(rows, "X"))
    return _cv_hits(rows, "Xb")


def _hit(r: dict, picks: dict) -> bool:
    return r["cel"] is not None and picks.get(r["klip"]) == r["cel"]


def _loc(r: dict, picks: dict) -> bool:
    j = picks.get(r["klip"])
    if j is None:
        return False
    b = r["R"][j]
    return inside(((b[0] + b[2]) / 2, (b[1] + b[3]) / 2), r["sbox"])


def zone_eval(rows: list[dict]) -> dict:
    """Lent vs fent (a közép kimarad) a repülési irányból.

    A: tanítás a kézi labdapontokon (a nem félretett klipeken), teszt a félretett klipek
       AUTOMATIKUS repülésén — a valódi, kattintás nélküli helyzet.
    B: csoportosított CV az automatikus repülésen, minden klipen (optimista: a labdafelismerő
       a klipek 80%-án tanult)."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import balanced_accuracy_score
    from sklearn.model_selection import GroupKFold
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler

    def clf():
        return make_pipeline(StandardScaler(), LogisticRegression(C=0.5, max_iter=3000, class_weight="balanced"))

    lf = [r for r in rows if r.get("zona_v") in (0, 2)]
    out = {}
    tr = [r for r in lf if not r["felretett"] and "repules_kezi" in r]
    te = [r for r in lf if r["felretett"]]
    te_ok = [r for r in te if "repules" in r]
    if tr and te_ok:
        m = clf().fit(np.array([r["repules_kezi"] for r in tr]), np.array([r["zona_v"] == 2 for r in tr], int))
        y = np.array([r["zona_v"] == 2 for r in te_ok], int)
        pred = m.predict(np.array([r["repules"] for r in te_ok]))
        out["felretett_automatikus"] = {"lent_fent_klipek": len(te), "repules_megvan": len(te_ok),
                                        "kiegyensulyozott": float(balanced_accuracy_score(y, pred)),
                                        "pontossag": float((pred == y).mean())}
        te_k = [r for r in te if "repules_kezi" in r]
        if te_k:
            yk = np.array([r["zona_v"] == 2 for r in te_k], int)
            pk = m.predict(np.array([r["repules_kezi"] for r in te_k]))
            out["felretett_kezi_labdaval"] = {"klipek": len(te_k),
                                              "kiegyensulyozott": float(balanced_accuracy_score(yk, pk))}
    au = [r for r in lf if "repules" in r]
    if len(au) >= 20:
        X = np.array([r["repules"] for r in au])
        y = np.array([r["zona_v"] == 2 for r in au], int)
        g = np.array([r["csoport"] for r in au])
        pred = np.zeros_like(y)
        for a, b in GroupKFold(5).split(X, y, g):
            pred[b] = clf().fit(X[a], y[a]).predict(X[b])
        out["osszes_automatikus_cv"] = {"lent_fent_klipek": len(lf), "repules_megvan": len(au),
                                        "kiegyensulyozott": float(balanced_accuracy_score(y, pred))}
    return out


def write_report(summ: dict, zone: dict, out: Path) -> Path:
    pct = lambda v: "–" if v is None else f"{100 * v:.0f}%"  # noqa: E731
    h, a = summ["felretett"], summ["osszes"]
    md = ["# Ki lő? — a lövő a labdából (1. meccs)", "",
          "A labdafelismerő (v2) a teljes 1920×1080 kockán, az elengedés körül ±2 kockán; a lövő az, akinek",
          "a csuklója a legbiztosabb labdához legközelebb van. Kézi labda-kattintás NINCS; az elengedés",
          "kockája még a kézi jelölésé.", "",
          "**Félretett** = a labdafelismerő tanításából kimaradt lövések (ugyanaz a 20%, mint a labda-",
          "kiértékelésnél) — ez a becsületes szám. Az **összes** klipen a labdafelismerő a klipek 80%-án",
          "tanult, ezért az optimista.", "",
          "| | félretett | összes (optimista) |", "|---|---|---|",
          f"| klipek | {h['klipek']} | {a['klipek']} |",
          f"| a lövő a pózbecslő emberei közt van | {h['lovo_a_jeloltek_kozt']} | {a['lovo_a_jeloltek_kozt']} |",
          f"| testpont-választó (eddigi), ha a lövő jelölt | {pct(h['testpont_valaszto_ha_jelolt'])} | "
          f"{pct(a['testpont_valaszto_ha_jelolt'])} |",
          f"| **labda-választó**, ha a lövő jelölt | **{pct(h['labda_valaszto_ha_jelolt'])}** | "
          f"{pct(a['labda_valaszto_ha_jelolt'])} |",
          f"| testpont + labda (tanult), ha a lövő jelölt | {pct(h['kombinalt_ha_jelolt'])} | "
          f"{pct(a['kombinalt_ha_jelolt'])} |",
          f"| labda-választó, az összes klipből | {pct(h['labda_valaszto_az_osszesbol'])} | "
          f"{pct(a['labda_valaszto_az_osszesbol'])} |",
          f"| a lövő HELYE — testpont-választó, az összesből | {pct(h['testpont_hely'])} | {pct(a['testpont_hely'])} |",
          f"| a lövő HELYE — testpont + labda, az összesből | {pct(h['kombinalt_hely'])} | {pct(a['kombinalt_hely'])} |",
          f"| **a lövő HELYE — labda-választó** (ember vagy labda), az összesből | "
          f"**{pct(h['lovo_helye_az_osszesbol'])}** | {pct(a['lovo_helye_az_osszesbol'])} |", "",
          "A „lövő helye” azt méri, hogy a választott pont (a választott ember közepe, vagy ha embert nem",
          "talál, a labda) a követett lövő dobozába esik-e. Ez kell a ráközelítő követés indításához — ott",
          "eddig a kézi kattintás állt.", "",
          "## Célzóna lent / fent — teljesen automatikusan", "",
          "A lövő köré vágott képen a labda követve (hbd.ball.track), az elválás → repülés iránya.", ""]
    if "felretett_automatikus" in zone:
        z = zone["felretett_automatikus"]
        md += [f"- Félretett lövések (lent/fent: {z['lent_fent_klipek']}, ebből követhető repülés: "
               f"{z['repules_megvan']}), kézi pontokon tanított modell, automatikus repüléssel: "
               f"**{pct(z['kiegyensulyozott'])} kiegyensúlyozott** (véletlen: 50%)."]
    if "felretett_kezi_labdaval" in zone:
        z = zone["felretett_kezi_labdaval"]
        md += [f"- Ugyanez a kézi labdapontokkal (felső korlát, {z['klipek']} klip): {pct(z['kiegyensulyozott'])}."]
    if "osszes_automatikus_cv" in zone:
        z = zone["osszes_automatikus_cv"]
        md += [f"- Minden klipen, automatikus repüléssel, csoportosított CV ({z['repules_megvan']} klip; optimista): "
               f"{pct(z['kiegyensulyozott'])}."]
    md += ["", "Megjegyzés: a félretett halmaz kicsi (≈40 lövés, ebből a lent/fent zónás még kevesebb) —",
           "a különbségek pár lövésen múlnak. A végleges számot a második meccs adja."]
    p = out / "lovo_labdabol.md"
    p.write_text("\n".join(md) + "\n")
    return p


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description="A lövő a labdából (kattintás nélkül)")
    ap.add_argument("--video", required=True)
    ap.add_argument("--clips", required=True)
    ap.add_argument("--poses", required=True)
    ap.add_argument("--zoom", required=True)
    ap.add_argument("--labels", required=True)
    ap.add_argument("--ball-labels", required=True)
    ap.add_argument("--ball-manifest", required=True)
    ap.add_argument("--weights", required=True)
    ap.add_argument("--cache", required=True)
    ap.add_argument("--out", required=True)
    ap.add_argument("--threads", type=int, default=2)
    ap.add_argument("--only", nargs="*", help="csak ezek a klipek (próbához)")
    ap.add_argument("--no-flight", action="store_true", help="a repülés-követés nélkül (gyors hangoláshoz)")
    args = ap.parse_args(argv)
    rows = run(args)["rows"]
    summ = summarize(rows)
    zone = zone_eval(rows)
    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    (out / "lovo_labdabol.json").write_text(json.dumps({"lovo": summ, "zona": zone}, ensure_ascii=False, indent=1))
    print(write_report(summ, zone, out).read_text())
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


In [ ]:
%%writefile hbd/scoreboard.py
"""Az állás KIOLVASÁSA az eredményjelzőből → a meccs teljes gól-idővonala.

A `broadcast` csak azt látja, hogy az állás-kivágás megváltozott-e; ez a modul
ki is olvassa a két számot. A közvetítés grafikájának betűtípusa fix, így a
számjegy-felismerés egyszerű sablon-illesztés:

  1. stabil állapotok: a grafika látszik és legalább STABLE_S-ig nem változik
     (az animáció, beúszás így kiesik); állapotonként a medián kép;
  2. a két szám a középvonalnál kettévágva; jelek = összefüggő világos foltok,
     egységes GLYPH méretre hozva;
  3. sablonok: a jelek k-közép csoportjai, amiket EGYSZER, szemmel címkézünk
     (közvetítőnként; `cluster` parancs → kép → `--labels`);
  4. idővonal: csak a növekvő, lépésenként legfeljebb +2 gólos olvasatok
     (a kettős lépés: két gól egy ismétlés alatt, a grafika nem látszott).

Ellenőrzés emberi jelölés nélkül: a végső olvasatnak egyeznie kell a hivatalos
végeredménnyel. A 2. meccsen (ARD-grafika): 35:33 olvasva, 68/68 gól.

    python -m hbd.scoreboard crops  --video m.mp4 --box 0.462 0.062 0.545 0.110 --stride 10 --out sb.npy
    python -m hbd.scoreboard cluster --crops sb.npy --broadcast m_broadcast.npz --out sablon/   # → klaszterek.png
    python -m hbd.scoreboard read --crops sb.npy --broadcast m_broadcast.npz \\
        --templates sablon/kozeppontok.npy --labels 2,1,9,4,3,7,5,6,8,0,1,2,x,3 --out allas.json
"""

from __future__ import annotations

import argparse
import json
import subprocess
from pathlib import Path

import numpy as np

# Egy állapot legalább ennyi ideig látszó, változatlan grafika (s).
STABLE_S = 2.0
# Szomszédos minták átlagos eltérése e fölött: a grafika épp változik (zaj ~0,1–0,3).
STEP_DIFF = 2.0
# A számjegy világos a sötét háttéren: ennél világosabb pixel a jel része.
BRIGHT = 110
# A jel legalább a kivágás magasságának ennyi része, és legalább ennyi px széles
# (a középső elválasztó vonal és a zaj így kiesik).
MIN_H_FRAC = 0.5
MIN_W_PX = 4
# A jelek egységes mérete (szél × mag) és a sablontól megengedett legnagyobb távolság.
GLYPH = (16, 24)
MAX_DIST = 40.0
# A középvonal a kivágásban (arány) és a két oldal elhagyott sávja körülötte (px).
SPLIT = 0.5
SPLIT_GAP = 2


def extract_crops(video: str, box, stride: int) -> np.ndarray:
    """Az állás-rész szürke kivágásai minden stride-adik kockán (N, H, W).

    A szélesség és a magasság PÁROS-ra kerekítve: a yuv420 kivágás páratlan
    méretet csendben páros-ra igazít, és a nyers kimenet sorai elcsúsznának."""
    from .video import ffmpeg_exe, probe

    w, h, _ = probe(video)
    x0, y0 = int(round(box[0] * w)), int(round(box[1] * h))
    cw, ch = int(round((box[2] - box[0]) * w)) // 2 * 2, int(round((box[3] - box[1]) * h)) // 2 * 2
    x0, y0 = x0 // 2 * 2, y0 // 2 * 2
    cmd = [ffmpeg_exe(), "-v", "error", "-threads", "1", "-i", video, "-vf",
           f"select='not(mod(n\\,{stride}))',crop={cw}:{ch}:{x0}:{y0},format=gray", "-vsync", "0",
           "-f", "rawvideo", "-pix_fmt", "gray", "-"]
    raw = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.DEVNULL, check=True).stdout
    n = len(raw) // (cw * ch)
    return np.frombuffer(raw[:n * cw * ch], np.uint8).reshape(n, ch, cw)


def stable_states(crops: np.ndarray, t: np.ndarray, present: np.ndarray) -> list[tuple[float, float, np.ndarray]]:
    """(kezdet, vég, medián kép) minden legalább STABLE_S hosszú, látszó, változatlan szakaszra."""
    n = len(crops)
    s = crops.reshape(n, -1).astype(np.float32)
    step = np.full(n, np.inf, np.float32)
    step[1:] = np.abs(s[1:] - s[:-1]).mean(1)
    ok = present[:n] & (step < STEP_DIFF)
    if n:
        ok[0] = present[0]
    out, i = [], 0
    while i < n:
        if not ok[i]:
            i += 1
            continue
        j = i
        while j + 1 < n and ok[j + 1]:
            j += 1
        if t[j] - t[i] >= STABLE_S:
            out.append((float(t[i]), float(t[j]), np.median(crops[i:j + 1], 0).astype(np.uint8)))
        i = j + 1
    return out


def glyphs(img: np.ndarray) -> list[list[np.ndarray]]:
    """A két oldal jelei balról jobbra, egységes méretre hozva: [[hazai jelek], [vendég jelek]]."""
    from PIL import Image
    from scipy import ndimage

    w = img.shape[1]
    mid = int(round(SPLIT * w))
    out = []
    for a, b in ((0, mid - SPLIT_GAP), (mid + SPLIT_GAP, w)):
        x = img[:, a:b] > BRIGHT
        lab, _ = ndimage.label(x)
        g = []
        for q, sl in enumerate(ndimage.find_objects(lab)):
            if sl is None:
                continue
            hh, ww = sl[0].stop - sl[0].start, sl[1].stop - sl[1].start
            if hh < MIN_H_FRAC * x.shape[0] or ww < MIN_W_PX:
                continue
            m = ((lab[sl] == q + 1) * 255).astype(np.uint8)
            g.append((sl[1].start, np.asarray(Image.fromarray(m).resize(GLYPH, Image.BILINEAR), np.float32) / 255.0))
        g.sort(key=lambda z: z[0])
        out.append([z[1] for z in g])
    return out


def read_state(img: np.ndarray, centers: np.ndarray, labels: list[int | None]) -> tuple[int, int] | None:
    """A két szám egy állapot képéről; None, ha bármelyik jel bizonytalan."""
    res = []
    for side in glyphs(img):
        if not 1 <= len(side) <= 2:
            return None
        digits = []
        for g in side:
            d = ((centers - g.ravel()) ** 2).sum(1)
            c = int(d.argmin())
            if labels[c] is None or d[c] > MAX_DIST:
                return None
            digits.append(str(labels[c]))
        res.append(int("".join(digits)))
    return res[0], res[1]


def timeline(readings: list[tuple[float, float, tuple[int, int] | None]]) -> dict:
    """Az állás-idővonal: csak a növekvő, lépésenként legfeljebb +2 gólos olvasatok.

    Egy gól ideje a [előző állapot vége, új állapot kezdete] sávba esik (a grafika
    közben nem látszott, vagy épp változott)."""
    seq = [{"t": 0.0, "t_elozo": 0.0, "allas": (0, 0)}]
    last_end, rejected = 0.0, []
    for t0, t1, r in readings:
        if r is None:
            continue
        h, v = seq[-1]["allas"]
        if r == (h, v):
            last_end = t1
            continue
        dh, dv = r[0] - h, r[1] - v
        if dh >= 0 and dv >= 0 and dh + dv <= 2:
            seq.append({"t": t0, "t_elozo": last_end, "allas": r})
            last_end = t1
        else:
            rejected.append({"t": t0, "olvasat": r, "elozo": (h, v)})
    goals = []
    for prev, cur in zip(seq[:-1], seq[1:]):
        for side, k in (("hazai", 0), ("vendeg", 1)):
            for _ in range(cur["allas"][k] - prev["allas"][k]):
                goals.append({"csapat": side, "t_tol": round(cur["t_elozo"], 1), "t_ig": round(cur["t"], 1),
                              "allas_utana": list(cur["allas"])})
    return {"vegeredmeny": list(seq[-1]["allas"]), "golok": goals, "valtozasok": len(seq) - 1,
            "elutasitott": rejected}


def main(argv=None) -> int:
    ap = argparse.ArgumentParser(description="Az állás kiolvasása az eredményjelzőből")
    sub = ap.add_subparsers(dest="cmd", required=True)
    c = sub.add_parser("crops")
    c.add_argument("--video", required=True)
    c.add_argument("--box", type=float, nargs=4, required=True, help="az állás-rész (x0 y0 x1 y1, 0..1)")
    c.add_argument("--stride", type=int, required=True, help="ugyanaz, mint a hbd.broadcast-nál")
    c.add_argument("--out", required=True)
    k = sub.add_parser("cluster")
    k.add_argument("--crops", required=True)
    k.add_argument("--broadcast", required=True)
    k.add_argument("--k", type=int, default=14)
    k.add_argument("--out", required=True)
    r = sub.add_parser("read")
    r.add_argument("--crops", required=True)
    r.add_argument("--broadcast", required=True)
    r.add_argument("--templates", required=True)
    r.add_argument("--labels", required=True, help="a klaszterek számjegyei vesszővel; x = nem számjegy")
    r.add_argument("--out", required=True)
    args = ap.parse_args(argv)
    if args.cmd == "crops":
        np.save(args.out, extract_crops(args.video, args.box, args.stride))
        return 0
    from .broadcast import SB_PRESENT_NCC, load

    crops = np.load(args.crops)
    bc = load(args.broadcast)
    n = min(len(crops), len(bc["t"]))
    states = stable_states(crops[:n], bc["t"][:n], bc["sb_ncc"][:n] > SB_PRESENT_NCC)
    if args.cmd == "cluster":
        from PIL import Image
        from sklearn.cluster import KMeans

        G = np.array([g.ravel() for _, _, img in states for side in glyphs(img) for g in side])
        km = KMeans(args.k, n_init=10, random_state=0).fit(G)
        out = Path(args.out)
        out.mkdir(parents=True, exist_ok=True)
        np.save(out / "kozeppontok.npy", km.cluster_centers_)
        gw, gh = GLYPH
        rows = []
        for ci in range(args.k):
            ex = [np.pad(e.reshape(gh, gw), 1, constant_values=0.5) for e in G[km.labels_ == ci][:8]]
            ex += [np.full((gh + 2, gw + 2), 0.5)] * (8 - len(ex))
            rows.append(np.hstack(ex))
        im = (np.vstack(rows) * 255).astype(np.uint8)
        Image.fromarray(im).resize((im.shape[1] * 3, im.shape[0] * 3), Image.NEAREST).save(out / "klaszterek.png")
        print(f"{len(states)} állapot, {len(G)} jel → {out / 'klaszterek.png'} (soronként egy klaszter; címkézd)")
        return 0
    centers = np.load(args.templates)
    labels = [None if x.strip() in ("x", "") else int(x) for x in args.labels.split(",")]
    readings = [(t0, t1, read_state(img, centers, labels)) for t0, t1, img in states]
    res = timeline(readings)
    res["allapotok"] = len(states)
    res["olvashato"] = sum(r[2] is not None for r in readings)
    Path(args.out).write_text(json.dumps(res, ensure_ascii=False, indent=1))
    print(f"végeredmény {res['vegeredmeny'][0]}:{res['vegeredmeny'][1]}, {len(res['golok'])} gól, "
          f"{res['olvashato']}/{len(states)} állapot olvasható, {len(res['elutasitott'])} elutasított olvasat")
    return 0


if __name__ == "__main__":
    raise SystemExit(main())


## 1. Testpont-kinyerés (GPU)

In [ ]:
from hbd.dataset import find_root, index_dataset
DATA = str(find_root('/kaggle/input'))
print('adat:', DATA, '| klipek:', len(index_dataset(DATA)))

In [ ]:
!python -m hbd.pose --data "$DATA" --out /kaggle/working/poses.npz --device 0 --preview 6

In [ ]:
from IPython.display import Image, display
display(Image('/kaggle/working/poses_preview.jpg'))

## 2. Kiértékelés és végső modellek

In [ ]:
!python -m hbd.train --poses /kaggle/working/poses.npz --out /kaggle/working/results

In [ ]:
from IPython.display import Markdown
display(Markdown(open('/kaggle/working/results/results.md').read()))